In [1]:
# orb_15m_retest_from_parquet.py
# Implements your ORB 15m Retest/Continuation strategy exactly,
# and runs it on every .parquet in the given folder.

import warnings
warnings.filterwarnings("ignore")

import os, glob, time
from datetime import timedelta, time as dtime
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd

# =============================
# USER PARAMETERS
# =============================

# Parquet directory (ALL files inside will be processed)
PARQUET_DIR = r"C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet"

# If timestamps in your parquet are NAIVE and represent UTC, keep True
ASSUME_NAIVE_TIMESTAMPS_ARE_UTC = True

# Session / timezone
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (quality + enough trades)
MAX_RETEST_MIN = 120            # minutes after breakout to wait for retest
RETEST_CONFIRM_CLOSE = True     # retest bar must close back through the level
BREAKOUT_CUSHION_PCT = 0.0005   # 0.05% cushion beyond OR to confirm breakout/retest
MAX_OPPOSITE_WICK_FRAC = 0.40   # reject entry bar if opposite wick > 40% of bar range

# Multiple trades per day
MAX_TRADES_PER_SESSION = 2
ALLOW_BOTH_SIDES = True

# Scale-out in thirds
USE_SCALE_OUT = True
TARGETS_R = [1.0, 2.0, 2.5]
SCALE_SPLIT = (1/3, 1/3, 1/3)

# Breakeven policy
MOVE_STOP_TO_BE_AT_R = 1.0      # arm BE after TP1 (==1R)
BE_ON_NEXT_BAR = True           # move stop to BE only on the next bar

# Continuation (fallback if no retest)
USE_CONTINUATION = True

# Time windows
BREAKOUT_DEADLINE   = "11:00"   # ET deadline for breakout
RETEST_DEADLINE     = "12:00"   # ET deadline for retest
CONTINUATION_CUTOFF = "12:00"   # ET cutoff for continuation entries

# Filters
USE_VWAP_FILTER   = True
USE_TREND_FILTER  = True        # 15m EMA trend filter
EMA_FAST = 20
EMA_SLOW = 50

USE_RSI_FILTER = True
RSI_LEN = 14
RSI_THRESH_LONG = 50.0
RSI_THRESH_SHORT = 50.0

USE_ADX_FILTER = False
ADX_LEN = 14
ADX_MIN = 18.0

USE_DAILY_TREND_FILTER = True
DAILY_EMA_FAST = 20
DAILY_EMA_SLOW = 50
DAILY_ATR_LEN  = 14

# OR width vs daily ATR band
USE_OR_WIDTH_ATR_FILTER = True
OR_ATR_MIN = 0.10
OR_ATR_MAX = 2.00

# Opening 15m volume quantile
USE_OPENING_VOL_FILTER = True
OPENING_VOL_LOOKBACK = 60
OPENING_VOL_QUANTILE = 0.30

# Gap filters
USE_GAP_FILTER = True
GAP_MIN = 0.003                # 0.3%
GAP_MAX = 0.04                 # 4.0%
ALIGN_WITH_GAP_DIR = False

USE_DOW_FILTER = False
ALLOWED_DOW = {1, 2, 3}        # Tue–Thu if enabled

# Optional index confirmation (set spy15 if you later load SPY locally)
USE_SPY_CONFIRM = False

# Entry realism when using close confirmation
#   "next_open" (recommended) = fill at next bar open after confirming close
#   "confirm_close"           = fill at confirming bar close
ENTRY_ON_CONFIRM = "next_open"

# Risk / costs
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00
FEE_PER_FILL = True            # charge fee per exit leg (TP/stop)

# Optional ATR trail on remainder (off by default)
USE_ATR_TRAIL = False
ATR15_LEN = 14
ATR15_MULT = 2.0

# Batch / persistence
SLEEP_BETWEEN_TICKERS = 0.2
AUTOSAVE_EVERY = 25

# Output files
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"
SUMMARY_CSV    = "orb_summary_sp500.csv"

# =============================
# Helpers
# =============================

def ensure_unique_columns(df: pd.DataFrame) -> pd.DataFrame:
    if getattr(df.columns, "duplicated", None) is not None and df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def sessionize(obj) -> pd.Series:
    idx = getattr(obj, "index", obj)
    return pd.to_datetime(pd.Index(idx).date)

def _to_clock(ts) -> dtime:
    return dtime(ts.hour, ts.minute)

def _clock_le(ts, hhmm: str) -> bool:
    h, m = map(int, hhmm.split(":"))
    return _to_clock(ts) <= dtime(h, m)

def wick_opposite_fraction(row: pd.Series, direction: str) -> float:
    h, l, o, c = float(row["High"]), float(row["Low"]), float(row.get("Open", np.nan)), float(row["Close"])
    rng = max(h - l, 1e-12)
    if direction == "long":
        opp = min(o, c) - l
    else:
        opp = h - max(o, c)
    return float(max(opp, 0.0) / rng)

def true_range(high, low, prev_close):
    return np.maximum.reduce([
        (high - low).values,
        np.abs(high - prev_close).values,
        np.abs(low - prev_close).values
    ])

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def _normalize_splits(num_targets: int, splits: tuple) -> List[float]:
    if num_targets <= 0:
        return [1.0]
    if not splits:
        return [1.0 / num_targets] * num_targets
    arr = list(splits[:num_targets])
    if len(arr) < num_targets:
        arr += [0.0] * (num_targets - len(arr))
    s = sum(arr)
    if s <= 0:
        return [1.0 / num_targets] * num_targets
    return [x / s for x in arr]

# Overall Totals (adds Profit Factor & Avg Win/Loss)
def print_overall_totals(trades: pd.DataFrame):
    print("\n=== Overall Totals ===")
    if trades is None or trades.empty:
        print("Total trades: 0")
        print("Total PnL ($): 0.00")
        print("Win rate: 0.00%")
        print("Avg R: 0.000 | Median R: 0.000")
        print("Profit Factor: 0.000")
        print("Avg Win ($): 0.00 | Avg Loss ($): 0.00")
        return

    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0)
    wins_mask = pnl > 0
    losses_mask = pnl < 0

    total_trades = int(len(pnl))
    total_pnl    = float(pnl.sum())
    win_rate     = float(wins_mask.mean() * 100.0) if total_trades else 0.0
    avg_r        = float(pd.to_numeric(trades["R_multiple"], errors="coerce").mean())
    median_r     = float(pd.to_numeric(trades["R_multiple"], errors="coerce").median())

    gross_profit = float(pnl[wins_mask].sum())
    gross_loss   = float(-pnl[losses_mask].sum())
    pf = (gross_profit / gross_loss) if gross_loss > 0 else (float("inf") if gross_profit > 0 else 0.0)

    avg_win  = float(pnl[wins_mask].mean()) if wins_mask.any() else 0.0
    avg_loss = float(pnl[losses_mask].mean()) if losses_mask.any() else 0.0

    print(f"Total trades: {total_trades}")
    print(f"Total PnL ($): {total_pnl:,.2f}")
    print(f"Win rate: {win_rate:.2f}%")
    print(f"Avg R: {avg_r:.3f} | Median R: {median_r:.3f}")
    print(f"Profit Factor: {pf:.3f}")
    print(f"Avg Win ($): {avg_win:,.2f} | Avg Loss ($): {avg_loss:,.2f}")

# =============================
# Indicators & features
# =============================

def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    for c in ("ORH","ORL"):
        if (df.columns == c).sum() > 0:
            df = df.drop(columns=[c])
    for col in ("High","Low"):
        if col not in df.columns:
            raise ValueError(f"compute_opening_range: missing column {col}")
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df["ORH"] = np.nan
    df["ORL"] = np.nan
    df.loc[first_bar_idx, "ORH"] = df.loc[first_bar_idx, "High"].astype(float).values
    df.loc[first_bar_idx, "ORL"] = df.loc[first_bar_idx, "Low"].astype(float).values
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return ensure_unique_columns(df)

def add_session_vwap(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df["Session"] = sessionize(df)
    tp = (df["High"] + df["Low"] + df["Close"]) / 3.0
    if "Volume" in df.columns:
        df["vwap_num"] = tp * df["Volume"]
        df["vwap_den"] = df["Volume"].replace(0, np.nan)
    else:
        df["vwap_num"] = tp
        df["vwap_den"] = 1.0
    df["VWAP"] = (df.groupby("Session")["vwap_num"].cumsum() /
                  df.groupby("Session")["vwap_den"].cumsum())
    return ensure_unique_columns(df.drop(columns=["vwap_num","vwap_den"]))

def add_ema_trend(df_15: pd.DataFrame, fast=EMA_FAST, slow=EMA_SLOW) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"EMA{fast}"] = df["Close"].ewm(span=fast, adjust=False).mean()
    df[f"EMA{slow}"] = df["Close"].ewm(span=slow, adjust=False).mean()
    return ensure_unique_columns(df)

def rsi(series: pd.Series, length=14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0)
    dn = -delta.clip(upper=0)
    avg_gain = up.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = dn.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan))
    return 100 - (100 / (1 + rs))

def add_rsi(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"RSI{length}"] = rsi(df["Close"], length)
    return ensure_unique_columns(df)

def add_adx(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    up_move = df["High"].diff()
    dn_move = -df["Low"].diff()
    plus_dm  = np.where((up_move > dn_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((dn_move > up_move) & (dn_move > 0), dn_move, 0.0)
    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    atr = tr.ewm(alpha=1/length, adjust=False).mean()
    plus_di  = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    dx = (abs(plus_di - minus_di) / (plus_di + minus_di).replace(0, np.nan)) * 100
    adx = dx.ewm(alpha=1/length, adjust=False).mean()
    df[f"ADX{length}"] = adx
    return ensure_unique_columns(df)

def add_atr_15m(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    df[f"ATR15_{length}"] = tr.ewm(alpha=1/length, adjust=False).mean()
    return ensure_unique_columns(df)

def build_daily_from_15m(df_15: pd.DataFrame) -> pd.DataFrame:
    d = ensure_unique_columns(df_15.copy())
    d["Session"] = sessionize(d)
    daily = d.groupby("Session").agg(
        Open=("Open","first"),
        High=("High","max"),
        Low=("Low","min"),
        Close=("Close","last"),
        Volume=("Volume","sum") if "Volume" in d.columns else ("Close","size"),
    )
    return daily

def add_daily_bias_and_atr(df_15: pd.DataFrame, ema_fast=20, ema_slow=50, atr_len=14) -> pd.DataFrame:
    dly = build_daily_from_15m(df_15)
    dly[f"EMA_D{ema_fast}"] = dly["Close"].ewm(span=ema_fast, adjust=False).mean()
    dly[f"EMA_D{ema_slow}"] = dly["Close"].ewm(span=ema_slow, adjust=False).mean()
    prev_close = dly["Close"].shift(1)
    tr = pd.Series(np.maximum.reduce([
        (dly["High"] - dly["Low"]).values,
        np.abs(dly["High"] - prev_close).values,
        np.abs(dly["Low"]  - prev_close).values
    ]), index=dly.index)
    atr_col = f"ATR_D{atr_len}"
    dly[atr_col] = tr.ewm(alpha=1/atr_len, adjust=False).mean()

    if "Volume" in df_15.columns:
        tmp = df_15.copy()
        tmp["Session"] = sessionize(tmp)
        open15_vol = tmp.groupby("Session")["Volume"].first()
        dly["Open15_Vol"] = open15_vol
        dly["Open15_Vol_Thresh"] = (
            dly["Open15_Vol"]
            .rolling(window=OPENING_VOL_LOOKBACK, min_periods=10)
            .quantile(OPENING_VOL_QUANTILE)
        )

    feature_cols = [f"EMA_D{ema_fast}", f"EMA_D{ema_slow}", atr_col, "Open15_Vol", "Open15_Vol_Thresh"]
    feature_cols = [c for c in feature_cols if c in dly.columns]
    out = ensure_unique_columns(df_15.copy())
    out["Session"] = sessionize(out)
    out = out.merge(dly[feature_cols], left_on="Session", right_index=True, how="left")
    return ensure_unique_columns(out)

# =============================
# Backtest (per-ticker)
# =============================

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = TARGETS_R,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE,
    spy15: Optional[pd.DataFrame] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    df = compute_opening_range(df_15)
    if USE_VWAP_FILTER:  df = add_session_vwap(df)
    if USE_TREND_FILTER: df = add_ema_trend(df, EMA_FAST, EMA_SLOW)
    df = add_daily_bias_and_atr(df, DAILY_EMA_FAST, DAILY_EMA_SLOW, DAILY_ATR_LEN)

    df["Session"] = sessionize(df)
    first_idx = df.groupby("Session").head(1).index
    prev_close_series = df.groupby("Session")["Close"].last().shift(1)
    prev_close_map = df["Session"].map(prev_close_series)
    df.loc[first_idx, "GapPct"] = (df.loc[first_idx, "Open"] / prev_close_map.loc[first_idx] - 1.0)
    df["GapPct"] = df.groupby("Session")["GapPct"].ffill()

    if USE_RSI_FILTER:  df = add_rsi(df, RSI_LEN)
    if USE_ADX_FILTER:  df = add_adx(df, ADX_LEN)
    if USE_ATR_TRAIL:   df = add_atr_15m(df, ATR15_LEN)
    df = ensure_unique_columns(df)

    sessions = df["Session"].unique()
    trades = []

    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh, orl = float(sdf["ORH"].iloc[0]), float(sdf["ORL"].iloc[0])
        if np.isnan(orh) or np.isnan(orl):
            continue

        after_open = sdf.iloc[1:].copy()
        long_breaks  = after_open[after_open["Close"] > orh * (1 + BREAKOUT_CUSHION_PCT)]
        short_breaks = after_open[after_open["Close"] < orl * (1 - BREAKOUT_CUSHION_PCT)]

        cands = []
        for idx, row in long_breaks.iterrows():
            cands.append(("long", idx, row))
        for idx, row in short_breaks.iterrows():
            cands.append(("short", idx, row))
        cands.sort(key=lambda x: x[1])

        trades_this_session = 0
        took_long = False
        took_short = False

        for direction, btime, breakout_row in cands:
            if not _clock_le(breakout_row.name, BREAKOUT_DEADLINE):
                continue
            if not ALLOW_BOTH_SIDES:
                if took_long and direction == "short": break
                if took_short and direction == "long": break
            if direction == "long" and took_long:   continue
            if direction == "short" and took_short: continue

            cutoff = btime + timedelta(minutes=max_retest_min)
            window = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
            level = orh if direction == "long" else orl

            touch = window[(window["Low"] <= level) & (window["High"] >= level)].head(1)

            retest_used = True
            if touch.empty:
                if USE_CONTINUATION and _clock_le(breakout_row.name, CONTINUATION_CUTOFF):
                    retest_used = False
                    retest_time = breakout_row.name
                    retest_bar = breakout_row
                else:
                    continue
            else:
                retest_bar = touch.iloc[0]
                retest_time = retest_bar.name

            if retest_used and retest_confirm_close:
                if direction == "long":
                    ok = (retest_bar["Close"] >= level * (1 + BREAKOUT_CUSHION_PCT))
                else:
                    ok = (retest_bar["Close"] <= level * (1 - BREAKOUT_CUSHION_PCT))
                if not ok: continue

            if MAX_OPPOSITE_WICK_FRAC is not None:
                if wick_opposite_fraction(retest_bar, direction) > MAX_OPPOSITE_WICK_FRAC:
                    continue

            if USE_DOW_FILTER and retest_time.weekday() not in ALLOWED_DOW:
                continue

            if USE_VWAP_FILTER and "VWAP" in sdf.columns:
                v_prev = sdf.loc[:retest_time, "VWAP"].tail(3).dropna().values
                rising  = (len(v_prev) < 2) or np.all(np.diff(v_prev) >= 0)
                falling = (len(v_prev) < 2) or np.all(np.diff(v_prev) <= 0)
                if direction == "long":
                    if not (retest_bar["Close"] > retest_bar["VWAP"] and rising): continue
                else:
                    if not (retest_bar["Close"] < retest_bar["VWAP"] and falling): continue

            if USE_TREND_FILTER and (f"EMA{EMA_FAST}" in sdf.columns) and (f"EMA{EMA_SLOW}" in sdf.columns):
                emaf = retest_bar.get(f"EMA{EMA_FAST}", np.nan); emas = retest_bar.get(f"EMA{EMA_SLOW}", np.nan)
                if np.isfinite(emaf) and np.isfinite(emas):
                    if direction == "long" and not (emaf > emas): continue
                    if direction == "short" and not (emaf < emas): continue

            if USE_DAILY_TREND_FILTER and f"EMA_D{DAILY_EMA_FAST}" in sdf.columns and f"EMA_D{DAILY_EMA_SLOW}" in sdf.columns:
                dfast = float(sdf[f"EMA_D{DAILY_EMA_FAST}"].iloc[0])
                dslow = float(sdf[f"EMA_D{DAILY_EMA_SLOW}"].iloc[0])
                if direction == "long" and not (dfast > dslow): continue
                if direction == "short" and not (dfast < dslow): continue

            atr_col = f"ATR_D{DAILY_ATR_LEN}"
            if USE_OR_WIDTH_ATR_FILTER and atr_col in sdf.columns:
                or_width = float(orh - orl)
                atr_d = float(sdf[atr_col].iloc[0])
                if atr_d <= 0: continue
                frac = or_width / atr_d
                if not (OR_ATR_MIN <= frac <= OR_ATR_MAX): continue

            if USE_RSI_FILTER and f"RSI{RSI_LEN}" in sdf.columns:
                rsi_val = float(retest_bar[f"RSI{RSI_LEN}"])
                if direction == "long" and not (rsi_val >= RSI_THRESH_LONG): continue
                if direction == "short" and not (rsi_val <= 100 - RSI_THRESH_SHORT): continue

            if USE_ADX_FILTER and f"ADX{ADX_LEN}" in sdf.columns:
                adx_val = float(retest_bar[f"ADX{ADX_LEN}"])
                if not (adx_val >= ADX_MIN): continue

            if USE_GAP_FILTER and "GapPct" in sdf.columns:
                gap = sdf.loc[sdf.index.min(), "GapPct"]
                if pd.notna(gap) and not (GAP_MIN <= abs(float(gap)) <= GAP_MAX):
                    continue

            if ALIGN_WITH_GAP_DIR and "GapPct" in sdf.columns:
                g = float(sdf["GapPct"].iloc[0]) if pd.notna(sdf["GapPct"].iloc[0]) else 0.0
                if (direction == "long" and g < 0) or (direction == "short" and g > 0):
                    continue

            if USE_SPY_CONFIRM and (spy15 is not None) and (retest_time in spy15.index):
                spy_row = spy15.loc[retest_time]
                if direction == "long" and not (spy_row["SPY_Close"] > spy_row["SPY_VWAP"]): continue
                if direction == "short" and not (spy_row["SPY_Close"] < spy_row["SPY_VWAP"]): continue

            # ===== Realistic Entry =====
            if retest_used and retest_confirm_close:
                if ENTRY_ON_CONFIRM.lower() == "next_open":
                    next_bar = sdf[sdf.index > retest_time].head(1)
                    if next_bar.empty:
                        continue
                    entry_time = next_bar.index[0]
                    raw_entry = float(next_bar["Open"].iloc[0])
                elif ENTRY_ON_CONFIRM.lower() == "confirm_close":
                    entry_time = retest_time
                    raw_entry = float(retest_bar["Close"])
                else:
                    entry_time = retest_time
                    raw_entry = float(retest_bar["Close"])
            else:
                entry_time = retest_time
                raw_entry = float(level)

            entry = _apply_slippage(raw_entry, slippage_bps, "buy" if direction == "long" else "sell")
            stop  = orl if direction == "long" else orh
            rps   = abs(entry - stop)
            if rps <= 1e-12:
                continue

            qty = POSITION_SIZE_DOLLARS / max(entry, 1e-12)
            side_mult = 1 if direction == "long" else -1

            targets_r = TARGETS_R if USE_SCALE_OUT else r_targets
            targets = [(entry + r * rps) if direction == "long" else (entry - r * rps) for r in targets_r]

            run = sdf[sdf.index >= entry_time].copy()
            qty_left = qty
            exits = []
            next_tp_idx = 0

            if USE_SCALE_OUT and len(targets) >= 1:
                splits = _normalize_splits(len(targets), SCALE_SPLIT)
                leg_sizes = [qty * s for s in splits]
            else:
                leg_sizes = [qty]

            be_level = entry
            be_pending = False
            be_set_time = None

            for ts, row in run.iterrows():
                hi, lo = float(row["High"]), float(row["Low"])

                if be_pending and (not BE_ON_NEXT_BAR or (be_set_time is not None and ts > be_set_time)):
                    stop = be_level
                    be_pending = False

                if USE_ATR_TRAIL and f"ATR15_{ATR15_LEN}" in sdf.columns and qty_left > 1e-9:
                    atr_now = float(row[f"ATR15_{ATR15_LEN}"])
                    if direction == "long":
                        stop = max(stop, hi - ATR15_MULT * atr_now)
                    else:
                        stop = min(stop, lo + ATR15_MULT * atr_now)

                # Stop first
                if lo <= stop <= hi:
                    px = _apply_slippage(stop, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                    label = "breakeven" if abs(px - be_level) < 1e-10 else "stop"
                    exits.append((label, ts, px, qty_left))
                    qty_left = 0.0
                    break

                # Targets (sequential)
                if next_tp_idx < len(targets):
                    tp = targets[next_tp_idx]
                    if lo <= tp <= hi:
                        px = _apply_slippage(tp, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                        fill_qty = leg_sizes[next_tp_idx] if next_tp_idx < len(leg_sizes) else qty_left
                        exits.append((f"tp{next_tp_idx+1}", ts, px, fill_qty))
                        qty_left -= fill_qty
                        if next_tp_idx == 0 and MOVE_STOP_TO_BE_AT_R is not None:
                            be_pending = True
                            be_set_time = ts
                        next_tp_idx += 1
                        if qty_left <= 1e-9:
                            break

                # If no TP1 by continuation cutoff, tighten to BE
                if USE_CONTINUATION and not _clock_le(ts, CONTINUATION_CUTOFF) and next_tp_idx == 0:
                    stop = be_level

            # EOD exit for any leftover
            if qty_left > 1e-9:
                last = run.iloc[-1]
                px = _apply_slippage(float(last["Close"]), SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                exits.append(("eod", run.index[-1], px, qty_left))
                qty_left = 0.0

            cash_pnl = sum((px - entry) * side_mult * q for (_, _, px, q) in exits)
            if FEE_PER_FILL:
                cash_pnl -= FEES_PER_TRADE * len(exits)
            else:
                cash_pnl -= FEES_PER_TRADE

            trades.append({
                "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
                "Session": ses,
                "Direction": direction,
                "ORH": orh, "ORL": orl,
                "EntryTime": entry_time,
                "Entry": entry, "Stop": stop,
                "Targets": targets,
                "Exits": [(lab, ts, px, q) for (lab, ts, px, q) in exits],
                "Qty": qty,
                "PnL_$": cash_pnl,
                "R_multiple": cash_pnl / (rps * max(qty, 1e-12))
            })

            if direction == "long":  took_long  = True
            if direction == "short": took_short = True
            trades_this_session += 1
            if trades_this_session >= MAX_TRADES_PER_SESSION:
                break

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Parquet loading
# =============================

STD_MAP = {"open":"Open","high":"High","low":"Low","close":"Close","volume":"Volume"}

def _standardize_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    lower = {c.lower(): c for c in df.columns}
    for k, std in STD_MAP.items():
        if k in lower:                rename[lower[k]] = std
        elif k.capitalize() in df.columns: rename[k.capitalize()] = std
        elif k.upper() in df.columns: rename[k.upper()] = std
    out = df.rename(columns=rename)
    drop_cols = [c for c in out.columns if str(c).lower().startswith("adj")]
    return out.drop(columns=drop_cols, errors="ignore")

def _choose_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for cand in ["EventAt","Datetime","datetime","Timestamp","timestamp","Date","date","Time","time"]:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            df = df.set_index(cand)
            break
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No datetime index/column found.")
    return df

def _maybe_resample_to_15m(df: pd.DataFrame) -> pd.DataFrame:
    if len(df.index) < 3:
        return df
    # If median step < 15m, upsample to 15m OHLCV
    deltas = (df.index[1:] - df.index[:-1]).asi8
    if len(deltas) == 0:
        return df
    med_ns = np.median(deltas)
    fifteen_ns = pd.Timedelta("15T").value
    if med_ns < fifteen_ns:
        o = df["Open"].resample("15T", label="left", closed="left").first()
        h = df["High"].resample("15T", label="left", closed="left").max()
        l = df["Low"].resample("15T", label="left", closed="left").min()
        c = df["Close"].resample("15T", label="left", closed="left").last()
        v = df["Volume"].resample("15T", label="left", closed="left").sum() if "Volume" in df.columns else None
        parts = {"Open":o,"High":h,"Low":l,"Close":c}
        if v is not None: parts["Volume"] = v
        out = pd.concat(parts, axis=1).dropna(subset=["Open","High","Low","Close"])
        return out
    return df

def load_parquet_15m(path: str, tz: str = TZ) -> pd.DataFrame:
    df = pd.read_parquet(path, engine="pyarrow")
    df = _choose_dt_index(df).sort_index()

    if df.index.tz is None:
        if ASSUME_NAIVE_TIMESTAMPS_ARE_UTC:
            df = df.tz_localize("UTC").tz_convert(tz)
        else:
            df = df.tz_localize(tz)
    else:
        df = df.tz_convert(tz)

    df = _standardize_ohlcv_columns(df)
    for need in ["Open","High","Low","Close"]:
        if need not in df.columns:
            raise ValueError(f"{os.path.basename(path)} missing required column: {need}")

    df = _maybe_resample_to_15m(df)
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    return ensure_unique_columns(df.sort_index())

# =============================
# Runner (folder of parquet files)
# =============================

def run_folder(folder: str):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"Found {len(files)} parquet files.")
    all_trades, all_equity = [], []

    for i, fpath in enumerate(files, 1):
        ticker = os.path.splitext(os.path.basename(fpath))[0].upper().replace("_","").replace("-","")
        print(f"\n== {ticker} ({i}/{len(files)}) ==")
        try:
            df15 = load_parquet_15m(fpath)
            df15["Ticker"] = ticker
        except Exception as e:
            print(f"Skipping {ticker}: {e}")
            continue

        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=SLIPPAGE_BPS,
            fees=FEES_PER_TRADE,
            retest_confirm_close=RETEST_CONFIRM_CLOSE,
            spy15=None
        )

        if not tlog.empty:
            tlog_out = tlog.copy()
            tlog_out["Targets"] = tlog_out["Targets"].apply(lambda xs: ";".join([f"{p:.6f}" for p in xs]))
            tlog_out["Exits"]   = tlog_out["Exits"].apply(lambda xs: ";".join([f"{t[0]}|{t[1]}|{t[2]:.6f}|{t[3]:.6f}" for t in xs]))
            all_trades.append(tlog_out)
        if not eq.empty:
            eq_out = eq.copy(); eq_out["Ticker"] = ticker
            all_equity.append(eq_out)

        if AUTOSAVE_EVERY and i % AUTOSAVE_EVERY == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

        time.sleep(SLEEP_BETWEEN_TICKERS)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

def main():
    trades, equity = run_folder(PARQUET_DIR)

    print_overall_totals(trades)

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

    if not trades.empty:
        summary = (
            trades.groupby("Ticker")["PnL_$"]
                  .agg(['count', 'sum', 'mean', 'std'])
                  .rename(columns={'count':'Trades','sum':'PnL_sum','mean':'AvgPnL','std':'StdPnL'})
                  .reset_index()
                  .sort_values("PnL_sum", ascending=False)
        )
        wr = trades.groupby("Ticker")["PnL_$"].apply(lambda s: (pd.to_numeric(s, errors="coerce") > 0).mean()*100.0).rename("WinRate_%")
        pos = trades.groupby("Ticker")["PnL_$"].apply(lambda s: pd.to_numeric(s, errors="coerce").clip(lower=0).sum()).rename("GrossProfit")
        neg = trades.groupby("Ticker")["PnL_$"].apply(lambda s: -pd.to_numeric(s, errors="coerce").clip(upper=0).sum()).rename("GrossLoss")
        pf = (pos / neg.replace(0, np.nan)).rename("PF")
        summary = summary.merge(wr, on="Ticker", how="left").merge(pf, on="Ticker", how="left")
        print("\n=== Summary by Ticker (top 50) ===")
        print(summary.head(50).to_string(index=False))
        summary.to_csv(SUMMARY_CSV, index=False)
    else:
        print("No trades generated with current settings.")

if __name__ == "__main__":
    main()


Found 20 parquet files.

== AAPL (1/20) ==

== AMD (2/20) ==
Skipping AMD: [Errno 9] Bad file descriptor: 'C:\\Users\\pcagm\\OneDrive\\Desktop\\downloads\\parquet\\parquet\\AMD.parquet'

== AMZN (3/20) ==

== CAT (4/20) ==
Skipping CAT: [Errno 9] Bad file descriptor: 'C:\\Users\\pcagm\\OneDrive\\Desktop\\downloads\\parquet\\parquet\\CAT.parquet'

== CHWY (5/20) ==
Skipping CHWY: [Errno 9] Bad file descriptor: 'C:\\Users\\pcagm\\OneDrive\\Desktop\\downloads\\parquet\\parquet\\CHWY.parquet'

== CVNA (6/20) ==
Skipping CVNA: [Errno 9] Bad file descriptor: 'C:\\Users\\pcagm\\OneDrive\\Desktop\\downloads\\parquet\\parquet\\CVNA.parquet'

== GLD (7/20) ==

== GOOGL (8/20) ==
Skipping GOOGL: [Errno 9] Bad file descriptor: 'C:\\Users\\pcagm\\OneDrive\\Desktop\\downloads\\parquet\\parquet\\GOOGL.parquet'

== GS (9/20) ==

== JPM (10/20) ==

== MSFT (11/20) ==
Skipping MSFT: [Errno 9] Bad file descriptor: 'C:\\Users\\pcagm\\OneDrive\\Desktop\\downloads\\parquet\\parquet\\MSFT.parquet'

== NET (1

In [12]:
# orb_15m_retest_from_parquet.py
# Runs your ORB (Opening Range Breakout) 15m Retest/Continuation strategy
# across all local parquet files in PARQUET_DIR (e.g., AMD.parquet, AAPL.parquet, ...).
#
# Strategy highlights (as requested):
# - Scale-out in thirds: TP1=1R, TP2=2R, TP3=2.5R (1/3 each)
# - Move stop to BE ONLY on the NEXT BAR after TP1 (conservative)
# - Retest confirmation (close back through level, with cushion)
# - Continuation fallback if no retest by cutoff
# - VWAP, EMA trend, RSI, daily EMA/ATR filters, OR-width vs daily ATR band, gap filters
# - Optional ATR trail (off by default)
# - Profit Factor and summary outputs

import warnings
warnings.filterwarnings("ignore")

import os, glob, time
from datetime import timedelta, time as dtime
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
from pathlib import Path

# =============================
# USER PARAMETERS
# =============================

# Folder that contains your parquet files (ex: "AMD.parquet", "AAPL.parquet", ...)
PARQUET_DIR = r"C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet"
# If timestamps in parquet are naive and represent UTC, set True (common). If already ET, set False.
ASSUME_NAIVE_TIMESTAMPS_ARE_UTC = True

# Session / timezone
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (quality + enough trades)
MAX_RETEST_MIN = 120            # minutes after breakout to wait for retest
RETEST_CONFIRM_CLOSE = True     # retest bar must close back through the level
BREAKOUT_CUSHION_PCT = 0.0005   # 0.05% cushion beyond OR to confirm breakout/retest
MAX_OPPOSITE_WICK_FRAC = 0.40   # reject entry bar if opposite wick > 40% of bar range

# Multiple trades per day
MAX_TRADES_PER_SESSION = 2      # up to 2 trades per day
ALLOW_BOTH_SIDES = True         # allow long and short in same session if both trigger

# Scale-out in thirds
USE_SCALE_OUT = True
TARGETS_R = [1.0, 2.0, 2.5]
SCALE_SPLIT = (1/3, 1/3, 1/3)

# Breakeven policy
MOVE_STOP_TO_BE_AT_R = 1.0      # arm BE after TP1 (==1R) is hit
BE_ON_NEXT_BAR = True           # move stop to BE only on the next bar (conservative)

# Continuation (fallback if no retest)
USE_CONTINUATION = True

# Time windows
BREAKOUT_DEADLINE   = "11:00"   # ET deadline for breakout close
RETEST_DEADLINE     = "12:00"   # ET deadline for retest
CONTINUATION_CUTOFF = "12:00"   # ET cutoff for continuation entries

# Filters
USE_VWAP_FILTER   = True
USE_TREND_FILTER  = True        # 15m EMA trend filter
EMA_FAST = 20
EMA_SLOW = 50

USE_RSI_FILTER = True
RSI_LEN = 14
RSI_THRESH_LONG = 50.0
RSI_THRESH_SHORT = 50.0

USE_ADX_FILTER = False
ADX_LEN = 14
ADX_MIN = 18.0

USE_DAILY_TREND_FILTER = True
DAILY_EMA_FAST = 20
DAILY_EMA_SLOW = 50
DAILY_ATR_LEN  = 14

# OR width vs daily ATR band
USE_OR_WIDTH_ATR_FILTER = True
OR_ATR_MIN = 0.10
OR_ATR_MAX = 2.00

# Opening 15m volume quantile
USE_OPENING_VOL_FILTER = True
OPENING_VOL_LOOKBACK = 60
OPENING_VOL_QUANTILE = 0.30

# Gap filters
USE_GAP_FILTER = True
GAP_MIN = 0.003                # 0.3%
GAP_MAX = 0.04                 # 4.0%
ALIGN_WITH_GAP_DIR = False

USE_DOW_FILTER = False
ALLOWED_DOW = {1, 2, 3}        # Tue–Thu if enabled

# Optional index confirmation (not used without SPY data here)
USE_SPY_CONFIRM = False

# Entry realism when using close confirmation
#   "next_open" (recommended) = fill at next bar open after confirming close
#   "confirm_close"           = fill at confirming bar close
ENTRY_ON_CONFIRM = "next_open"

# Risk / costs
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00
FEE_PER_FILL = True            # charge fee per exit leg (TP/stop)

# Optional ATR trail on remainder (off by default)
USE_ATR_TRAIL = False
ATR15_LEN = 14
ATR15_MULT = 2.0

# Batch / persistence
SLEEP_BETWEEN_TICKERS = 0.2
AUTOSAVE_EVERY = 25

# Output files
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"
SUMMARY_CSV    = "orb_summary_sp500.csv"

# =============================
# Helpers (generic)
# =============================

def ensure_unique_columns(df: pd.DataFrame) -> pd.DataFrame:
    if getattr(df.columns, "duplicated", None) is not None and df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def sessionize(obj) -> pd.Series:
    """Return a per-bar session key (one per trading day). Accepts DataFrame or index-like."""
    idx = getattr(obj, "index", obj)
    return pd.to_datetime(pd.Index(idx).date)

def _to_clock(ts) -> dtime:
    return dtime(ts.hour, ts.minute)

def _clock_le(ts, hhmm: str) -> bool:
    h, m = map(int, hhmm.split(":"))
    return _to_clock(ts) <= dtime(h, m)

def wick_opposite_fraction(row: pd.Series, direction: str) -> float:
    h, l, o, c = float(row["High"]), float(row["Low"]), float(row.get("Open", np.nan)), float(row["Close"])
    rng = max(h - l, 1e-12)
    if direction == "long":
        opp = min(o, c) - l
    else:
        opp = h - max(o, c)
    return float(max(opp, 0.0) / rng)

def true_range(high, low, prev_close):
    return np.maximum.reduce([
        (high - low).values,
        np.abs(high - prev_close).values,
        np.abs(low - prev_close).values
    ])

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def _normalize_splits(num_targets: int, splits: tuple) -> List[float]:
    """Normalize/pad/truncate the splits to match number of targets."""
    if num_targets <= 0:
        return [1.0]
    if not splits:
        return [1.0 / num_targets] * num_targets
    arr = list(splits[:num_targets])
    if len(arr) < num_targets:
        arr += [0.0] * (num_targets - len(arr))
    s = sum(arr)
    if s <= 0:
        return [1.0 / num_targets] * num_targets
    return [x / s for x in arr]

# Overall Totals (adds Profit Factor & Avg Win/Loss)
def print_overall_totals(trades: pd.DataFrame):
    """Print overall totals across all tickers/trades (with Profit Factor)."""
    print("\n=== Overall Totals ===")
    if trades is None or trades.empty:
        print("Total trades: 0")
        print("Total PnL ($): 0.00")
        print("Win rate: 0.00%")
        print("Avg R: 0.000 | Median R: 0.000")
        print("Profit Factor: 0.000")
        print("Avg Win ($): 0.00 | Avg Loss ($): 0.00")
        return

    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0)
    wins_mask = pnl > 0
    losses_mask = pnl < 0

    total_trades = int(len(pnl))
    total_pnl    = float(pnl.sum())
    win_rate     = float(wins_mask.mean() * 100.0) if total_trades else 0.0
    avg_r        = float(pd.to_numeric(trades["R_multiple"], errors="coerce").mean())
    median_r     = float(pd.to_numeric(trades["R_multiple"], errors="coerce").median())

    gross_profit = float(pnl[wins_mask].sum())
    gross_loss   = float(-pnl[losses_mask].sum())  # positive
    pf = (gross_profit / gross_loss) if gross_loss > 0 else (float("inf") if gross_profit > 0 else 0.0)

    avg_win  = float(pnl[wins_mask].mean()) if wins_mask.any() else 0.0
    avg_loss = float(pnl[losses_mask].mean()) if losses_mask.any() else 0.0  # negative

    print(f"Total trades: {total_trades}")
    print(f"Total PnL ($): {total_pnl:,.2f}")
    print(f"Win rate: {win_rate:.2f}%")
    print(f"Avg R: {avg_r:.3f} | Median R: {median_r:.3f}")
    print(f"Profit Factor: {pf:.3f}")
    print(f"Avg Win ($): {avg_win:,.2f} | Avg Loss ($): {avg_loss:,.2f}")

# =============================
# Indicators & features
# =============================

def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    for c in ("ORH","ORL"):
        if (df.columns == c).sum() > 0:
            df = df.drop(columns=[c])

    for col in ("High","Low"):
        if col not in df.columns:
            raise ValueError(f"compute_opening_range: missing column {col}")

    if not df.index.is_monotonic_increasing:
        df = df.sort_index()

    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df["ORH"] = np.nan
    df["ORL"] = np.nan
    df.loc[first_bar_idx, "ORH"] = df.loc[first_bar_idx, "High"].astype(float).values
    df.loc[first_bar_idx, "ORL"] = df.loc[first_bar_idx, "Low"].astype(float).values
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return ensure_unique_columns(df)

def add_session_vwap(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df["Session"] = sessionize(df)
    tp = (df["High"] + df["Low"] + df["Close"]) / 3.0
    if "Volume" in df.columns:
        df["vwap_num"] = tp * df["Volume"]
        df["vwap_den"] = df["Volume"].replace(0, np.nan)
    else:
        df["vwap_num"] = tp
        df["vwap_den"] = 1.0
    df["VWAP"] = (df.groupby("Session")["vwap_num"].cumsum() /
                  df.groupby("Session")["vwap_den"].cumsum())
    return ensure_unique_columns(df.drop(columns=["vwap_num","vwap_den"]))

def add_ema_trend(df_15: pd.DataFrame, fast=EMA_FAST, slow=EMA_SLOW) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"EMA{fast}"] = df["Close"].ewm(span=fast, adjust=False).mean()
    df[f"EMA{slow}"] = df["Close"].ewm(span=slow, adjust=False).mean()
    return ensure_unique_columns(df)

def rsi(series: pd.Series, length=14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0)
    dn = -delta.clip(upper=0)
    avg_gain = up.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = dn.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan))
    return 100 - (100 / (1 + rs))

def add_rsi(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"RSI{length}"] = rsi(df["Close"], length)
    return ensure_unique_columns(df)

def add_adx(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    up_move = df["High"].diff()
    dn_move = -df["Low"].diff()
    plus_dm  = np.where((up_move > dn_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((dn_move > up_move) & (dn_move > 0), dn_move, 0.0)

    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    atr = tr.ewm(alpha=1/length, adjust=False).mean()

    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    dx = (abs(plus_di - minus_di) / (plus_di + minus_di).replace(0, np.nan)) * 100
    adx = dx.ewm(alpha=1/length, adjust=False).mean()
    df[f"ADX{length}"] = adx
    return ensure_unique_columns(df)

def add_atr_15m(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    df[f"ATR15_{length}"] = tr.ewm(alpha=1/length, adjust=False).mean()
    return ensure_unique_columns(df)

def build_daily_from_15m(df_15: pd.DataFrame) -> pd.DataFrame:
    d = ensure_unique_columns(df_15.copy())
    d["Session"] = sessionize(d)
    daily = d.groupby("Session").agg(
        Open=("Open","first"),
        High=("High","max"),
        Low=("Low","min"),
        Close=("Close","last"),
        Volume=("Volume","sum") if "Volume" in d.columns else ("Close","size"),
    )
    return daily

def add_daily_bias_and_atr(df_15: pd.DataFrame, ema_fast=20, ema_slow=50, atr_len=14) -> pd.DataFrame:
    dly = build_daily_from_15m(df_15)

    # Daily EMAs (trend bias)
    dly[f"EMA_D{ema_fast}"] = dly["Close"].ewm(span=ema_fast, adjust=False).mean()
    dly[f"EMA_D{ema_slow}"] = dly["Close"].ewm(span=ema_slow, adjust=False).mean()

    # Daily ATR
    prev_close = dly["Close"].shift(1)
    tr = pd.Series(np.maximum.reduce([
        (dly["High"] - dly["Low"]).values,
        np.abs(dly["High"] - prev_close).values,
        np.abs(dly["Low"]  - prev_close).values
    ]), index=dly.index)
    atr_col = f"ATR_D{atr_len}"
    dly[atr_col] = tr.ewm(alpha=1/atr_len, adjust=False).mean()

    # Opening 15m volume + rolling quantile threshold (if Volume exists)
    if "Volume" in df_15.columns:
        tmp = df_15.copy()
        tmp["Session"] = sessionize(tmp)
        open15_vol = tmp.groupby("Session")["Volume"].first()
        dly["Open15_Vol"] = open15_vol
        dly["Open15_Vol_Thresh"] = (
            dly["Open15_Vol"]
            .rolling(window=OPENING_VOL_LOOKBACK, min_periods=10)
            .quantile(OPENING_VOL_QUANTILE)
        )

    # Merge ONLY feature columns
    feature_cols = [f"EMA_D{ema_fast}", f"EMA_D{ema_slow}", atr_col, "Open15_Vol", "Open15_Vol_Thresh"]
    feature_cols = [c for c in feature_cols if c in dly.columns]

    out = ensure_unique_columns(df_15.copy())
    out["Session"] = sessionize(out)
    out = out.merge(dly[feature_cols], left_on="Session", right_index=True, how="left")
    return ensure_unique_columns(out)

# =============================
# Backtest (per-ticker)
# =============================

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = TARGETS_R,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE,
    spy15: Optional[pd.DataFrame] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    # Opening range + features
    df = compute_opening_range(df_15)
    if USE_VWAP_FILTER:  df = add_session_vwap(df)
    if USE_TREND_FILTER: df = add_ema_trend(df, EMA_FAST, EMA_SLOW)
    df = add_daily_bias_and_atr(df, DAILY_EMA_FAST, DAILY_EMA_SLOW, DAILY_ATR_LEN)

    # Gap (first 15m vs previous day's close)
    df["Session"] = sessionize(df)
    first_idx = df.groupby("Session").head(1).index
    prev_close_series = df.groupby("Session")["Close"].last().shift(1)
    prev_close_map = df["Session"].map(prev_close_series)
    df.loc[first_idx, "GapPct"] = (df.loc[first_idx, "Open"] / prev_close_map.loc[first_idx] - 1.0)
    df["GapPct"] = df.groupby("Session")["GapPct"].ffill()

    # Intraday indicators
    if USE_RSI_FILTER:  df = add_rsi(df, RSI_LEN)
    if USE_ADX_FILTER:  df = add_adx(df, ADX_LEN)
    if USE_ATR_TRAIL:   df = add_atr_15m(df, ATR15_LEN)
    df = ensure_unique_columns(df)

    sessions = df["Session"].unique()
    trades = []

    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh, orl = float(sdf["ORH"].iloc[0]), float(sdf["ORL"].iloc[0])
        if np.isnan(orh) or np.isnan(orl):
            continue

        # All breakout candidates after the OR bar (use cushion)
        after_open = sdf.iloc[1:].copy()
        long_breaks  = after_open[after_open["Close"] > orh * (1 + BREAKOUT_CUSHION_PCT)]
        short_breaks = after_open[after_open["Close"] < orl * (1 - BREAKOUT_CUSHION_PCT)]

        cands = []
        for idx, row in long_breaks.iterrows():
            cands.append(("long", idx, row))
        for idx, row in short_breaks.iterrows():
            cands.append(("short", idx, row))
        cands.sort(key=lambda x: x[1])  # by timestamp

        trades_this_session = 0
        took_long = False
        took_short = False

        for direction, btime, breakout_row in cands:
            # Respect time window for breakout bar
            if not _clock_le(breakout_row.name, BREAKOUT_DEADLINE):
                continue
            if not ALLOW_BOTH_SIDES:
                if took_long and direction == "short":
                    break
                if took_short and direction == "long":
                    break
            if direction == "long" and took_long:
                continue
            if direction == "short" and took_short:
                continue

            cutoff = btime + timedelta(minutes=max_retest_min)
            window = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
            level = orh if direction == "long" else orl

            # Retest logic
            touch = window[(window["Low"] <= level) & (window["High"] >= level)].head(1)

            retest_used = True
            if touch.empty:
                # Continuation fallback
                if USE_CONTINUATION and _clock_le(breakout_row.name, CONTINUATION_CUTOFF):
                    retest_used = False
                    retest_time = breakout_row.name
                    retest_bar = breakout_row
                else:
                    continue
            else:
                retest_bar = touch.iloc[0]
                retest_time = retest_bar.name

            # Retest confirm close w/ cushion
            if retest_used and retest_confirm_close:
                if direction == "long":
                    ok = (retest_bar["Close"] >= level * (1 + BREAKOUT_CUSHION_PCT))
                else:
                    ok = (retest_bar["Close"] <= level * (1 - BREAKOUT_CUSHION_PCT))
                if not ok:
                    continue

            # Wick filter on entry bar (retest or continuation)
            if MAX_OPPOSITE_WICK_FRAC is not None:
                if wick_opposite_fraction(retest_bar, direction) > MAX_OPPOSITE_WICK_FRAC:
                    continue

            # Filters
            if USE_DOW_FILTER and retest_time.weekday() not in ALLOWED_DOW:
                continue

            if USE_VWAP_FILTER and "VWAP" in sdf.columns:
                v_prev = sdf.loc[:retest_time, "VWAP"].tail(3).dropna().values
                rising  = (len(v_prev) < 2) or np.all(np.diff(v_prev) >= 0)
                falling = (len(v_prev) < 2) or np.all(np.diff(v_prev) <= 0)
                if direction == "long":
                    if not (retest_bar["Close"] > retest_bar["VWAP"] and rising):
                        continue
                else:
                    if not (retest_bar["Close"] < retest_bar["VWAP"] and falling):
                        continue

            if USE_TREND_FILTER and (f"EMA{EMA_FAST}" in sdf.columns) and (f"EMA{EMA_SLOW}" in sdf.columns):
                emaf = retest_bar.get(f"EMA{EMA_FAST}", np.nan); emas = retest_bar.get(f"EMA{EMA_SLOW}", np.nan)
                if np.isfinite(emaf) and np.isfinite(emas):
                    if direction == "long" and not (emaf > emas): continue
                    if direction == "short" and not (emaf < emas): continue

            if USE_DAILY_TREND_FILTER and f"EMA_D{DAILY_EMA_FAST}" in sdf.columns and f"EMA_D{DAILY_EMA_SLOW}" in sdf.columns:
                dfast = float(sdf[f"EMA_D{DAILY_EMA_FAST}"].iloc[0])
                dslow = float(sdf[f"EMA_D{DAILY_EMA_SLOW}"].iloc[0])
                if direction == "long" and not (dfast > dslow): continue
                if direction == "short" and not (dfast < dslow): continue

            atr_col = f"ATR_D{DAILY_ATR_LEN}"
            if USE_OR_WIDTH_ATR_FILTER and atr_col in sdf.columns:
                or_width = float(orh - orl)
                atr_d = float(sdf[atr_col].iloc[0])
                if atr_d <= 0: continue
                frac = or_width / atr_d
                if not (OR_ATR_MIN <= frac <= OR_ATR_MAX): continue

            # Opening 15m volume quantile gate (if volume exists)
            if USE_OPENING_VOL_FILTER and "Open15_Vol_Thresh" in sdf.columns and "Open15_Vol" in sdf.columns:
                thresh = float(sdf["Open15_Vol_Thresh"].iloc[0]) if pd.notna(sdf["Open15_Vol_Thresh"].iloc[0]) else None
                open_vol = float(sdf["Open15_Vol"].iloc[0]) if pd.notna(sdf["Open15_Vol"].iloc[0]) else None
                if thresh is not None and open_vol is not None and open_vol < thresh:
                    continue

            if USE_RSI_FILTER and f"RSI{RSI_LEN}" in sdf.columns:
                rsi_val = float(retest_bar[f"RSI{RSI_LEN}"])
                if direction == "long" and not (rsi_val >= RSI_THRESH_LONG): continue
                if direction == "short" and not (rsi_val <= 100 - RSI_THRESH_SHORT): continue

            if USE_ADX_FILTER and f"ADX{ADX_LEN}" in sdf.columns:
                adx_val = float(retest_bar[f"ADX{ADX_LEN}"])
                if not (adx_val >= ADX_MIN): continue

            if USE_GAP_FILTER and "GapPct" in sdf.columns:
                gap = sdf.loc[sdf.index.min(), "GapPct"]
                if pd.notna(gap) and not (GAP_MIN <= abs(float(gap)) <= GAP_MAX):
                    continue

            if ALIGN_WITH_GAP_DIR and "GapPct" in sdf.columns:
                g = float(sdf["GapPct"].iloc[0]) if pd.notna(sdf["GapPct"].iloc[0]) else 0.0
                if (direction == "long" and g < 0) or (direction == "short" and g > 0):
                    continue

            if USE_SPY_CONFIRM and spy15 is not None and retest_time in spy15.index:
                spy_row = spy15.loc[retest_time]
                if direction == "long" and not (spy_row["SPY_Close"] > spy_row["SPY_VWAP"]): continue
                if direction == "short" and not (spy_row["SPY_Close"] < spy_row["SPY_VWAP"]): continue

            # Time windows for the entry bar
            if retest_used and (not _clock_le(retest_time, RETEST_DEADLINE)): continue
            if (not retest_used) and (not _clock_le(retest_time, CONTINUATION_CUTOFF)): continue

            # ===== Realistic Entry =====
            if retest_used and retest_confirm_close:
                if ENTRY_ON_CONFIRM.lower() == "next_open":
                    next_bar = sdf[sdf.index > retest_time].head(1)
                    if next_bar.empty:
                        continue
                    entry_time = next_bar.index[0]
                    raw_entry = float(next_bar["Open"].iloc[0])
                elif ENTRY_ON_CONFIRM.lower() == "confirm_close":
                    entry_time = retest_time
                    raw_entry = float(retest_bar["Close"])
                else:
                    entry_time = retest_time
                    raw_entry = float(retest_bar["Close"])
            else:
                # No confirm-close requirement: simulate a limit resting at level
                entry_time = retest_time
                raw_entry = float(level)

            entry = _apply_slippage(raw_entry, slippage_bps, "buy" if direction == "long" else "sell")
            stop  = orl if direction == "long" else orh
            rps   = abs(entry - stop)
            if rps <= 1e-12:
                continue

            qty = POSITION_SIZE_DOLLARS / max(entry, 1e-12)
            side_mult = 1 if direction == "long" else -1

            # Targets
            targets_r = TARGETS_R if USE_SCALE_OUT else r_targets
            targets = [(entry + r * rps) if direction == "long" else (entry - r * rps) for r in targets_r]

            # Walk-forward exits (start from entry_time)
            run = sdf[sdf.index >= entry_time].copy()
            qty_left = qty
            exits = []
            next_tp_idx = 0

            # Leg sizes (normalize to sum=1 across number of targets)
            if USE_SCALE_OUT and len(targets) >= 1:
                splits = _normalize_splits(len(targets), SCALE_SPLIT)
                leg_sizes = [qty * s for s in splits]
            else:
                leg_sizes = [qty]

            # BE arming on NEXT bar only after TP1
            be_level = entry
            be_pending = False
            be_set_time = None

            for ts, row in run.iterrows():
                hi, lo = float(row["High"]), float(row["Low"])

                # If BE was armed last bar, move stop now
                if be_pending and (not BE_ON_NEXT_BAR or (be_set_time is not None and ts > be_set_time)):
                    stop = be_level
                    be_pending = False

                # Optional ATR trail on remainder
                if USE_ATR_TRAIL and f"ATR15_{ATR15_LEN}" in sdf.columns and qty_left > 1e-9:
                    atr_now = float(row[f"ATR15_{ATR15_LEN}"])
                    if direction == "long":
                        stop = max(stop, hi - ATR15_MULT * atr_now)
                    else:
                        stop = min(stop, lo + ATR15_MULT * atr_now)

                # Stop first
                if lo <= stop <= hi:
                    px = _apply_slippage(stop, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                    label = "breakeven" if abs(px - be_level) < 1e-10 else "stop"
                    exits.append((label, ts, px, qty_left))
                    qty_left = 0.0
                    break

                # Targets (sequential)
                if next_tp_idx < len(targets):
                    tp = targets[next_tp_idx]
                    if lo <= tp <= hi:
                        px = _apply_slippage(tp, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                        fill_qty = leg_sizes[next_tp_idx] if next_tp_idx < len(leg_sizes) else qty_left
                        exits.append((f"tp{next_tp_idx+1}", ts, px, fill_qty))
                        qty_left -= fill_qty
                        # Arm BE after TP1 (==1R) – to be applied next bar
                        if next_tp_idx == 0 and MOVE_STOP_TO_BE_AT_R is not None:
                            be_pending = True
                            be_set_time = ts
                        next_tp_idx += 1
                        if qty_left <= 1e-9:
                            break

                # If no TP1 by continuation cutoff, tighten to BE
                if USE_CONTINUATION and not _clock_le(ts, CONTINUATION_CUTOFF) and next_tp_idx == 0:
                    stop = be_level

            # EOD exit for any leftover
            if qty_left > 1e-9:
                last = run.iloc[-1]
                px = _apply_slippage(float(last["Close"]), SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                exits.append(("eod", run.index[-1], px, qty_left))
                qty_left = 0.0

            # P&L + fees
            cash_pnl = sum((px - entry) * side_mult * q for (_, _, px, q) in exits)
            if FEE_PER_FILL:
                cash_pnl -= FEES_PER_TRADE * len(exits)
            else:
                cash_pnl -= FEES_PER_TRADE

            trades.append({
                "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
                "Session": ses,
                "Direction": direction,
                "ORH": orh, "ORL": orl,
                "EntryTime": entry_time,
                "Entry": entry, "Stop": stop,
                "Targets": targets,
                "Exits": [(lab, ts, px, q) for (lab, ts, px, q) in exits],
                "Qty": qty,
                "PnL_$": cash_pnl,
                "R_multiple": cash_pnl / (rps * max(qty, 1e-12))
            })

            # mark side taken
            if direction == "long":  took_long  = True
            if direction == "short": took_short = True
            trades_this_session += 1
            if trades_this_session >= MAX_TRADES_PER_SESSION:
                break

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Robust parquet loader (Windows/OneDrive-friendly)
# =============================

def _read_parquet_robust(path: str, max_retries: int = 3, sleep_sec: float = 0.6) -> pd.DataFrame:
    """
    Robust parquet reader for Windows + OneDrive + Spark/Arrow directory datasets:
    - If `path` is a *directory* (e.g., AMD.parquet/ with part files), read via pyarrow.dataset.
    - If `path` is a *single file*, try pandas+pyarrow, then memory_map, then rb+pq.read_table.
    Retries between attempts give OneDrive time to hydrate files.
    """
    import time
    from pathlib import Path
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pyarrow.dataset as ds

    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {path}")

    last_err = None

    # ---- Directory-style dataset (Spark/Arrow) ----
    if p.is_dir():
        for attempt in range(1, max_retries + 1):
            try:
                dataset = ds.dataset(str(p), format="parquet")
                table = dataset.to_table()
                return table.to_pandas()
            except Exception as e:
                last_err = e
                if attempt < max_retries:
                    time.sleep(sleep_sec)
        raise OSError(
            f"Failed to read parquet dataset directory after retries: {path}\nLast error: {last_err}\n"
            "Tip: ensure the folder and its part files are 'Available offline' in OneDrive."
        )

    # ---- Single parquet file ----
    for attempt in range(1, max_retries + 1):
        # 1) pandas + pyarrow
        try:
            return pd.read_parquet(str(p), engine="pyarrow")
        except Exception as e:
            last_err = e
            time.sleep(sleep_sec)

        # 2) memory-map parquet
        try:
            with pa.memory_map(str(p), 'r') as source:
                pf = pq.ParquetFile(source)
                table = pf.read()
            return table.to_pandas()
        except Exception as e:
            last_err = e
            time.sleep(sleep_sec)

        # 3) raw rb handle
        try:
            with open(str(p), "rb") as f:
                table = pq.read_table(f)
            return table.to_pandas()
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                time.sleep(sleep_sec)

    hint = (
        "If this is a OneDrive cloud-only placeholder, mark it 'Always keep on this device' and try again."
    )
    raise OSError(f"Failed to read parquet file after retries: {path}\nLast error: {last_err}\n{hint}")
STD_MAP = {"open":"Open","high":"High","low":"Low","close":"Close","volume":"Volume"}

def _standardize_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    lower = {c.lower(): c for c in df.columns}
    for k, std in STD_MAP.items():
        if k in lower:                rename[lower[k]] = std
        elif k.capitalize() in df.columns: rename[k.capitalize()] = std
        elif k.upper() in df.columns: rename[k.upper()] = std
    out = df.rename(columns=rename)
    drop_cols = [c for c in out.columns if str(c).lower().startswith("adj")]
    return out.drop(columns=drop_cols, errors="ignore")

def _choose_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for cand in ["EventAt","Datetime","datetime","Timestamp","timestamp","Date","date","Time","time"]:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            df = df.set_index(cand)
            break
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No datetime index/column found.")
    return df

def _maybe_resample_to_15m(df: pd.DataFrame) -> pd.DataFrame:
    if len(df.index) < 3:
        return df
    deltas = (df.index[1:] - df.index[:-1]).asi8
    if len(deltas) == 0:
        return df
    med_ns = np.median(deltas)
    fifteen_ns = pd.Timedelta("15T").value
    if med_ns < fifteen_ns:
        o = df["Open"].resample("15T", label="left", closed="left").first()
        h = df["High"].resample("15T", label="left", closed="left").max()
        l = df["Low"].resample("15T", label="left", closed="left").min()
        c = df["Close"].resample("15T", label="left", closed="left").last()
        v = df["Volume"].resample("15T", label="left", closed="left").sum() if "Volume" in df.columns else None
        parts = {"Open":o,"High":h,"Low":l,"Close":c}
        if v is not None:
            parts["Volume"] = v
        out = pd.concat(parts, axis=1).dropna(subset=["Open","High","Low","Close"])
        return out
    return df

def load_parquet_15m(path: str, tz: str = "America/New_York",
                     assume_naive_is_utc: bool = ASSUME_NAIVE_TIMESTAMPS_ARE_UTC,
                     reg_start: str = REG_SESSION_START, reg_end: str = REG_SESSION_END) -> pd.DataFrame:
    df = _read_parquet_robust(path)
    df = _choose_dt_index(df).sort_index()

    # Timezone handling
    if df.index.tz is None:
        if assume_naive_is_utc:
            df = df.tz_localize("UTC").tz_convert(tz)
        else:
            df = df.tz_localize(tz)
    else:
        df = df.tz_convert(tz)

    # Standardize columns and resample if needed
    df = _standardize_ohlcv_columns(df)
    for need in ["Open","High","Low","Close"]:
        if need not in df.columns:
            raise ValueError(f"{os.path.basename(path)} missing required column: {need}")
    df = _maybe_resample_to_15m(df)

    # Regular hours only
    df = df.between_time(reg_start, reg_end)
    return df.sort_index()

# =============================
# Runner
# =============================

def run_folder(folder: str):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"Found {len(files)} parquet files.")
    all_trades, all_equity = [], []
    processed = 0

    for i, fpath in enumerate(files, 1):
        ticker = os.path.splitext(os.path.basename(fpath))[0].upper().replace("_","").replace("-","")
        print(f"\n== {ticker} ({i}/{len(files)}) ==")
        try:
            df15 = load_parquet_15m(
                fpath,
                tz=TZ,
                assume_naive_is_utc=ASSUME_NAIVE_TIMESTAMPS_ARE_UTC,
                reg_start=REG_SESSION_START,
                reg_end=REG_SESSION_END
            )
            df15["Ticker"] = ticker
        except Exception as e:
            print(f"Skipping {ticker}: {e}")
            continue

        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            r_targets=TARGETS_R,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=SLIPPAGE_BPS,
            fees=FEES_PER_TRADE,
            retest_confirm_close=RETEST_CONFIRM_CLOSE,
            spy15=None  # not used here
        )

        if not tlog.empty:
            # For CSV: stringify lists of exits/targets
            tlog_out = tlog.copy()
            tlog_out["Targets"] = tlog_out["Targets"].apply(lambda xs: ";".join([f"{p:.6f}" for p in xs]))
            tlog_out["Exits"]   = tlog_out["Exits"].apply(lambda xs: ";".join([f"{t[0]}|{t[1]}|{t[2]:.6f}|{t[3]:.6f}" for t in xs]))
            all_trades.append(tlog_out)

        if not eq.empty:
            eq_out = eq.copy(); eq_out["Ticker"] = ticker
            all_equity.append(eq_out)

        processed += 1
        if AUTOSAVE_EVERY and processed % AUTOSAVE_EVERY == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

        time.sleep(SLEEP_BETWEEN_TICKERS)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

def main():
    trades, equity = run_folder(PARQUET_DIR)

    # Overall totals + write CSVs
    print_overall_totals(trades)

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

    # Per-ticker summary (with WinRate & PF)
    if not trades.empty:
        s = (
            trades.groupby("Ticker")["PnL_$"]
                  .agg(['count', 'sum', 'mean', 'std'])
                  .rename(columns={'count':'Trades','sum':'PnL_sum','mean':'AvgPnL','std':'StdPnL'})
                  .reset_index()
                  .sort_values("PnL_sum", ascending=False)
        )
        wr = trades.groupby("Ticker")["PnL_$"].apply(lambda x: (pd.to_numeric(x, errors="coerce") > 0).mean()*100.0).rename("WinRate_%")
        gp = trades.groupby("Ticker")["PnL_$"].apply(lambda x: pd.to_numeric(x, errors="coerce").clip(lower=0).sum()).rename("GrossProfit")
        gl = trades.groupby("Ticker")["PnL_$"].apply(lambda x: -pd.to_numeric(x, errors="coerce").clip(upper=0).sum()).rename("GrossLoss")
        pf = (gp / gl.replace(0, np.nan)).rename("PF")
        summary = s.merge(wr, on="Ticker", how="left").merge(pf, on="Ticker", how="left")
        print("\n=== Summary by Ticker (top 50) ===")
        print(summary.head(50).to_string(index=False))
        summary.to_csv(SUMMARY_CSV, index=False)
        print(f"Saved summary: {SUMMARY_CSV}")
    else:
        print("No trades generated with current settings.")

if __name__ == "__main__":
    main()


Found 20 parquet files.

== AAPL (1/20) ==

== AMD (2/20) ==
Skipping AMD: Path not found: C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMD.parquet

== AMZN (3/20) ==
Skipping AMZN: Path not found: C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMZN.parquet

== CAT (4/20) ==
Skipping CAT: Path not found: C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CAT.parquet

== CHWY (5/20) ==
Skipping CHWY: Path not found: C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CHWY.parquet

== CVNA (6/20) ==
Skipping CVNA: Path not found: C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CVNA.parquet

== GLD (7/20) ==
Skipping GLD: Path not found: C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GLD.parquet

== GOOGL (8/20) ==
Skipping GOOGL: Path not found: C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GOOGL.parquet

== GS (9/20) ==

== JPM (10/20) ==
Skipping JPM: Path not found: C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\

In [22]:
# orb_15m_retest_from_parquet_batch.py
# Reads all .parquet files in PARQUET_DIR and runs the ORB 15m Retest/Continuation strategy
# Uses your requested logic: TP1=1R, TP2=2R, TP3=2.5R in thirds, BE on the next bar after TP1.

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import time
from pathlib import Path
from datetime import timedelta, time as dtime
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype, is_datetime64tz_dtype

# =============================
# USER SETTINGS
# =============================
PARQUET_DIR = r"C:\Users\pcagm\Downloads\parquet"   # <— your folder (matches your screenshot)

# Session / timezone
LOCAL_TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# Timestamps in your parquet are usually UTC; flip to False if they’re already ET.
ASSUME_EVENTAT_IS_UTC = True

# =============================
# STRATEGY PARAMETERS (exactly as you specified)
# =============================

MAX_RETEST_MIN = 120
RETEST_CONFIRM_CLOSE = True
BREAKOUT_CUSHION_PCT = 0.0005
MAX_OPPOSITE_WICK_FRAC = 0.40

MAX_TRADES_PER_SESSION = 2
ALLOW_BOTH_SIDES = True

USE_SCALE_OUT = True
TARGETS_R = [1.0, 2.0, 2.5]
SCALE_SPLIT = (1/3, 1/3, 1/3)

MOVE_STOP_TO_BE_AT_R = 1.0
BE_ON_NEXT_BAR = True

USE_CONTINUATION = True

BREAKOUT_DEADLINE   = "11:00"
RETEST_DEADLINE     = "12:00"
CONTINUATION_CUTOFF = "12:00"

USE_VWAP_FILTER   = True
USE_TREND_FILTER  = True
EMA_FAST = 20
EMA_SLOW = 50

USE_RSI_FILTER = True
RSI_LEN = 14
RSI_THRESH_LONG = 50.0
RSI_THRESH_SHORT = 50.0

USE_ADX_FILTER = False
ADX_LEN = 14
ADX_MIN = 18.0

USE_DAILY_TREND_FILTER = True
DAILY_EMA_FAST = 20
DAILY_EMA_SLOW = 50
DAILY_ATR_LEN  = 14

USE_OR_WIDTH_ATR_FILTER = True
OR_ATR_MIN = 0.10
OR_ATR_MAX = 2.00

USE_OPENING_VOL_FILTER = True
OPENING_VOL_LOOKBACK = 60
OPENING_VOL_QUANTILE = 0.30

USE_GAP_FILTER = True
GAP_MIN = 0.003
GAP_MAX = 0.04
ALIGN_WITH_GAP_DIR = False

USE_DOW_FILTER = False
ALLOWED_DOW = {1, 2, 3}

ENTRY_ON_CONFIRM = "next_open"   # or "confirm_close"

POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00
FEE_PER_FILL = True  # per exit leg

USE_ATR_TRAIL = False
ATR15_LEN = 14
ATR15_MULT = 2.0

TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"
SUMMARY_CSV    = "orb_summary_sp500.csv"

# =============================
# I/O helpers
# =============================

def _read_parquet_robust(path: Path, max_retries: int = 3, sleep_sec: float = 0.6, debug_listdir_once=[True]) -> pd.DataFrame:
    """
    Read either a single parquet file or a directory-style parquet dataset.
    No early exists() check — we just try multiple methods.
    """
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pyarrow.dataset as ds

    last_err = None
    pstr = str(path)

    # Directory-style dataset (folder named *.parquet with part-*.parquet files)
    if os.path.isdir(pstr):
        for attempt in range(1, max_retries + 1):
            try:
                dataset = ds.dataset(pstr, format="parquet")
                table = dataset.to_table()
                return table.to_pandas()
            except Exception as e:
                last_err = e
                time.sleep(sleep_sec)
        raise OSError(f"Failed to read parquet DATASET directory after retries: {pstr}\nLast error: {last_err}")

    # Single .parquet file
    for attempt in range(1, max_retries + 1):
        try:
            return pd.read_parquet(pstr, engine="pyarrow")
        except Exception as e:
            last_err = e
            time.sleep(sleep_sec)

        try:
            with pa.memory_map(pstr, 'r') as source:
                pf = pq.ParquetFile(source)
                table = pf.read()
            return table.to_pandas()
        except Exception as e:
            last_err = e
            time.sleep(sleep_sec)

        try:
            with open(pstr, "rb") as f:
                table = pq.read_table(f)
            return table.to_pandas()
        except Exception as e:
            last_err = e
            time.sleep(sleep_sec)

    # Debug visibility of the directory once, if it *looks* like the entry vanished
    if debug_listdir_once[0]:
        debug_listdir_once[0] = False
        parent = path.parent
        try:
            print(f"[debug] os.path.exists({pstr}) -> {os.path.exists(pstr)} | isfile={os.path.isfile(pstr)} | isdir={os.path.isdir(pstr)}")
            print(f"[debug] Listing of {parent}:")
            for name in os.listdir(parent):
                print(f"    - {name}")
        except Exception as e:
            print(f"[debug] Could not list {parent}: {e}")

    raise OSError(f"Failed to read parquet FILE after retries: {pstr}\nLast error: {last_err}")

def _first_present(d: pd.DataFrame, names: List[str]) -> Optional[str]:
    for n in names:
        if n in d.columns:
            return n
    return None

def load_symbol_parquet(file_path: Path, ticker_from_name: str) -> pd.DataFrame:
    df = _read_parquet_robust(file_path)

    # Normalize names
    df.columns = [str(c).strip() for c in df.columns]

    # Time column
    ts_col = _first_present(df, ["EventAt", "Datetime", "DateTime", "Timestamp", "ts", "date", "time"])
    if ts_col is None:
        raise ValueError(f"{file_path.name}: Could not find a timestamp column. Columns: {list(df.columns)[:15]}")

    # Ensure datetime
    if not is_datetime64_any_dtype(df[ts_col]):
        df[ts_col] = pd.to_datetime(df[ts_col], utc=False, errors="coerce")

    # TZ
    if is_datetime64tz_dtype(df[ts_col]):
        df[ts_col] = df[ts_col].dt.tz_convert(LOCAL_TZ)
    else:
        if ASSUME_EVENTAT_IS_UTC:
            df[ts_col] = df[ts_col].dt.tz_localize("UTC").dt.tz_convert(LOCAL_TZ)
        else:
            df[ts_col] = df[ts_col].dt.tz_localize(LOCAL_TZ)

    # Symbol
    sym_col = _first_present(df, ["symbol", "Symbol", "ticker", "Ticker"])
    if sym_col is not None:
        def _clean_sym(x):
            if isinstance(x, (bytes, bytearray)):
                try:
                    x = x.decode("utf-8")
                except Exception:
                    x = str(x)
            return str(x).strip().strip("\"'").replace("b'", "").replace("b\"", "")
        df["symbol"] = df[sym_col].apply(_clean_sym)
    else:
        df["symbol"] = ticker_from_name

    # Restrict to ticker (if multi-symbol file)
    sub = df[df["symbol"].str.upper() == ticker_from_name.upper()].copy()
    if sub.empty:
        sub = df.copy()
        sub["symbol"] = ticker_from_name

    # Columns
    oc = _first_present(sub, ["Open", "open", "o"])
    hc = _first_present(sub, ["High", "high", "h"])
    lc = _first_present(sub, ["Low", "low", "l"])
    cc = _first_present(sub, ["Close", "close", "c"])
    vc = _first_present(sub, ["Volume", "volume", "v"])

    for need, name in zip(["Open","High","Low","Close"], [oc,hc,lc,cc]):
        if name is None:
            raise ValueError(f"{file_path.name}: Missing required price column for {need}.")

    keep_map = {"EventAt": ts_col, "Open": oc, "High": hc, "Low": lc, "Close": cc}
    if vc is not None:
        keep_map["Volume"] = vc

    out = sub[list(keep_map.values()) + ["symbol"]].rename(columns={v:k for k,v in keep_map.items()})
    out = out.sort_values("EventAt").set_index("EventAt")
    return out

# =============================
# Indicators & helpers
# =============================

def ensure_unique_columns(df: pd.DataFrame) -> pd.DataFrame:
    if getattr(df.columns, "duplicated", None) is not None and df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def sessionize(obj) -> pd.Series:
    idx = getattr(obj, "index", obj)
    return pd.to_datetime(pd.Index(idx).date)

def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    for c in ("ORH","ORL"):
        if (df.columns == c).sum() > 0:
            df = df.drop(columns=[c])
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df["ORH"] = np.nan
    df["ORL"] = np.nan
    df.loc[first_bar_idx, "ORH"] = df.loc[first_bar_idx, "High"].astype(float).values
    df.loc[first_bar_idx, "ORL"] = df.loc[first_bar_idx, "Low"].astype(float).values
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return ensure_unique_columns(df)

def add_session_vwap(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df["Session"] = sessionize(df)
    tp = (df["High"] + df["Low"] + df["Close"]) / 3.0
    if "Volume" in df.columns:
        df["vwap_num"] = tp * df["Volume"]
        df["vwap_den"] = df["Volume"].replace(0, np.nan)
    else:
        df["vwap_num"] = tp
        df["vwap_den"] = 1.0
    df["VWAP"] = (df.groupby("Session")["vwap_num"].cumsum() /
                  df.groupby("Session")["vwap_den"].cumsum())
    return ensure_unique_columns(df.drop(columns=["vwap_num","vwap_den"]))

def add_ema_trend(df_15: pd.DataFrame, fast=20, slow=50) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"EMA{fast}"] = df["Close"].ewm(span=fast, adjust=False).mean()
    df[f"EMA{slow}"] = df["Close"].ewm(span=slow, adjust=False).mean()
    return ensure_unique_columns(df)

def rsi(series: pd.Series, length=14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0)
    dn = -delta.clip(upper=0)
    avg_gain = up.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = dn.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan))
    return 100 - (100 / (1 + rs))

def add_rsi(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"RSI{length}"] = rsi(df["Close"], length)
    return ensure_unique_columns(df)

def true_range(high, low, prev_close):
    return np.maximum.reduce([
        (high - low).values,
        np.abs(high - prev_close).values,
        np.abs(low - prev_close).values
    ])

def add_adx(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    up_move = df["High"].diff()
    dn_move = -df["Low"].diff()
    plus_dm  = np.where((up_move > dn_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((dn_move > up_move) & (dn_move > 0), dn_move, 0.0)

    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    atr = tr.ewm(alpha=1/length, adjust=False).mean()

    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    dx = (abs(plus_di - minus_di) / (plus_di + minus_di).replace(0, np.nan)) * 100
    adx = dx.ewm(alpha=1/length, adjust=False).mean()
    df[f"ADX{length}"] = adx
    return ensure_unique_columns(df)

def build_daily_from_15m(df_15: pd.DataFrame) -> pd.DataFrame:
    d = ensure_unique_columns(df_15.copy())
    d["Session"] = sessionize(d)
    daily = d.groupby("Session").agg(
        Open=("Open","first"),
        High=("High","max"),
        Low=("Low","min"),
        Close=("Close","last"),
        Volume=("Volume","sum") if "Volume" in d.columns else ("Close","size"),
    )
    return daily

def add_daily_bias_and_atr(df_15: pd.DataFrame, ema_fast=20, ema_slow=50, atr_len=14) -> pd.DataFrame:
    dly = build_daily_from_15m(df_15)
    dly[f"EMA_D{ema_fast}"] = dly["Close"].ewm(span=ema_fast, adjust=False).mean()
    dly[f"EMA_D{ema_slow}"] = dly["Close"].ewm(span=ema_slow, adjust=False).mean()

    prev_close = dly["Close"].shift(1)
    tr = pd.Series(np.maximum.reduce([
        (dly["High"] - dly["Low"]).values,
        np.abs(dly["High"] - prev_close).values,
        np.abs(dly["Low"]  - prev_close).values
    ]), index=dly.index)
    atr_col = f"ATR_D{atr_len}"
    dly[atr_col] = tr.ewm(alpha=1/atr_len, adjust=False).mean()

    if "Volume" in df_15.columns:
        tmp = df_15.copy()
        tmp["Session"] = sessionize(tmp)
        open15_vol = tmp.groupby("Session")["Volume"].first()
        dly["Open15_Vol"] = open15_vol
        dly["Open15_Vol_Thresh"] = (
            dly["Open15_Vol"]
            .rolling(window=OPENING_VOL_LOOKBACK, min_periods=10)
            .quantile(OPENING_VOL_QUANTILE)
        )

    feature_cols = [f"EMA_D{ema_fast}", f"EMA_D{ema_slow}", atr_col, "Open15_Vol", "Open15_Vol_Thresh"]
    feature_cols = [c for c in feature_cols if c in dly.columns]

    out = ensure_unique_columns(df_15.copy())
    out["Session"] = sessionize(out)
    out = out.merge(dly[feature_cols], left_on="Session", right_index=True, how="left")
    return ensure_unique_columns(out)

def wick_opposite_fraction(row: pd.Series, direction: str) -> float:
    h, l, o, c = float(row["High"]), float(row["Low"]), float(row.get("Open", np.nan)), float(row["Close"])
    rng = max(h - l, 1e-12)
    if direction == "long":
        opp = min(o, c) - l
    else:
        opp = h - max(o, c)
    return float(max(opp, 0.0) / rng)

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def _normalize_splits(num_targets: int, splits: tuple) -> List[float]:
    if num_targets <= 0:
        return [1.0]
    if not splits:
        return [1.0 / num_targets] * num_targets
    arr = list(splits[:num_targets])
    if len(arr) < num_targets:
        arr += [0.0] * (num_targets - len(arr))
    s = sum(arr)
    if s <= 0:
        return [1.0 / num_targets] * num_targets
    return [x / s for x in arr]

def _to_clock(ts) -> dtime:
    return dtime(ts.hour, ts.minute)

def _clock_le(ts, hhmm: str) -> bool:
    h, m = map(int, hhmm.split(":"))
    return _to_clock(ts) <= dtime(h, m)

def to_15m(df: pd.DataFrame) -> pd.DataFrame:
    o = df["Open"].resample("15T", label="left", closed="left").first()
    h = df["High"].resample("15T", label="left", closed="left").max()
    l = df["Low"].resample("15T", label="left", closed="left").min()
    c = df["Close"].resample("15T", label="left", closed="left").last()
    if "Volume" in df.columns:
        v = df["Volume"].resample("15T", label="left", closed="left").sum()
        out = pd.concat({"Open": o, "High": h, "Low": l, "Close": c, "Volume": v}, axis=1)
    else:
        out = pd.concat({"Open": o, "High": h, "Low": l, "Close": c}, axis=1)
    out = out.dropna(how="any")
    out["Ticker"] = df["symbol"].iloc[0] if "symbol" in df.columns and len(df) else ""
    return out.between_time(REG_SESSION_START, REG_SESSION_END)

# =============================
# Strategy (per ticker)
# =============================

def backtest_orb_retest(df_15: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = compute_opening_range(df_15)
    if USE_VWAP_FILTER:  df = add_session_vwap(df)
    if USE_TREND_FILTER: df = add_ema_trend(df, EMA_FAST, EMA_SLOW)
    df = add_daily_bias_and_atr(df, DAILY_EMA_FAST, DAILY_EMA_SLOW, DAILY_ATR_LEN)

    df["Session"] = sessionize(df)
    first_idx = df.groupby("Session").head(1).index
    prev_close_series = df.groupby("Session")["Close"].last().shift(1)
    prev_close_map = df["Session"].map(prev_close_series)
    df.loc[first_idx, "GapPct"] = (df.loc[first_idx, "Open"] / prev_close_map.loc[first_idx] - 1.0)
    df["GapPct"] = df.groupby("Session")["GapPct"].ffill()

    if USE_RSI_FILTER:  df = add_rsi(df, RSI_LEN)
    if USE_ADX_FILTER:  df = add_adx(df, ADX_LEN)
    if USE_ATR_TRAIL:   df = add_adx(df, ATR15_LEN)  # ATR15 calc omitted unless enabled
    df = ensure_unique_columns(df)

    trades = []
    for ses in df["Session"].unique():
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh, orl = float(sdf["ORH"].iloc[0]), float(sdf["ORL"].iloc[0])
        if np.isnan(orh) or np.isnan(orl):
            continue

        after_open = sdf.iloc[1:].copy()
        long_breaks  = after_open[after_open["Close"] > orh * (1 + BREAKOUT_CUSHION_PCT)]
        short_breaks = after_open[after_open["Close"] < orl * (1 - BREAKOUT_CUSHION_PCT)]

        cands = []
        for idx, row in long_breaks.iterrows():
            cands.append(("long", idx, row))
        for idx, row in short_breaks.iterrows():
            cands.append(("short", idx, row))
        cands.sort(key=lambda x: x[1])

        trades_this_session = 0
        took_long = took_short = False

        for direction, btime, breakout_row in cands:
            if not _clock_le(breakout_row.name, BREAKOUT_DEADLINE):
                continue
            if not ALLOW_BOTH_SIDES:
                if took_long and direction == "short": break
                if took_short and direction == "long": break
            if direction == "long" and took_long:   continue
            if direction == "short" and took_short: continue

            cutoff = btime + timedelta(minutes=MAX_RETEST_MIN)
            window = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
            level = orh if direction == "long" else orl

            touch = window[(window["Low"] <= level) & (window["High"] >= level)].head(1)

            retest_used = True
            if touch.empty:
                if USE_CONTINUATION and _clock_le(breakout_row.name, CONTINUATION_CUTOFF):
                    retest_used = False
                    retest_time = breakout_row.name
                    retest_bar = breakout_row
                else:
                    continue
            else:
                retest_bar = touch.iloc[0]
                retest_time = retest_bar.name

            if retest_used and RETEST_CONFIRM_CLOSE:
                if direction == "long":
                    ok = (retest_bar["Close"] >= level * (1 + BREAKOUT_CUSHION_PCT))
                else:
                    ok = (retest_bar["Close"] <= level * (1 - BREAKOUT_CUSHION_PCT))
                if not ok:
                    continue

            if MAX_OPPOSITE_WICK_FRAC is not None:
                if wick_opposite_fraction(retest_bar, direction) > MAX_OPPOSITE_WICK_FRAC:
                    continue

            if USE_DOW_FILTER and retest_time.weekday() not in ALLOWED_DOW:
                continue

            if USE_VWAP_FILTER and "VWAP" in sdf.columns:
                v_prev = sdf.loc[:retest_time, "VWAP"].tail(3).dropna().values
                rising  = (len(v_prev) < 2) or np.all(np.diff(v_prev) >= 0)
                falling = (len(v_prev) < 2) or np.all(np.diff(v_prev) <= 0)
                if direction == "long":
                    if not (retest_bar["Close"] > retest_bar["VWAP"] and rising):  continue
                else:
                    if not (retest_bar["Close"] < retest_bar["VWAP"] and falling): continue

            if USE_TREND_FILTER and (f"EMA{EMA_FAST}" in sdf.columns) and (f"EMA{EMA_SLOW}" in sdf.columns):
                emaf = retest_bar.get(f"EMA{EMA_FAST}", np.nan); emas = retest_bar.get(f"EMA{EMA_SLOW}", np.nan)
                if np.isfinite(emaf) and np.isfinite(emas):
                    if direction == "long" and not (emaf > emas): continue
                    if direction == "short" and not (emaf < emas): continue

            if USE_DAILY_TREND_FILTER and f"EMA_D{DAILY_EMA_FAST}" in sdf.columns and f"EMA_D{DAILY_EMA_SLOW}" in sdf.columns:
                dfast = float(sdf[f"EMA_D{DAILY_EMA_FAST}"].iloc[0])
                dslow = float(sdf[f"EMA_D{DAILY_EMA_SLOW}"].iloc[0])
                if direction == "long" and not (dfast > dslow): continue
                if direction == "short" and not (dfast < dslow): continue

            atr_col = f"ATR_D{DAILY_ATR_LEN}"
            if USE_OR_WIDTH_ATR_FILTER and atr_col in sdf.columns:
                or_width = float(orh - orl)
                atr_d = float(sdf[atr_col].iloc[0])
                if atr_d <= 0: continue
                frac = or_width / atr_d
                if not (OR_ATR_MIN <= frac <= OR_ATR_MAX): continue

            if USE_RSI_FILTER and f"RSI{RSI_LEN}" in sdf.columns:
                rsi_val = float(retest_bar[f"RSI{RSI_LEN}"])
                if direction == "long" and not (rsi_val >= RSI_THRESH_LONG): continue
                if direction == "short" and not (rsi_val <= 100 - RSI_THRESH_SHORT): continue

            if USE_ADX_FILTER and f"ADX{ADX_LEN}" in sdf.columns:
                adx_val = float(retest_bar[f"ADX{ADX_LEN}"])
                if not (adx_val >= ADX_MIN): continue

            if USE_GAP_FILTER and "GapPct" in sdf.columns:
                gap = sdf.loc[sdf.index.min(), "GapPct"]
                if pd.notna(gap) and not (GAP_MIN <= abs(float(gap)) <= GAP_MAX): continue

            if ALIGN_WITH_GAP_DIR and "GapPct" in sdf.columns:
                g = float(sdf["GapPct"].iloc[0]) if pd.notna(sdf["GapPct"].iloc[0]) else 0.0
                if (direction == "long" and g < 0) or (direction == "short" and g > 0): continue

            if retest_used and (not _clock_le(retest_time, RETEST_DEADLINE)): continue
            if (not retest_used) and (not _clock_le(retest_time, CONTINUATION_CUTOFF)): continue

            # Entry handling
            if retest_used and RETEST_CONFIRM_CLOSE:
                if ENTRY_ON_CONFIRM.lower() == "next_open":
                    next_bar = sdf[sdf.index > retest_time].head(1)
                    if next_bar.empty: continue
                    entry_time = next_bar.index[0]
                    raw_entry = float(next_bar["Open"].iloc[0])
                elif ENTRY_ON_CONFIRM.lower() == "confirm_close":
                    entry_time = retest_time
                    raw_entry = float(retest_bar["Close"])
                else:
                    entry_time = retest_time
                    raw_entry = float(retest_bar["Close"])
            else:
                entry_time = retest_time
                raw_entry = float(level)

            entry = _apply_slippage(raw_entry, SLIPPAGE_BPS, "buy" if direction == "long" else "sell")
            stop  = orl if direction == "long" else orh
            rps   = abs(entry - stop)
            if rps <= 1e-12: continue

            qty = POSITION_SIZE_DOLLARS / max(entry, 1e-12)
            side_mult = 1 if direction == "long" else -1
            targets = [(entry + r * rps) if direction == "long" else (entry - r * rps) for r in TARGETS_R]

            run = sdf[sdf.index >= entry_time].copy()
            qty_left = qty
            exits = []
            next_tp_idx = 0

            splits = _normalize_splits(len(targets), SCALE_SPLIT)
            leg_sizes = [qty * s for s in splits]

            be_level = entry
            be_pending = False
            be_set_time = None

            for ts, row in run.iterrows():
                hi, lo = float(row["High"]), float(row["Low"])

                if be_pending and (not BE_ON_NEXT_BAR or (be_set_time is not None and ts > be_set_time)):
                    stop = be_level
                    be_pending = False

                if USE_ATR_TRAIL and f"ATR15_{ATR15_LEN}" in sdf.columns and qty_left > 1e-9:
                    atr_now = float(row[f"ATR15_{ATR15_LEN}"])
                    if direction == "long":
                        stop = max(stop, hi - ATR15_MULT * atr_now)
                    else:
                        stop = min(stop, lo + ATR15_MULT * atr_now)

                if lo <= stop <= hi:
                    px = _apply_slippage(stop, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                    label = "breakeven" if abs(px - be_level) < 1e-10 else "stop"
                    exits.append((label, ts, px, qty_left))
                    qty_left = 0.0
                    break

                if next_tp_idx < len(targets):
                    tp = targets[next_tp_idx]
                    if lo <= tp <= hi:
                        px = _apply_slippage(tp, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                        fill_qty = leg_sizes[next_tp_idx] if next_tp_idx < len(leg_sizes) else qty_left
                        exits.append((f"tp{next_tp_idx+1}", ts, px, fill_qty))
                        qty_left -= fill_qty
                        if next_tp_idx == 0 and MOVE_STOP_TO_BE_AT_R is not None:
                            be_pending = True
                            be_set_time = ts
                        next_tp_idx += 1
                        if qty_left <= 1e-9:
                            break

                if USE_CONTINUATION and not _clock_le(ts, CONTINUATION_CUTOFF) and next_tp_idx == 0:
                    stop = be_level

            if qty_left > 1e-9:
                last = run.iloc[-1]
                px = _apply_slippage(float(last["Close"]), SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                exits.append(("eod", run.index[-1], px, qty_left))
                qty_left = 0.0

            cash_pnl = sum((px - entry) * side_mult * q for (_, _, px, q) in exits)
            cash_pnl -= (FEES_PER_TRADE * len(exits)) if FEE_PER_FILL else FEES_PER_TRADE

            trades.append({
                "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
                "Session": ses,
                "Direction": direction,
                "ORH": orh, "ORL": orl,
                "EntryTime": entry_time,
                "Entry": entry, "Stop": stop,
                "Targets": targets,
                "Exits": [(lab, ts, px, q) for (lab, ts, px, q) in exits],
                "Qty": qty,
                "PnL_$": cash_pnl,
                "R_multiple": cash_pnl / (rps * max(qty, 1e-12))
            })

            if direction == "long":  took_long  = True
            if direction == "short": took_short = True
            trades_this_session += 1
            if trades_this_session >= MAX_TRADES_PER_SESSION:
                break

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Batch runner
# =============================

def main():
    base = Path(PARQUET_DIR).expanduser()
    print(f"Reading from PARQUET_DIR: {base}")
    if not os.path.isdir(str(base)):
        raise FileNotFoundError(f"Folder does not exist: {base}")

    # Find both single-file .parquet and dataset directories that end with .parquet
    files = [Path(p) for p in glob.glob(os.path.join(str(base), "*.parquet"))]
    files.sort(key=lambda p: p.name.lower())
    print(f"Found {len(files)} parquet files/datasets.\n")

    all_trades, all_equity = [], []
    for i, path in enumerate(files, start=1):
        ticker = path.stem.upper()
        print(f"== {ticker} ({i}/{len(files)}) ==")
        try:
            raw = load_symbol_parquet(path, ticker)
        except Exception as e:
            print(f"Skipping {ticker}: {e}")
            continue

        if raw.empty:
            print(f"Skipping {ticker}: no rows after load/filter.")
            continue

        try:
            df15 = to_15m(raw)
            if df15.empty:
                print(f"Skipping {ticker}: 15m resample produced no bars.")
                continue
        except Exception as e:
            print(f"Skipping {ticker}: resample error: {e}")
            continue

        try:
            tlog, eq = backtest_orb_retest(df15)
        except Exception as e:
            print(f"Skipping {ticker}: strategy error: {e}")
            continue

        if not tlog.empty:
            tlog_out = tlog.copy()
            tlog_out["Targets"] = tlog_out["Targets"].apply(lambda xs: ";".join([f"{p:.6f}" for p in xs]))
            tlog_out["Exits"]   = tlog_out["Exits"].apply(lambda xs: ";".join([f"{t[0]}|{t[1]}|{t[2]:.6f}|{t[3]:.6f}" for t in xs]))
            all_trades.append(tlog_out)
        if not eq.empty:
            eq_out = eq.copy(); eq_out["Ticker"] = ticker
            all_equity.append(eq_out)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")
    else:
        print("No trades generated with current settings.")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

    if not trades.empty:
        summary = (
            trades.groupby("Ticker")["PnL_$"]
                  .agg(['count','sum','mean','std'])
                  .rename(columns={'count':'Trades','sum':'PnL_sum','mean':'AvgPnL','std':'StdPnL'})
                  .reset_index()
                  .sort_values("PnL_sum", ascending=False)
        )
        wr = trades.groupby("Ticker")["PnL_$"].apply(lambda s: (pd.to_numeric(s, errors="coerce") > 0).mean()*100.0).rename("WinRate_%")
        pos = trades.groupby("Ticker")["PnL_$"].apply(lambda s: pd.to_numeric(s, errors="coerce").clip(lower=0).sum()).rename("GrossProfit")
        neg = trades.groupby("Ticker")["PnL_$"].apply(lambda s: -pd.to_numeric(s, errors="coerce").clip(upper=0).sum()).rename("GrossLoss")
        pf = (pos / neg.replace(0, np.nan)).rename("PF")
        summary = summary.merge(wr, on="Ticker", how="left").merge(pf, on="Ticker", how="left")
        print("\n=== Summary by Ticker (top 50) ===")
        print(summary.head(50).to_string(index=False))
        summary.to_csv(SUMMARY_CSV, index=False)
        print(f"Saved summary: {SUMMARY_CSV}")

if __name__ == "__main__":
    main()


Reading from PARQUET_DIR: C:\Users\pcagm\Downloads\parquet
Found 20 parquet files/datasets.

== AAPL (1/20) ==
== AMD (2/20) ==
[debug] os.path.exists(C:\Users\pcagm\Downloads\parquet\AMD.parquet) -> False | isfile=False | isdir=False
[debug] Listing of C:\Users\pcagm\Downloads\parquet:
    - AAPL.parquet
    - AMD.parquet
    - AMZN.parquet
    - CAT.parquet
    - CHWY.parquet
    - CVNA.parquet
    - GLD.parquet
    - GOOGL.parquet
    - GS.parquet
    - JPM.parquet
    - MSFT.parquet
    - NET.parquet
    - NFLX.parquet
    - NVDA.parquet
    - QQQ.parquet
    - SE.parquet
    - SPY.parquet
    - TEM.parquet
    - TSLA.parquet
    - TSM.parquet
Skipping AMD: Failed to read parquet FILE after retries: C:\Users\pcagm\Downloads\parquet\AMD.parquet
Last error: [Errno 9] Bad file descriptor: 'C:\\Users\\pcagm\\Downloads\\parquet\\AMD.parquet'
== AMZN (3/20) ==
Skipping AMZN: Failed to read parquet FILE after retries: C:\Users\pcagm\Downloads\parquet\AMZN.parquet
Last error: [Errno 9] Bad

In [12]:
# orb_15m_retest_from_parquet.py
# Implements your ORB 15m Retest/Continuation strategy exactly,
# and runs it on every .parquet in the given folder.

import warnings
warnings.filterwarnings("ignore")

import os, glob, time
from datetime import timedelta, time as dtime
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd

# =============================
# USER PARAMETERS
# =============================

# Parquet directory (ALL files inside will be processed)
PARQUET_DIR = r"C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet"

# If timestamps in your parquet are NAIVE and represent UTC, keep True
ASSUME_NAIVE_TIMESTAMPS_ARE_UTC = True

# Session / timezone
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (quality + enough trades)
MAX_RETEST_MIN = 120            # minutes after breakout to wait for retest
RETEST_CONFIRM_CLOSE = True     # retest bar must close back through the level
BREAKOUT_CUSHION_PCT = 0.0005   # 0.05% cushion beyond OR to confirm breakout/retest
MAX_OPPOSITE_WICK_FRAC = 0.40   # reject entry bar if opposite wick > 40% of bar range

# Multiple trades per day
MAX_TRADES_PER_SESSION = 2
ALLOW_BOTH_SIDES = True

# Scale-out in thirds
USE_SCALE_OUT = True
TARGETS_R = [1.0, 2.0, 2.5]
SCALE_SPLIT = (1/3, 1/3, 1/3)

# Breakeven policy
MOVE_STOP_TO_BE_AT_R = 1.0      # arm BE after TP1 (==1R)
BE_ON_NEXT_BAR = True           # move stop to BE only on the next bar

# Continuation (fallback if no retest)
USE_CONTINUATION = True

# Time windows
BREAKOUT_DEADLINE   = "11:00"   # ET deadline for breakout
RETEST_DEADLINE     = "12:00"   # ET deadline for retest
CONTINUATION_CUTOFF = "12:00"   # ET cutoff for continuation entries

# Filters
USE_VWAP_FILTER   = True
USE_TREND_FILTER  = True        # 15m EMA trend filter
EMA_FAST = 20
EMA_SLOW = 50

USE_RSI_FILTER = True
RSI_LEN = 14
RSI_THRESH_LONG = 50.0
RSI_THRESH_SHORT = 50.0

USE_ADX_FILTER = False
ADX_LEN = 14
ADX_MIN = 18.0

USE_DAILY_TREND_FILTER = True
DAILY_EMA_FAST = 20
DAILY_EMA_SLOW = 50
DAILY_ATR_LEN  = 14

# OR width vs daily ATR band
USE_OR_WIDTH_ATR_FILTER = True
OR_ATR_MIN = 0.10
OR_ATR_MAX = 2.00

# Opening 15m volume quantile
USE_OPENING_VOL_FILTER = True
OPENING_VOL_LOOKBACK = 60
OPENING_VOL_QUANTILE = 0.30

# Gap filters
USE_GAP_FILTER = True
GAP_MIN = 0.003                # 0.3%
GAP_MAX = 0.04                 # 4.0%
ALIGN_WITH_GAP_DIR = False

USE_DOW_FILTER = False
ALLOWED_DOW = {1, 2, 3}        # Tue–Thu if enabled

# Optional index confirmation (set spy15 if you later load SPY locally)
USE_SPY_CONFIRM = False

# Entry realism when using close confirmation
#   "next_open" (recommended) = fill at next bar open after confirming close
#   "confirm_close"           = fill at confirming bar close
ENTRY_ON_CONFIRM = "next_open"

# Risk / costs
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00
FEE_PER_FILL = True            # charge fee per exit leg (TP/stop)

# Optional ATR trail on remainder (off by default)
USE_ATR_TRAIL = False
ATR15_LEN = 14
ATR15_MULT = 2.0

# Batch / persistence
SLEEP_BETWEEN_TICKERS = 0.2
AUTOSAVE_EVERY = 25

# Output files
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"
SUMMARY_CSV    = "orb_summary_sp500.csv"

# =============================
# Helpers
# =============================

def ensure_unique_columns(df: pd.DataFrame) -> pd.DataFrame:
    if getattr(df.columns, "duplicated", None) is not None and df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def sessionize(obj) -> pd.Series:
    idx = getattr(obj, "index", obj)
    return pd.to_datetime(pd.Index(idx).date)

def _to_clock(ts) -> dtime:
    return dtime(ts.hour, ts.minute)

def _clock_le(ts, hhmm: str) -> bool:
    h, m = map(int, hhmm.split(":"))
    return _to_clock(ts) <= dtime(h, m)

def wick_opposite_fraction(row: pd.Series, direction: str) -> float:
    h, l, o, c = float(row["High"]), float(row["Low"]), float(row.get("Open", np.nan)), float(row["Close"])
    rng = max(h - l, 1e-12)
    if direction == "long":
        opp = min(o, c) - l
    else:
        opp = h - max(o, c)
    return float(max(opp, 0.0) / rng)

def true_range(high, low, prev_close):
    return np.maximum.reduce([
        (high - low).values,
        np.abs(high - prev_close).values,
        np.abs(low - prev_close).values
    ])

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def _normalize_splits(num_targets: int, splits: tuple) -> List[float]:
    if num_targets <= 0:
        return [1.0]
    if not splits:
        return [1.0 / num_targets] * num_targets
    arr = list(splits[:num_targets])
    if len(arr) < num_targets:
        arr += [0.0] * (num_targets - len(arr))
    s = sum(arr)
    if s <= 0:
        return [1.0 / num_targets] * num_targets
    return [x / s for x in arr]

# Overall Totals (adds Profit Factor & Avg Win/Loss)
def print_overall_totals(trades: pd.DataFrame):
    print("\n=== Overall Totals ===")
    if trades is None or trades.empty:
        print("Total trades: 0")
        print("Total PnL ($): 0.00")
        print("Win rate: 0.00%")
        print("Avg R: 0.000 | Median R: 0.000")
        print("Profit Factor: 0.000")
        print("Avg Win ($): 0.00 | Avg Loss ($): 0.00")
        return

    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0)
    wins_mask = pnl > 0
    losses_mask = pnl < 0

    total_trades = int(len(pnl))
    total_pnl    = float(pnl.sum())
    win_rate     = float(wins_mask.mean() * 100.0) if total_trades else 0.0
    avg_r        = float(pd.to_numeric(trades["R_multiple"], errors="coerce").mean())
    median_r     = float(pd.to_numeric(trades["R_multiple"], errors="coerce").median())

    gross_profit = float(pnl[wins_mask].sum())
    gross_loss   = float(-pnl[losses_mask].sum())
    pf = (gross_profit / gross_loss) if gross_loss > 0 else (float("inf") if gross_profit > 0 else 0.0)

    avg_win  = float(pnl[wins_mask].mean()) if wins_mask.any() else 0.0
    avg_loss = float(pnl[losses_mask].mean()) if losses_mask.any() else 0.0

    print(f"Total trades: {total_trades}")
    print(f"Total PnL ($): {total_pnl:,.2f}")
    print(f"Win rate: {win_rate:.2f}%")
    print(f"Avg R: {avg_r:.3f} | Median R: {median_r:.3f}")
    print(f"Profit Factor: {pf:.3f}")
    print(f"Avg Win ($): {avg_win:,.2f} | Avg Loss ($): {avg_loss:,.2f}")

# =============================
# Indicators & features
# =============================

def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    for c in ("ORH","ORL"):
        if (df.columns == c).sum() > 0:
            df = df.drop(columns=[c])
    for col in ("High","Low"):
        if col not in df.columns:
            raise ValueError(f"compute_opening_range: missing column {col}")
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df["ORH"] = np.nan
    df["ORL"] = np.nan
    df.loc[first_bar_idx, "ORH"] = df.loc[first_bar_idx, "High"].astype(float).values
    df.loc[first_bar_idx, "ORL"] = df.loc[first_bar_idx, "Low"].astype(float).values
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return ensure_unique_columns(df)

def add_session_vwap(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df["Session"] = sessionize(df)
    tp = (df["High"] + df["Low"] + df["Close"]) / 3.0
    if "Volume" in df.columns:
        df["vwap_num"] = tp * df["Volume"]
        df["vwap_den"] = df["Volume"].replace(0, np.nan)
    else:
        df["vwap_num"] = tp
        df["vwap_den"] = 1.0
    df["VWAP"] = (df.groupby("Session")["vwap_num"].cumsum() /
                  df.groupby("Session")["vwap_den"].cumsum())
    return ensure_unique_columns(df.drop(columns=["vwap_num","vwap_den"]))

def add_ema_trend(df_15: pd.DataFrame, fast=EMA_FAST, slow=EMA_SLOW) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"EMA{fast}"] = df["Close"].ewm(span=fast, adjust=False).mean()
    df[f"EMA{slow}"] = df["Close"].ewm(span=slow, adjust=False).mean()
    return ensure_unique_columns(df)

def rsi(series: pd.Series, length=14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0)
    dn = -delta.clip(upper=0)
    avg_gain = up.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = dn.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan))
    return 100 - (100 / (1 + rs))

def add_rsi(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"RSI{length}"] = rsi(df["Close"], length)
    return ensure_unique_columns(df)

def add_adx(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    up_move = df["High"].diff()
    dn_move = -df["Low"].diff()
    plus_dm  = np.where((up_move > dn_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((dn_move > up_move) & (dn_move > 0), dn_move, 0.0)
    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    atr = tr.ewm(alpha=1/length, adjust=False).mean()
    plus_di  = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    dx = (abs(plus_di - minus_di) / (plus_di + minus_di).replace(0, np.nan)) * 100
    adx = dx.ewm(alpha=1/length, adjust=False).mean()
    df[f"ADX{length}"] = adx
    return ensure_unique_columns(df)

def add_atr_15m(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    df[f"ATR15_{length}"] = tr.ewm(alpha=1/length, adjust=False).mean()
    return ensure_unique_columns(df)

def build_daily_from_15m(df_15: pd.DataFrame) -> pd.DataFrame:
    d = ensure_unique_columns(df_15.copy())
    d["Session"] = sessionize(d)
    daily = d.groupby("Session").agg(
        Open=("Open","first"),
        High=("High","max"),
        Low=("Low","min"),
        Close=("Close","last"),
        Volume=("Volume","sum") if "Volume" in d.columns else ("Close","size"),
    )
    return daily

def add_daily_bias_and_atr(df_15: pd.DataFrame, ema_fast=20, ema_slow=50, atr_len=14) -> pd.DataFrame:
    dly = build_daily_from_15m(df_15)
    dly[f"EMA_D{ema_fast}"] = dly["Close"].ewm(span=ema_fast, adjust=False).mean()
    dly[f"EMA_D{ema_slow}"] = dly["Close"].ewm(span=ema_slow, adjust=False).mean()
    prev_close = dly["Close"].shift(1)
    tr = pd.Series(np.maximum.reduce([
        (dly["High"] - dly["Low"]).values,
        np.abs(dly["High"] - prev_close).values,
        np.abs(dly["Low"]  - prev_close).values
    ]), index=dly.index)
    atr_col = f"ATR_D{atr_len}"
    dly[atr_col] = tr.ewm(alpha=1/atr_len, adjust=False).mean()

    if "Volume" in df_15.columns:
        tmp = df_15.copy()
        tmp["Session"] = sessionize(tmp)
        open15_vol = tmp.groupby("Session")["Volume"].first()
        dly["Open15_Vol"] = open15_vol
        dly["Open15_Vol_Thresh"] = (
            dly["Open15_Vol"]
            .rolling(window=OPENING_VOL_LOOKBACK, min_periods=10)
            .quantile(OPENING_VOL_QUANTILE)
        )

    feature_cols = [f"EMA_D{ema_fast}", f"EMA_D{ema_slow}", atr_col, "Open15_Vol", "Open15_Vol_Thresh"]
    feature_cols = [c for c in feature_cols if c in dly.columns]
    out = ensure_unique_columns(df_15.copy())
    out["Session"] = sessionize(out)
    out = out.merge(dly[feature_cols], left_on="Session", right_index=True, how="left")
    return ensure_unique_columns(out)

# =============================
# Backtest (per-ticker)
# =============================

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = TARGETS_R,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE,
    spy15: Optional[pd.DataFrame] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    df = compute_opening_range(df_15)
    if USE_VWAP_FILTER:  df = add_session_vwap(df)
    if USE_TREND_FILTER: df = add_ema_trend(df, EMA_FAST, EMA_SLOW)
    df = add_daily_bias_and_atr(df, DAILY_EMA_FAST, DAILY_EMA_SLOW, DAILY_ATR_LEN)

    df["Session"] = sessionize(df)
    first_idx = df.groupby("Session").head(1).index
    prev_close_series = df.groupby("Session")["Close"].last().shift(1)
    prev_close_map = df["Session"].map(prev_close_series)
    df.loc[first_idx, "GapPct"] = (df.loc[first_idx, "Open"] / prev_close_map.loc[first_idx] - 1.0)
    df["GapPct"] = df.groupby("Session")["GapPct"].ffill()

    if USE_RSI_FILTER:  df = add_rsi(df, RSI_LEN)
    if USE_ADX_FILTER:  df = add_adx(df, ADX_LEN)
    if USE_ATR_TRAIL:   df = add_atr_15m(df, ATR15_LEN)
    df = ensure_unique_columns(df)

    sessions = df["Session"].unique()
    trades = []

    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh, orl = float(sdf["ORH"].iloc[0]), float(sdf["ORL"].iloc[0])
        if np.isnan(orh) or np.isnan(orl):
            continue

        after_open = sdf.iloc[1:].copy()
        long_breaks  = after_open[after_open["Close"] > orh * (1 + BREAKOUT_CUSHION_PCT)]
        short_breaks = after_open[after_open["Close"] < orl * (1 - BREAKOUT_CUSHION_PCT)]

        cands = []
        for idx, row in long_breaks.iterrows():
            cands.append(("long", idx, row))
        for idx, row in short_breaks.iterrows():
            cands.append(("short", idx, row))
        cands.sort(key=lambda x: x[1])

        trades_this_session = 0
        took_long = False
        took_short = False

        for direction, btime, breakout_row in cands:
            if not _clock_le(breakout_row.name, BREAKOUT_DEADLINE):
                continue
            if not ALLOW_BOTH_SIDES:
                if took_long and direction == "short": break
                if took_short and direction == "long": break
            if direction == "long" and took_long:   continue
            if direction == "short" and took_short: continue

            cutoff = btime + timedelta(minutes=max_retest_min)
            window = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
            level = orh if direction == "long" else orl

            touch = window[(window["Low"] <= level) & (window["High"] >= level)].head(1)

            retest_used = True
            if touch.empty:
                if USE_CONTINUATION and _clock_le(breakout_row.name, CONTINUATION_CUTOFF):
                    retest_used = False
                    retest_time = breakout_row.name
                    retest_bar = breakout_row
                else:
                    continue
            else:
                retest_bar = touch.iloc[0]
                retest_time = retest_bar.name

            if retest_used and retest_confirm_close:
                if direction == "long":
                    ok = (retest_bar["Close"] >= level * (1 + BREAKOUT_CUSHION_PCT))
                else:
                    ok = (retest_bar["Close"] <= level * (1 - BREAKOUT_CUSHION_PCT))
                if not ok: continue

            if MAX_OPPOSITE_WICK_FRAC is not None:
                if wick_opposite_fraction(retest_bar, direction) > MAX_OPPOSITE_WICK_FRAC:
                    continue

            if USE_DOW_FILTER and retest_time.weekday() not in ALLOWED_DOW:
                continue

            if USE_VWAP_FILTER and "VWAP" in sdf.columns:
                v_prev = sdf.loc[:retest_time, "VWAP"].tail(3).dropna().values
                rising  = (len(v_prev) < 2) or np.all(np.diff(v_prev) >= 0)
                falling = (len(v_prev) < 2) or np.all(np.diff(v_prev) <= 0)
                if direction == "long":
                    if not (retest_bar["Close"] > retest_bar["VWAP"] and rising): continue
                else:
                    if not (retest_bar["Close"] < retest_bar["VWAP"] and falling): continue

            if USE_TREND_FILTER and (f"EMA{EMA_FAST}" in sdf.columns) and (f"EMA{EMA_SLOW}" in sdf.columns):
                emaf = retest_bar.get(f"EMA{EMA_FAST}", np.nan); emas = retest_bar.get(f"EMA{EMA_SLOW}", np.nan)
                if np.isfinite(emaf) and np.isfinite(emas):
                    if direction == "long" and not (emaf > emas): continue
                    if direction == "short" and not (emaf < emas): continue

            if USE_DAILY_TREND_FILTER and f"EMA_D{DAILY_EMA_FAST}" in sdf.columns and f"EMA_D{DAILY_EMA_SLOW}" in sdf.columns:
                dfast = float(sdf[f"EMA_D{DAILY_EMA_FAST}"].iloc[0])
                dslow = float(sdf[f"EMA_D{DAILY_EMA_SLOW}"].iloc[0])
                if direction == "long" and not (dfast > dslow): continue
                if direction == "short" and not (dfast < dslow): continue

            atr_col = f"ATR_D{DAILY_ATR_LEN}"
            if USE_OR_WIDTH_ATR_FILTER and atr_col in sdf.columns:
                or_width = float(orh - orl)
                atr_d = float(sdf[atr_col].iloc[0])
                if atr_d <= 0: continue
                frac = or_width / atr_d
                if not (OR_ATR_MIN <= frac <= OR_ATR_MAX): continue

            if USE_RSI_FILTER and f"RSI{RSI_LEN}" in sdf.columns:
                rsi_val = float(retest_bar[f"RSI{RSI_LEN}"])
                if direction == "long" and not (rsi_val >= RSI_THRESH_LONG): continue
                if direction == "short" and not (rsi_val <= 100 - RSI_THRESH_SHORT): continue

            if USE_ADX_FILTER and f"ADX{ADX_LEN}" in sdf.columns:
                adx_val = float(retest_bar[f"ADX{ADX_LEN}"])
                if not (adx_val >= ADX_MIN): continue

            if USE_GAP_FILTER and "GapPct" in sdf.columns:
                gap = sdf.loc[sdf.index.min(), "GapPct"]
                if pd.notna(gap) and not (GAP_MIN <= abs(float(gap)) <= GAP_MAX):
                    continue

            if ALIGN_WITH_GAP_DIR and "GapPct" in sdf.columns:
                g = float(sdf["GapPct"].iloc[0]) if pd.notna(sdf["GapPct"].iloc[0]) else 0.0
                if (direction == "long" and g < 0) or (direction == "short" and g > 0):
                    continue

            if USE_SPY_CONFIRM and (spy15 is not None) and (retest_time in spy15.index):
                spy_row = spy15.loc[retest_time]
                if direction == "long" and not (spy_row["SPY_Close"] > spy_row["SPY_VWAP"]): continue
                if direction == "short" and not (spy_row["SPY_Close"] < spy_row["SPY_VWAP"]): continue

            # ===== Realistic Entry =====
            if retest_used and retest_confirm_close:
                if ENTRY_ON_CONFIRM.lower() == "next_open":
                    next_bar = sdf[sdf.index > retest_time].head(1)
                    if next_bar.empty:
                        continue
                    entry_time = next_bar.index[0]
                    raw_entry = float(next_bar["Open"].iloc[0])
                elif ENTRY_ON_CONFIRM.lower() == "confirm_close":
                    entry_time = retest_time
                    raw_entry = float(retest_bar["Close"])
                else:
                    entry_time = retest_time
                    raw_entry = float(retest_bar["Close"])
            else:
                entry_time = retest_time
                raw_entry = float(level)

            entry = _apply_slippage(raw_entry, slippage_bps, "buy" if direction == "long" else "sell")
            stop  = orl if direction == "long" else orh
            rps   = abs(entry - stop)
            if rps <= 1e-12:
                continue

            qty = POSITION_SIZE_DOLLARS / max(entry, 1e-12)
            side_mult = 1 if direction == "long" else -1

            targets_r = TARGETS_R if USE_SCALE_OUT else r_targets
            targets = [(entry + r * rps) if direction == "long" else (entry - r * rps) for r in targets_r]

            run = sdf[sdf.index >= entry_time].copy()
            qty_left = qty
            exits = []
            next_tp_idx = 0

            if USE_SCALE_OUT and len(targets) >= 1:
                splits = _normalize_splits(len(targets), SCALE_SPLIT)
                leg_sizes = [qty * s for s in splits]
            else:
                leg_sizes = [qty]

            be_level = entry
            be_pending = False
            be_set_time = None

            for ts, row in run.iterrows():
                hi, lo = float(row["High"]), float(row["Low"])

                if be_pending and (not BE_ON_NEXT_BAR or (be_set_time is not None and ts > be_set_time)):
                    stop = be_level
                    be_pending = False

                if USE_ATR_TRAIL and f"ATR15_{ATR15_LEN}" in sdf.columns and qty_left > 1e-9:
                    atr_now = float(row[f"ATR15_{ATR15_LEN}"])
                    if direction == "long":
                        stop = max(stop, hi - ATR15_MULT * atr_now)
                    else:
                        stop = min(stop, lo + ATR15_MULT * atr_now)

                # Stop first
                if lo <= stop <= hi:
                    px = _apply_slippage(stop, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                    label = "breakeven" if abs(px - be_level) < 1e-10 else "stop"
                    exits.append((label, ts, px, qty_left))
                    qty_left = 0.0
                    break

                # Targets (sequential)
                if next_tp_idx < len(targets):
                    tp = targets[next_tp_idx]
                    if lo <= tp <= hi:
                        px = _apply_slippage(tp, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                        fill_qty = leg_sizes[next_tp_idx] if next_tp_idx < len(leg_sizes) else qty_left
                        exits.append((f"tp{next_tp_idx+1}", ts, px, fill_qty))
                        qty_left -= fill_qty
                        if next_tp_idx == 0 and MOVE_STOP_TO_BE_AT_R is not None:
                            be_pending = True
                            be_set_time = ts
                        next_tp_idx += 1
                        if qty_left <= 1e-9:
                            break

                # If no TP1 by continuation cutoff, tighten to BE
                if USE_CONTINUATION and not _clock_le(ts, CONTINUATION_CUTOFF) and next_tp_idx == 0:
                    stop = be_level

            # EOD exit for any leftover
            if qty_left > 1e-9:
                last = run.iloc[-1]
                px = _apply_slippage(float(last["Close"]), SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                exits.append(("eod", run.index[-1], px, qty_left))
                qty_left = 0.0

            cash_pnl = sum((px - entry) * side_mult * q for (_, _, px, q) in exits)
            if FEE_PER_FILL:
                cash_pnl -= FEES_PER_TRADE * len(exits)
            else:
                cash_pnl -= FEES_PER_TRADE

            trades.append({
                "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
                "Session": ses,
                "Direction": direction,
                "ORH": orh, "ORL": orl,
                "EntryTime": entry_time,
                "Entry": entry, "Stop": stop,
                "Targets": targets,
                "Exits": [(lab, ts, px, q) for (lab, ts, px, q) in exits],
                "Qty": qty,
                "PnL_$": cash_pnl,
                "R_multiple": cash_pnl / (rps * max(qty, 1e-12))
            })

            if direction == "long":  took_long  = True
            if direction == "short": took_short = True
            trades_this_session += 1
            if trades_this_session >= MAX_TRADES_PER_SESSION:
                break

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Parquet loading
# =============================

STD_MAP = {"open":"Open","high":"High","low":"Low","close":"Close","volume":"Volume"}

def _standardize_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    lower = {c.lower(): c for c in df.columns}
    for k, std in STD_MAP.items():
        if k in lower:                rename[lower[k]] = std
        elif k.capitalize() in df.columns: rename[k.capitalize()] = std
        elif k.upper() in df.columns: rename[k.upper()] = std
    out = df.rename(columns=rename)
    drop_cols = [c for c in out.columns if str(c).lower().startswith("adj")]
    return out.drop(columns=drop_cols, errors="ignore")

def _choose_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for cand in ["EventAt","Datetime","datetime","Timestamp","timestamp","Date","date","Time","time"]:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            df = df.set_index(cand)
            break
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No datetime index/column found.")
    return df

def _maybe_resample_to_15m(df: pd.DataFrame) -> pd.DataFrame:
    if len(df.index) < 3:
        return df
    # If median step < 15m, upsample to 15m OHLCV
    deltas = (df.index[1:] - df.index[:-1]).asi8
    if len(deltas) == 0:
        return df
    med_ns = np.median(deltas)
    fifteen_ns = pd.Timedelta("15T").value
    if med_ns < fifteen_ns:
        o = df["Open"].resample("15T", label="left", closed="left").first()
        h = df["High"].resample("15T", label="left", closed="left").max()
        l = df["Low"].resample("15T", label="left", closed="left").min()
        c = df["Close"].resample("15T", label="left", closed="left").last()
        v = df["Volume"].resample("15T", label="left", closed="left").sum() if "Volume" in df.columns else None
        parts = {"Open":o,"High":h,"Low":l,"Close":c}
        if v is not None: parts["Volume"] = v
        out = pd.concat(parts, axis=1).dropna(subset=["Open","High","Low","Close"])
        return out
    return df

def load_parquet_15m(path: str, tz: str = TZ) -> pd.DataFrame:
    df = pd.read_parquet(path, engine="pyarrow")
    df = _choose_dt_index(df).sort_index()

    if df.index.tz is None:
        if ASSUME_NAIVE_TIMESTAMPS_ARE_UTC:
            df = df.tz_localize("UTC").tz_convert(tz)
        else:
            df = df.tz_localize(tz)
    else:
        df = df.tz_convert(tz)

    df = _standardize_ohlcv_columns(df)
    for need in ["Open","High","Low","Close"]:
        if need not in df.columns:
            raise ValueError(f"{os.path.basename(path)} missing required column: {need}")

    df = _maybe_resample_to_15m(df)
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    return ensure_unique_columns(df.sort_index())

# =============================
# Runner (folder of parquet files)
# =============================

def run_folder(folder: str):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"Found {len(files)} parquet files.")
    all_trades, all_equity = [], []

    for i, fpath in enumerate(files, 1):
        ticker = os.path.splitext(os.path.basename(fpath))[0].upper().replace("_","").replace("-","")
        print(f"\n== {ticker} ({i}/{len(files)}) ==")
        try:
            df15 = load_parquet_15m(fpath)
            df15["Ticker"] = ticker
        except Exception as e:
            print(f"Skipping {ticker}: {e}")
            continue

        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=SLIPPAGE_BPS,
            fees=FEES_PER_TRADE,
            retest_confirm_close=RETEST_CONFIRM_CLOSE,
            spy15=None
        )

        if not tlog.empty:
            tlog_out = tlog.copy()
            tlog_out["Targets"] = tlog_out["Targets"].apply(lambda xs: ";".join([f"{p:.6f}" for p in xs]))
            tlog_out["Exits"]   = tlog_out["Exits"].apply(lambda xs: ";".join([f"{t[0]}|{t[1]}|{t[2]:.6f}|{t[3]:.6f}" for t in xs]))
            all_trades.append(tlog_out)
        if not eq.empty:
            eq_out = eq.copy(); eq_out["Ticker"] = ticker
            all_equity.append(eq_out)

        if AUTOSAVE_EVERY and i % AUTOSAVE_EVERY == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

        time.sleep(SLEEP_BETWEEN_TICKERS)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

def main():
    trades, equity = run_folder(PARQUET_DIR)

    print_overall_totals(trades)

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

    if not trades.empty:
        summary = (
            trades.groupby("Ticker")["PnL_$"]
                  .agg(['count', 'sum', 'mean', 'std'])
                  .rename(columns={'count':'Trades','sum':'PnL_sum','mean':'AvgPnL','std':'StdPnL'})
                  .reset_index()
                  .sort_values("PnL_sum", ascending=False)
        )
        wr = trades.groupby("Ticker")["PnL_$"].apply(lambda s: (pd.to_numeric(s, errors="coerce") > 0).mean()*100.0).rename("WinRate_%")
        pos = trades.groupby("Ticker")["PnL_$"].apply(lambda s: pd.to_numeric(s, errors="coerce").clip(lower=0).sum()).rename("GrossProfit")
        neg = trades.groupby("Ticker")["PnL_$"].apply(lambda s: -pd.to_numeric(s, errors="coerce").clip(upper=0).sum()).rename("GrossLoss")
        pf = (pos / neg.replace(0, np.nan)).rename("PF")
        summary = summary.merge(wr, on="Ticker", how="left").merge(pf, on="Ticker", how="left")
        print("\n=== Summary by Ticker (top 50) ===")
        print(summary.head(50).to_string(index=False))
        summary.to_csv(SUMMARY_CSV, index=False)
    else:
        print("No trades generated with current settings.")

if __name__ == "__main__":
    main()


Found 20 parquet files.

== AAPL (1/20) ==

== AMD (2/20) ==

== AMZN (3/20) ==

== CAT (4/20) ==

== CHWY (5/20) ==

== CVNA (6/20) ==

== GLD (7/20) ==


KeyboardInterrupt: 

In [37]:
# orb_15m_retest_from_folder.py
import warnings
warnings.filterwarnings("ignore")

import os, glob
from datetime import timedelta
from typing import List, Tuple
import numpy as np
import pandas as pd

# =============================
# USER PARAMETERS
# =============================
PARQUET_DIR = r"C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet"

ASSUME_NAIVE_TIMESTAMPS_ARE_UTC = True
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# If your parquet files are already 15m bars, set False.
# If you aren't sure, leave "auto" to detect based on median interval.
RESAMPLE_TO_15M = "auto"   # True | False | "auto"
SESSION_CUT = True          # keep only 09:30–16:00

# === ORB Retest parameters ===
RETEST_CONFIRM_CLOSE = True
MAX_RETEST_MIN = 120               # minutes after breakout to wait for retest
R_MULTIPLES = [1.0, 2.0]           # take first target hit, else stop/EOD
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00

# Autosave cadence
AUTOSAVE_EVERY = 10

# Output files
TRADES_CSV_ALL = "orb_trades_from_folder.csv"
EQUITY_CSV_ALL = "orb_equity_from_folder.csv"
SUMMARY_CSV    = "orb_summary_from_folder.csv"

# =============================
# Helpers (unchanged reading path)
# =============================
def list_parquet_files(folder: str):
    return sorted(glob.glob(os.path.join(folder, "*.parquet")))

def infer_ticker(path: str) -> str:
    return os.path.splitext(os.path.basename(path))[0].upper()

def normalize(df: pd.DataFrame) -> pd.DataFrame:
    cols = {c.lower(): c for c in df.columns}
    mapping = {}
    for want, cand in {
        "Open": ["open","o","open_price","price_open","opening_price"],
        "High": ["high","h","high_price","price_high"],
        "Low":  ["low","l","low_price","price_low"],
        "Close":["close","c","close_price","price_close","last","last_price"],
        "Volume":["volume","vol","v","volume_qty","qty","size"],
    }.items():
        for c in cand:
            if c in cols:
                mapping[cols[c]] = want
                break
    df = df.rename(columns=mapping)
    for c in list(df.columns):
        if str(c).lower().startswith("adj"):
            df = df.drop(columns=[c], errors="ignore")
    for c in ["Open","High","Low","Close","Volume"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def ensure_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    candidates = [c for c in df.columns if str(c).lower() in {"datetime","date","timestamp","time","ts"}]
    if not candidates:
        df.index = pd.to_datetime(df.index, errors="coerce", utc=False)
    else:
        dt = candidates[0]
        df[dt] = pd.to_datetime(df[dt], errors="coerce", utc=False)
        df = df.set_index(dt, drop=True)
    return df

def localize(df: pd.DataFrame, tz: str) -> pd.DataFrame:
    if df.index.tz is None:
        if ASSUME_NAIVE_TIMESTAMPS_ARE_UTC:
            df.index = df.index.tz_localize("UTC").tz_convert(tz)
        else:
            df.index = df.index.tz_localize(tz)
    else:
        df.index = df.index.tz_convert(tz)
    return df

def infer_interval_minutes(idx: pd.DatetimeIndex) -> float:
    if len(idx) < 2: return np.nan
    diffs = pd.Series(idx[1:] - idx[:-1]).dt.total_seconds() / 60.0
    return float(diffs.median())

def resample_15m(df: pd.DataFrame) -> pd.DataFrame:
    agg = {"Open":"first","High":"max","Low":"min","Close":"last"}
    if "Volume" in df.columns: agg["Volume"]="sum"
    return (df.sort_index()
              .resample("15T", label="right", closed="right")
              .agg(agg)
              .dropna(subset=["Open","High","Low","Close"]))

def load_file(path: str) -> pd.DataFrame:
    try:
        raw = pd.read_parquet(path, engine="pyarrow")
    except Exception as e:
        print(f"[READ ERR] {path}: {e}"); return pd.DataFrame()
    if raw is None or raw.empty:
        print(f"[EMPTY] {path}"); return pd.DataFrame()

    df = normalize(raw)
    df = ensure_dt_index(df)
    df = df[~df.index.isna()]
    df = df[~df.index.duplicated(keep="last")].sort_index()
    if df.empty: return pd.DataFrame()

    df = localize(df, TZ)

    # Decide resample
    do_resample = False
    if RESAMPLE_TO_15M is True:
        do_resample = True
    elif RESAMPLE_TO_15M == "auto":
        interval_min = infer_interval_minutes(df.index)
        if np.isnan(interval_min) or interval_min < 10:
            do_resample = True
        elif interval_min > 30:
            # too coarse (hourly/daily) for intraday ORB
            print(f"  [SKIP] interval ≈ {interval_min:.2f}m too coarse for 15m ORB.")
            return pd.DataFrame()

    if do_resample:
        try:
            df = resample_15m(df)
        except Exception as e:
            print(f"[RESAMPLE ERR] {os.path.basename(path)}: {e}")
            return pd.DataFrame()

    if SESSION_CUT:
        df = df.between_time(REG_SESSION_START, REG_SESSION_END)

    # Final OHLC check
    for need in ["Open","High","Low","Close"]:
        if need not in df.columns:
            print(f"[COL MISSING] {os.path.basename(path)} missing {need}")
            return pd.DataFrame()

    df["Ticker"] = infer_ticker(path)
    return df

# =============================
# ORB logic
# =============================
def sessionize(df: pd.DataFrame) -> pd.Series:
    return pd.to_datetime(df.index.date)

def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    """Sets ORH/ORL = first 15m bar's high/low for each session, then ffill within the day."""
    df = df_15.copy()
    df["Session"] = sessionize(df)
    first_idx = df.groupby("Session").head(1).index
    df["ORH"] = np.nan
    df["ORL"] = np.nan
    df.loc[first_idx, "ORH"] = df.loc[first_idx, "High"].astype(float)
    df.loc[first_idx, "ORL"] = df.loc[first_idx, "Low"].astype(float)
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return df

def _apply_slippage(price: float, bps: float, side: str) -> float:
    """bps = basis points; side='buy' or 'sell'."""
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = 120,
    r_targets: List[float] = [1.0, 2.0],
    dollars: float = 10_000,
    slippage_bps: float = 1.0,
    fees: float = 0.0,
    retest_confirm_close: bool = True
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    ORB (opening range breakout) with retest entry:
      1) Find first breakout (close beyond ORH/ORL) after the opening bar.
      2) Wait up to `max_retest_min` for a retest back to the OR level (touch intrabar).
      3) Optionally require the retest bar to close back through the level.
      4) Enter at the level with slippage; stop at opposite OR level.
      5) Exit on first target hit (R multiples), or stop, or EOD.
    """
    df = df_15.copy()
    df["Session"] = sessionize(df)
    trades = []

    for ses in df["Session"].unique():
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh = float(sdf["ORH"].iloc[0])
        orl = float(sdf["ORL"].iloc[0])
        if not np.isfinite(orh) or not np.isfinite(orl):
            continue

        # first breakout after the opening bar
        sdf_after = sdf.iloc[1:].copy()
        long_break  = sdf_after[sdf_after["Close"] > orh].head(1)
        short_break = sdf_after[sdf_after["Close"] < orl].head(1)

        if long_break.empty and short_break.empty:
            continue

        if not long_break.empty and not short_break.empty:
            direction = "long" if long_break.index[0] < short_break.index[0] else "short"
            breakout_row = long_break.iloc[0] if direction == "long" else short_break.iloc[0]
        elif not long_break.empty:
            direction = "long"; breakout_row = long_break.iloc[0]
        else:
            direction = "short"; breakout_row = short_break.iloc[0]

        btime = breakout_row.name
        cutoff = btime + timedelta(minutes=max_retest_min)
        after_break = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
        if after_break.empty:
            continue

        level = orh if direction == "long" else orl
        touch = after_break[(after_break["Low"] <= level) & (after_break["High"] >= level)].head(1)
        if touch.empty:
            continue

        retest_bar = touch.iloc[0]
        retest_time = retest_bar.name

        if retest_confirm_close:
            ok = (retest_bar["Close"] >= level) if direction == "long" else (retest_bar["Close"] <= level)
            if not ok:
                continue

        entry = _apply_slippage(level, slippage_bps, "buy" if direction == "long" else "sell")
        stop = orl if direction == "long" else orh
        risk_per_share = abs(entry - stop)
        if risk_per_share <= 1e-12:
            continue

        qty = dollars / entry
        side_mult = 1 if direction == "long" else -1
        t_prices = [(entry + r*risk_per_share) if direction == "long" else (entry - r*risk_per_share) for r in r_targets]

        # simulate forward from retest bar
        sdf_run = sdf[sdf.index >= retest_time].copy()
        exits = []
        for ts, row in sdf_run.iterrows():
            low, high = float(row["Low"]), float(row["High"])
            # stop first
            if low <= stop <= high:
                px = _apply_slippage(stop, slippage_bps, "sell" if direction=="long" else "buy")
                exits.append(("stop", ts, px))
                break
            # check targets (first target hit exits fully)
            hit = False
            for i, tp in enumerate(t_prices):
                if low <= tp <= high:
                    px = _apply_slippage(tp, slippage_bps, "sell" if direction=="long" else "buy")
                    exits.append((f"tp{i+1}", ts, px))
                    hit = True
            if hit:
                break

        if not exits:
            last = sdf_run.iloc[-1]
            px = _apply_slippage(float(last["Close"]), slippage_bps, "sell" if direction=="long" else "buy")
            exits.append(("eod", sdf_run.index[-1], px))

        n_parts = len(exits)
        qty_each = qty / n_parts
        cash_pnl = sum((px - entry) * side_mult * qty_each for _,_,px in exits) - FEES_PER_TRADE

        trades.append({
            "Ticker": sdf["Ticker"].iloc[0],
            "Session": ses,
            "ORH": orh, "ORL": orl,
            "Direction": direction,
            "EntryTime": retest_time,
            "Entry": entry, "Stop": stop,
            "Targets": t_prices,
            "Exits": exits,
            "Qty": qty,
            "PnL_$": cash_pnl,
            "R_multiple": cash_pnl / (risk_per_share * qty)
        })

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Runner
# =============================
def run_folder():
    files = list_parquet_files(PARQUET_DIR)
    print(f"Found {len(files)} parquet files.")
    all_trades, all_eq = [], []

    for i, path in enumerate(files, 1):
        tk = infer_ticker(path)
        print(f"\n== {tk} ({i}/{len(files)}) ==\n{path}")
        df = load_file(path)
        if df.empty:
            print("No usable data after normalization/resample/session filter.")
            continue

        # === Apply ORB strategy ===
        df = compute_opening_range(df)
        tlog, eq = backtest_orb_retest(
            df,
            max_retest_min=MAX_RETEST_MIN,
            r_targets=R_MULTIPLES,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=SLIPPAGE_BPS,
            fees=FEES_PER_TRADE,
            retest_confirm_close=RETEST_CONFIRM_CLOSE
        )

        if not tlog.empty:
            out = tlog.copy()
            out["Targets"] = out["Targets"].apply(lambda xs: ";".join(f"{p:.6f}" for p in xs))
            out["Exits"]   = out["Exits"].apply(lambda xs: ";".join(f"{t[0]}|{t[1]}|{t[2]:.6f}" for t in xs))
            all_trades.append(out)

        if not eq.empty:
            eq2 = eq.copy()
            eq2["Ticker"] = tk
            all_eq.append(eq2)

        # Autosave
        if i % AUTOSAVE_EVERY == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_eq:
                pd.concat(all_eq, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

    # Final outputs
    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_eq, ignore_index=True) if all_eq else pd.DataFrame()

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

    # ===== Per-ticker summary + overall totals =====
    if not trades.empty:
        g = trades.groupby("Ticker")
        per_ticker = pd.DataFrame({
            "Trades":   g["PnL_$"].count(),
            "PnL_sum":  g["PnL_$"].sum(),
            "AvgPnL":   g["PnL_$"].mean(),
            "StdPnL":   g["PnL_$"].std(),
            "Wins":     g["PnL_$"].apply(lambda s: (s > 0).sum()),
            "Losses":   g["PnL_$"].apply(lambda s: (s < 0).sum()),
            "WinRate":  g["PnL_$"].apply(lambda s: (s > 0).mean()),
            "AvgWin":   g["PnL_$"].apply(lambda s: s[s > 0].mean() if (s > 0).any() else np.nan),
            "AvgLoss":  g["PnL_$"].apply(lambda s: s[s < 0].mean() if (s < 0).any() else np.nan),
            "AvgR":     g["R_multiple"].mean(),
            "MedianR":  g["R_multiple"].median(),
            "PF":       g["PnL_$"].apply(lambda s: (s[s > 0].sum() / abs(s[s < 0].sum()))
                                         if abs(s[s < 0].sum()) > 0 else np.nan),
        }).reset_index()

        per_ticker["WinRate"] = (per_ticker["WinRate"] * 100).round(2)
        per_ticker["LossMag"] = per_ticker["AvgLoss"].abs()
        per_ticker.to_csv(SUMMARY_CSV, index=False)
        print(f"Saved summary: {SUMMARY_CSV}")

        total_trades = len(trades)
        total_pnl    = trades["PnL_$"].sum()
        win_rate     = (trades["PnL_$"] > 0).mean() * 100.0
        avg_r        = trades["R_multiple"].mean()
        median_r     = trades["R_multiple"].median()
        avg_win_all  = trades.loc[trades["PnL_$"] > 0, "PnL_$"].mean()
        avg_loss_all = trades.loc[trades["PnL_$"] < 0, "PnL_$"].mean()
        pf_all = trades.loc[trades["PnL_$"] > 0, "PnL_$"].sum() / abs(trades.loc[trades["PnL_$"] < 0, "PnL_$"].sum())

        print("\n=== Overall Totals ===")
        print(f"Total trades: {total_trades}")
        print(f"Total PnL ($): {total_pnl:,.2f}")
        print(f"Win rate: {win_rate:.2f}%")
        print(f"Avg R: {avg_r:.3f} | Median R: {median_r:.3f}")
        print(f"Avg Win ($): {0.0 if pd.isna(avg_win_all) else avg_win_all:,.2f}")
        print(f"Avg Loss ($): {0.0 if pd.isna(avg_loss_all) else avg_loss_all:,.2f}")
        print(f"Profit Factor: {pf_all:.3f}")
    else:
        print("No trades generated with current settings.")

if __name__=="__main__":
    run_folder()


Found 20 parquet files.

== AAPL (1/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AAPL.parquet
No usable data after normalization/resample/session filter.

== AMD (2/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMD.parquet
No usable data after normalization/resample/session filter.

== AMZN (3/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMZN.parquet
No usable data after normalization/resample/session filter.

== CAT (4/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CAT.parquet
No usable data after normalization/resample/session filter.

== CHWY (5/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CHWY.parquet
No usable data after normalization/resample/session filter.

== CVNA (6/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CVNA.parquet
No usable data after normalization/resample/session filter.

== GLD (7/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GLD.

In [17]:
# orb_15m_retest_from_parquet_no_indicators.py
# ORB 15m Retest/Continuation strategy on every .parquet in the folder,
# with ALL indicators and indicator-based filters removed.

import warnings
warnings.filterwarnings("ignore")

import os, glob, time
from datetime import timedelta, time as dtime
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd

# =============================
# USER PARAMETERS
# =============================

# Parquet directory (ALL files inside will be processed)
PARQUET_DIR = r"C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet"

# If timestamps in your parquet are NAIVE and represent UTC, keep True
ASSUME_NAIVE_TIMESTAMPS_ARE_UTC = True

# Session / timezone
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (quality + enough trades)
MAX_RETEST_MIN = 120            # minutes after breakout to wait for retest
RETEST_CONFIRM_CLOSE = True     # retest bar must close back through the level
BREAKOUT_CUSHION_PCT = 0.0005   # 0.05% cushion beyond OR to confirm breakout/retest
MAX_OPPOSITE_WICK_FRAC = 0.40   # reject entry bar if opposite wick > 40% of bar range

# Multiple trades per day
MAX_TRADES_PER_SESSION = 2
ALLOW_BOTH_SIDES = True

# Scale-out in thirds
USE_SCALE_OUT = True
TARGETS_R = [1.0, 2.0, 2.5]
SCALE_SPLIT = (1/3, 1/3, 1/3)

# Breakeven policy
MOVE_STOP_TO_BE_AT_R = 1.0      # arm BE after TP1 (==1R)
BE_ON_NEXT_BAR = True           # move stop to BE only on the next bar

# Continuation (fallback if no retest)
USE_CONTINUATION = True

# Time windows
BREAKOUT_DEADLINE   = "11:00"   # ET deadline for breakout
RETEST_DEADLINE     = "12:00"   # ET deadline for retest (kept for reference; not used separately)
CONTINUATION_CUTOFF = "12:00"   # ET cutoff for continuation entries

# --- Non-indicator filters (kept) ---
USE_GAP_FILTER = True           # filter by open gap vs prior close
GAP_MIN = 0.003                 # 0.3%
GAP_MAX = 0.04                  # 4.0%
ALIGN_WITH_GAP_DIR = False      # require trade direction to match gap direction

USE_DOW_FILTER = False          # optional: restrict by day of week
ALLOWED_DOW = {1, 2, 3}         # Tue–Thu if enabled

# Entry realism when using close confirmation
#   "next_open" (recommended) = fill at next bar open after confirming close
#   "confirm_close"           = fill at confirming bar close
ENTRY_ON_CONFIRM = "next_open"

# Risk / costs
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00
FEE_PER_FILL = True             # charge fee per exit leg (TP/stop)

# Batch / persistence
SLEEP_BETWEEN_TICKERS = 0.2
AUTOSAVE_EVERY = 25

# Output files
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"
SUMMARY_CSV    = "orb_summary_sp500.csv"

# =============================
# Helpers
# =============================

def ensure_unique_columns(df: pd.DataFrame) -> pd.DataFrame:
    if getattr(df.columns, "duplicated", None) is not None and df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def sessionize(obj) -> pd.Series:
    idx = getattr(obj, "index", obj)
    return pd.to_datetime(pd.Index(idx).date)

def _to_clock(ts) -> dtime:
    return dtime(ts.hour, ts.minute)

def _clock_le(ts, hhmm: str) -> bool:
    h, m = map(int, hhmm.split(":"))
    return _to_clock(ts) <= dtime(h, m)

def wick_opposite_fraction(row: pd.Series, direction: str) -> float:
    h, l, o, c = float(row["High"]), float(row["Low"]), float(row.get("Open", np.nan)), float(row["Close"])
    rng = max(h - l, 1e-12)
    if direction == "long":
        opp = min(o, c) - l
    else:
        opp = h - max(o, c)
    return float(max(opp, 0.0) / rng)

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def _normalize_splits(num_targets: int, splits: tuple) -> List[float]:
    if num_targets <= 0:
        return [1.0]
    if not splits:
        return [1.0 / num_targets] * num_targets
    arr = list(splits[:num_targets])
    if len(arr) < num_targets:
        arr += [0.0] * (num_targets - len(arr))
    s = sum(arr)
    if s <= 0:
        return [1.0 / num_targets] * num_targets
    return [x / s for x in arr]

# Overall Totals (adds Profit Factor & Avg Win/Loss)
def print_overall_totals(trades: pd.DataFrame):
    print("\n=== Overall Totals ===")
    if trades is None or trades.empty:
        print("Total trades: 0")
        print("Total PnL ($): 0.00")
        print("Win rate: 0.00%")
        print("Avg R: 0.000 | Median R: 0.000")
        print("Profit Factor: 0.000")
        print("Avg Win ($): 0.00 | Avg Loss ($): 0.00")
        return

    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0)
    wins_mask = pnl > 0
    losses_mask = pnl < 0

    total_trades = int(len(pnl))
    total_pnl    = float(pnl.sum())
    win_rate     = float(wins_mask.mean() * 100.0) if total_trades else 0.0
    avg_r        = float(pd.to_numeric(trades["R_multiple"], errors="coerce").mean())
    median_r     = float(pd.to_numeric(trades["R_multiple"], errors="coerce").median())

    gross_profit = float(pnl[wins_mask].sum())
    gross_loss   = float(-pnl[losses_mask].sum())
    pf = (gross_profit / gross_loss) if gross_loss > 0 else (float("inf") if gross_profit > 0 else 0.0)

    avg_win  = float(pnl[wins_mask].mean()) if wins_mask.any() else 0.0
    avg_loss = float(pnl[losses_mask].mean()) if losses_mask.any() else 0.0

    print(f"Total trades: {total_trades}")
    print(f"Total PnL ($): {total_pnl:,.2f}")
    print(f"Win rate: {win_rate:.2f}%")
    print(f"Avg R: {avg_r:.3f} | Median R: {median_r:.3f}")
    print(f"Profit Factor: {pf:.3f}")
    print(f"Avg Win ($): {avg_win:,.2f} | Avg Loss ($): {avg_loss:,.2f}")

# =============================
# Opening Range (no indicators)
# =============================

def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    """
    Opening range = first bar's High/Low of each session (assumes 15m bars or
    native bars already cut to regular session); forward-filled intra-day.
    """
    df = ensure_unique_columns(df_15.copy())
    for c in ("ORH","ORL"):
        if (df.columns == c).sum() > 0:
            df = df.drop(columns=[c])
    for col in ("High","Low"):
        if col not in df.columns:
            raise ValueError(f"compute_opening_range: missing column {col}")
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df["ORH"] = np.nan
    df["ORL"] = np.nan
    df.loc[first_bar_idx, "ORH"] = df.loc[first_bar_idx, "High"].astype(float).values
    df.loc[first_bar_idx, "ORL"] = df.loc[first_bar_idx, "Low"].astype(float).values
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return ensure_unique_columns(df)

# =============================
# Backtest (per-ticker) — indicators removed
# =============================

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = TARGETS_R,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    # Only compute ORH/ORL and non-indicator context (gap)
    df = compute_opening_range(df_15)
    df["Session"] = sessionize(df)

    # Previous close (for gap)
    first_idx = df.groupby("Session").head(1).index
    prev_close_series = df.groupby("Session")["Close"].last().shift(1)
    prev_close_map = df["Session"].map(prev_close_series)
    df.loc[first_idx, "GapPct"] = (df.loc[first_idx, "Open"] / prev_close_map.loc[first_idx] - 1.0)
    df["GapPct"] = df.groupby("Session")["GapPct"].ffill()

    sessions = df["Session"].unique()
    trades = []

    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh, orl = float(sdf["ORH"].iloc[0]), float(sdf["ORL"].iloc[0])
        if np.isnan(orh) or np.isnan(orl):
            continue

        # Find ALL breakouts after the opening bar with cushion
        after_open = sdf.iloc[1:].copy()
        long_breaks  = after_open[after_open["Close"] > orh * (1 + BREAKOUT_CUSHION_PCT)]
        short_breaks = after_open[after_open["Close"] < orl * (1 - BREAKOUT_CUSHION_PCT)]

        # Merge, sorted by time
        cands = []
        for idx, row in long_breaks.iterrows():
            cands.append(("long", idx, row))
        for idx, row in short_breaks.iterrows():
            cands.append(("short", idx, row))
        cands.sort(key=lambda x: x[1])

        trades_this_session = 0
        took_long = False
        took_short = False

        for direction, btime, breakout_row in cands:
            # breakout must happen before deadline
            if not _clock_le(breakout_row.name, BREAKOUT_DEADLINE):
                continue

            # One-side or both-sides control
            if not ALLOW_BOTH_SIDES:
                if took_long and direction == "short": break
                if took_short and direction == "long": break
            if direction == "long" and took_long:   continue
            if direction == "short" and took_short: continue

            # Non-indicator filters
            if USE_DOW_FILTER and btime.weekday() not in ALLOWED_DOW:
                continue
            if USE_GAP_FILTER and "GapPct" in sdf.columns:
                gap = sdf.loc[sdf.index.min(), "GapPct"]
                if pd.notna(gap) and not (GAP_MIN <= abs(float(gap)) <= GAP_MAX):
                    continue
                if ALIGN_WITH_GAP_DIR and pd.notna(gap):
                    g = float(gap)
                    if (direction == "long" and g < 0) or (direction == "short" and g > 0):
                        continue

            # Look for retest until cutoff
            cutoff = btime + timedelta(minutes=max_retest_min)
            window = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
            level = orh if direction == "long" else orl
            touch = window[(window["Low"] <= level) & (window["High"] >= level)].head(1)

            retest_used = True
            if touch.empty:
                if USE_CONTINUATION and _clock_le(breakout_row.name, CONTINUATION_CUTOFF):
                    retest_used = False
                    retest_time = breakout_row.name
                    retest_bar = breakout_row
                else:
                    continue
            else:
                retest_bar = touch.iloc[0]
                retest_time = retest_bar.name

            # Confirm close through level if requested (with cushion)
            if retest_used and retest_confirm_close:
                if direction == "long":
                    ok = (retest_bar["Close"] >= level * (1 + BREAKOUT_CUSHION_PCT))
                else:
                    ok = (retest_bar["Close"] <= level * (1 - BREAKOUT_CUSHION_PCT))
                if not ok:
                    continue

            # Reject with big opposite wick
            if MAX_OPPOSITE_WICK_FRAC is not None:
                if wick_opposite_fraction(retest_bar, direction) > MAX_OPPOSITE_WICK_FRAC:
                    continue

            # ===== Entry price (realistic) =====
            if retest_used and retest_confirm_close:
                if ENTRY_ON_CONFIRM.lower() == "next_open":
                    next_bar = sdf[sdf.index > retest_time].head(1)
                    if next_bar.empty:
                        continue
                    entry_time = next_bar.index[0]
                    raw_entry = float(next_bar["Open"].iloc[0])
                elif ENTRY_ON_CONFIRM.lower() == "confirm_close":
                    entry_time = retest_time
                    raw_entry = float(retest_bar["Close"])
                else:
                    entry_time = retest_time
                    raw_entry = float(retest_bar["Close"])
            else:
                entry_time = retest_time
                raw_entry = float(level)

            entry = _apply_slippage(raw_entry, SLIPPAGE_BPS, "buy" if direction == "long" else "sell")
            stop  = orl if direction == "long" else orh
            rps   = abs(entry - stop)
            if rps <= 1e-12:
                continue

            qty = POSITION_SIZE_DOLLARS / max(entry, 1e-12)
            side_mult = 1 if direction == "long" else -1

            targets_r = TARGETS_R if USE_SCALE_OUT else r_targets
            targets = [(entry + r * rps) if direction == "long" else (entry - r * rps) for r in targets_r]

            # Walk forward bar-by-bar
            run = sdf[sdf.index >= entry_time].copy()
            qty_left = qty
            exits = []
            next_tp_idx = 0

            if USE_SCALE_OUT and len(targets) >= 1:
                splits = _normalize_splits(len(targets), SCALE_SPLIT)
                leg_sizes = [qty * s for s in splits]
            else:
                leg_sizes = [qty]

            be_level = entry
            be_pending = False
            be_set_time = None

            for ts, row in run.iterrows():
                hi, lo = float(row["High"]), float(row["Low"])

                # move to BE (armed after TP1), optionally on next bar
                if be_pending and (not BE_ON_NEXT_BAR or (be_set_time is not None and ts > be_set_time)):
                    stop = be_level
                    be_pending = False

                # Stop first
                if lo <= stop <= hi:
                    px = _apply_slippage(stop, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                    label = "breakeven" if abs(px - be_level) < 1e-10 else "stop"
                    exits.append((label, ts, px, qty_left))
                    qty_left = 0.0
                    break

                # Targets (sequential)
                if next_tp_idx < len(targets):
                    tp = targets[next_tp_idx]
                    if lo <= tp <= hi:
                        px = _apply_slippage(tp, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                        fill_qty = leg_sizes[next_tp_idx] if next_tp_idx < len(leg_sizes) else qty_left
                        exits.append((f"tp{next_tp_idx+1}", ts, px, fill_qty))
                        qty_left -= fill_qty
                        if next_tp_idx == 0 and MOVE_STOP_TO_BE_AT_R is not None:
                            be_pending = True
                            be_set_time = ts
                        next_tp_idx += 1
                        if qty_left <= 1e-9:
                            break

                # If no TP1 by continuation cutoff, tighten to BE
                if USE_CONTINUATION and not _clock_le(ts, CONTINUATION_CUTOFF) and next_tp_idx == 0:
                    stop = be_level

            # EOD exit for any leftover
            if qty_left > 1e-9:
                last = run.iloc[-1]
                px = _apply_slippage(float(last["Close"]), SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                exits.append(("eod", run.index[-1], px, qty_left))
                qty_left = 0.0

            cash_pnl = sum((px - entry) * side_mult * q for (_, _, px, q) in exits)
            if FEE_PER_FILL:
                cash_pnl -= FEES_PER_TRADE * len(exits)
            else:
                cash_pnl -= FEES_PER_TRADE

            trades.append({
                "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
                "Session": ses,
                "Direction": direction,
                "ORH": orh, "ORL": orl,
                "EntryTime": entry_time,
                "Entry": entry, "Stop": stop,
                "Targets": targets,
                "Exits": [(lab, ts, px, q) for (lab, ts, px, q) in exits],
                "Qty": qty,
                "PnL_$": cash_pnl,
                "R_multiple": cash_pnl / (rps * max(qty, 1e-12))
            })

            if direction == "long":  took_long  = True
            if direction == "short": took_short = True
            trades_this_session += 1
            if trades_this_session >= MAX_TRADES_PER_SESSION:
                break

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Parquet loading (unchanged)
# =============================

STD_MAP = {"open":"Open","high":"High","low":"Low","close":"Close","volume":"Volume"}

def _standardize_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    lower = {c.lower(): c for c in df.columns}
    for k, std in STD_MAP.items():
        if k in lower:                rename[lower[k]] = std
        elif k.capitalize() in df.columns: rename[k.capitalize()] = std
        elif k.upper() in df.columns: rename[k.upper()] = std
    out = df.rename(columns=rename)
    drop_cols = [c for c in out.columns if str(c).lower().startswith("adj")]
    return out.drop(columns=drop_cols, errors="ignore")

def _choose_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for cand in ["EventAt","Datetime","datetime","Timestamp","timestamp","Date","date","Time","time"]:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            df = df.set_index(cand)
            break
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No datetime index/column found.")
    return df

def _maybe_resample_to_15m(df: pd.DataFrame) -> pd.DataFrame:
    if len(df.index) < 3:
        return df
    # If median step < 15m, upsample to 15m OHLCV
    deltas = (df.index[1:] - df.index[:-1]).asi8
    if len(deltas) == 0:
        return df
    med_ns = np.median(deltas)
    fifteen_ns = pd.Timedelta("15T").value
    if med_ns < fifteen_ns:
        o = df["Open"].resample("15T", label="left", closed="left").first()
        h = df["High"].resample("15T", label="left", closed="left").max()
        l = df["Low"].resample("15T", label="left", closed="left").min()
        c = df["Close"].resample("15T", label="left", closed="left").last()
        v = df["Volume"].resample("15T", label="left", closed="left").sum() if "Volume" in df.columns else None
        parts = {"Open":o,"High":h,"Low":l,"Close":c}
        if v is not None: parts["Volume"] = v
        out = pd.concat(parts, axis=1).dropna(subset=["Open","High","Low","Close"])
        return out
    return df

def load_parquet_15m(path: str, tz: str = TZ) -> pd.DataFrame:
    df = pd.read_parquet(path, engine="pyarrow")
    df = _choose_dt_index(df).sort_index()

    if df.index.tz is None:
        if ASSUME_NAIVE_TIMESTAMPS_ARE_UTC:
            df = df.tz_localize("UTC").tz_convert(tz)
        else:
            df = df.tz_localize(tz)
    else:
        df = df.tz_convert(tz)

    df = _standardize_ohlcv_columns(df)
    for need in ["Open","High","Low","Close"]:
        if need not in df.columns:
            raise ValueError(f"{os.path.basename(path)} missing required column: {need}")

    df = _maybe_resample_to_15m(df)
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    return ensure_unique_columns(df.sort_index())

# =============================
# Runner (folder of parquet files)
# =============================

def run_folder(folder: str):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"Found {len(files)} parquet files.")
    all_trades, all_equity = [], []

    for i, fpath in enumerate(files, 1):
        ticker = os.path.splitext(os.path.basename(fpath))[0].upper().replace("_","").replace("-","")
        print(f"\n== {ticker} ({i}/{len(files)}) ==")
        try:
            df15 = load_parquet_15m(fpath)
            df15["Ticker"] = ticker
        except Exception as e:
            print(f"Skipping {ticker}: {e}")
            continue

        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=SLIPPAGE_BPS,
            fees=FEES_PER_TRADE,
            retest_confirm_close=RETEST_CONFIRM_CLOSE,
        )

        if not tlog.empty:
            tlog_out = tlog.copy()
            tlog_out["Targets"] = tlog_out["Targets"].apply(lambda xs: ";".join([f"{p:.6f}" for p in xs]))
            tlog_out["Exits"]   = tlog_out["Exits"].apply(lambda xs: ";".join([f"{t[0]}|{t[1]}|{t[2]:.6f}|{t[3]:.6f}" for t in xs]))
            all_trades.append(tlog_out)
        if not eq.empty:
            eq_out = eq.copy(); eq_out["Ticker"] = ticker
            all_equity.append(eq_out)

        if AUTOSAVE_EVERY and i % AUTOSAVE_EVERY == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

        time.sleep(SLEEP_BETWEEN_TICKERS)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

def main():
    trades, equity = run_folder(PARQUET_DIR)

    print_overall_totals(trades)

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

    if not trades.empty:
        summary = (
            trades.groupby("Ticker")["PnL_$"]
                  .agg(['count', 'sum', 'mean', 'std'])
                  .rename(columns={'count':'Trades','sum':'PnL_sum','mean':'AvgPnL','std':'StdPnL'})
                  .reset_index()
                  .sort_values("PnL_sum", ascending=False)
        )
        wr = trades.groupby("Ticker")["PnL_$"].apply(lambda s: (pd.to_numeric(s, errors="coerce") > 0).mean()*100.0).rename("WinRate_%")
        pos = trades.groupby("Ticker")["PnL_$"].apply(lambda s: pd.to_numeric(s, errors="coerce").clip(lower=0).sum()).rename("GrossProfit")
        neg = trades.groupby("Ticker")["PnL_$"].apply(lambda s: -pd.to_numeric(s, errors="coerce").clip(upper=0).sum()).rename("GrossLoss")
        pf = (pos / neg.replace(0, np.nan)).rename("PF")
        summary = summary.merge(wr, on="Ticker", how="left").merge(pf, on="Ticker", how="left")
        print("\n=== Summary by Ticker (top 50) ===")
        print(summary.head(50).to_string(index=False))
        summary.to_csv(SUMMARY_CSV, index=False)
    else:
        print("No trades generated with current settings.")

if __name__ == "__main__":
    main()


Found 20 parquet files.

== AAPL (1/20) ==

== AMD (2/20) ==

== AMZN (3/20) ==

== CAT (4/20) ==

== CHWY (5/20) ==

== CVNA (6/20) ==

== GLD (7/20) ==

== GOOGL (8/20) ==

== GS (9/20) ==

== JPM (10/20) ==

== MSFT (11/20) ==

== NET (12/20) ==

== NFLX (13/20) ==

== NVDA (14/20) ==

== QQQ (15/20) ==

== SE (16/20) ==

== SPY (17/20) ==

== TEM (18/20) ==

== TSLA (19/20) ==

== TSM (20/20) ==

=== Overall Totals ===
Total trades: 11997
Total PnL ($): 1,047,366.06
Win rate: 70.43%
Avg R: 0.709 | Median R: 0.687
Profit Factor: 7.548
Avg Win ($): 142.88 | Avg Loss ($): -45.10
Saved trades: orb_trades_sp500.csv
Saved equity: orb_equity_sp500.csv

=== Summary by Ticker (top 50) ===
Ticker  Trades       PnL_sum     AvgPnL     StdPnL  WinRate_%        PF
  CVNA     722 133032.386193 184.255383 308.690759  67.313019  6.514638
  TSLA     824  94283.444630 114.421656 166.819613  66.140777  8.092799
   AMD     727  85530.234480 117.648190 168.559543  65.061898  7.625598
    SE     713  855

In [19]:
# orb_15m_retest_pure.py
# Pure 15-minute ORB retest backtest on all .parquet files in a folder.

import warnings
warnings.filterwarnings("ignore")

import os, glob, time
from datetime import timedelta
from typing import List, Tuple
import numpy as np
import pandas as pd

# =============================
# USER PARAMETERS
# =============================
PARQUET_DIR = r"C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet"

ASSUME_NAIVE_TIMESTAMPS_ARE_UTC = True
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# Trading assumptions (kept minimal)
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00

# I/O
AUTOSAVE_EVERY = 25
TRADES_CSV_ALL = "orb_trades_pure.csv"
EQUITY_CSV_ALL = "orb_equity_pure.csv"
SUMMARY_CSV    = "orb_summary_pure.csv"

# =============================
# Helpers
# =============================
def ensure_unique_columns(df: pd.DataFrame) -> pd.DataFrame:
    if getattr(df.columns, "duplicated", None) is not None and df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def sessionize(obj) -> pd.Series:
    idx = getattr(obj, "index", obj)
    return pd.to_datetime(pd.Index(idx).date)

def apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

# =============================
# Parquet loading (15m only)
# =============================
STD_MAP = {"open":"Open","high":"High","low":"Low","close":"Close","volume":"Volume"}

def _standardize_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    lower = {c.lower(): c for c in df.columns}
    for k, std in STD_MAP.items():
        if k in lower:                rename[lower[k]] = std
        elif k.capitalize() in df.columns: rename[k.capitalize()] = std
        elif k.upper() in df.columns: rename[k.upper()] = std
    out = df.rename(columns=rename)
    drop_cols = [c for c in out.columns if str(c).lower().startswith("adj")]
    return out.drop(columns=drop_cols, errors="ignore")

def _choose_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for cand in ["EventAt","Datetime","datetime","Timestamp","timestamp","Date","date","Time","time"]:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            df = df.set_index(cand)
            break
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No datetime index/column found.")
    return df

def _infer_median_interval_minutes(idx: pd.DatetimeIndex) -> float:
    if len(idx) < 2: return np.nan
    d = (idx[1:] - idx[:-1]).total_seconds() / 60.0
    return float(np.median(d)) if len(d) else np.nan

def _resample_to_15m_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    """
    - If bars are <15m, resample to exact 15m OHLC(V).
    - If bars are exactly 15m, leave as is.
    - If bars are >15m, return empty (not usable for pure 15m ORB).
    """
    m = _infer_median_interval_minutes(df.index)
    if np.isnan(m):
        return pd.DataFrame()
    if m < 15 - 1e-6:
        agg = {"Open":"first","High":"max","Low":"min","Close":"last"}
        if "Volume" in df.columns: agg["Volume"] = "sum"
        out = (df.sort_index()
                 .resample("15T", label="right", closed="right")
                 .agg(agg)
                 .dropna(subset=["Open","High","Low","Close"]))
        return out
    elif abs(m - 15) <= 1e-6:
        return df.sort_index()
    else:
        # coarser than 15m → skip for a pure 15m test
        return pd.DataFrame()

def load_parquet_15m(path: str, tz: str = TZ) -> pd.DataFrame:
    df = pd.read_parquet(path, engine="pyarrow")
    df = _choose_dt_index(df).sort_index()

    if df.index.tz is None:
        if ASSUME_NAIVE_TIMESTAMPS_ARE_UTC:
            df = df.tz_localize("UTC").tz_convert(tz)
        else:
            df = df.tz_localize(tz)
    else:
        df = df.tz_convert(tz)

    df = _standardize_ohlcv_columns(df)
    for need in ["Open","High","Low","Close"]:
        if need not in df.columns:
            raise ValueError(f"{os.path.basename(path)} missing required column: {need}")

    df = _resample_to_15m_if_needed(df)
    if df.empty:
        return df

    # Regular session only
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    return ensure_unique_columns(df.sort_index())

# =============================
# ORB: compute opening range (first 15m bar)
# =============================
def compute_opening_range(df15: pd.DataFrame) -> pd.DataFrame:
    df = df15.copy()
    df["Session"] = sessionize(df)
    # first 15m bar of each session
    first_idx = df.groupby("Session").head(1).index
    df["ORH"] = np.nan
    df["ORL"] = np.nan
    df.loc[first_idx, "ORH"] = df.loc[first_idx, "High"].astype(float).values
    df.loc[first_idx, "ORL"] = df.loc[first_idx, "Low"].astype(float).values
    # forward-fill intraday
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return df

# =============================
# Pure ORB Retest Backtest
# =============================
def backtest_orb_retest_pure(df15: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Pure spec:
      - Identify first 15m bar (ORH/ORL).
      - Find first breakout *close* above ORH or below ORL after the OR bar.
      - Wait for a retest (intrabar touch of the OR level).
      - Enter at the OR level price on the retest (with slippage).
      - Stop at the opposite OR level.
      - Exit remaining at end of day if no stop.
      - Max 1 trade per session.
    """
    df = compute_opening_range(df15)
    trades = []

    for ses, sdf in df.groupby("Session", sort=True):
        if len(sdf) < 3:
            continue

        orh = float(sdf["ORH"].iloc[0])
        orl = float(sdf["ORL"].iloc[0])
        if not np.isfinite(orh) or not np.isfinite(orl):
            continue

        # Find first breakout close after the OR bar
        after = sdf.iloc[1:].copy()
        long_break  = after[after["Close"] > orh].head(1)
        short_break = after[after["Close"] < orl].head(1)
        if long_break.empty and short_break.empty:
            continue

        if not long_break.empty and not short_break.empty:
            direction = "long" if long_break.index[0] < short_break.index[0] else "short"
            br_row = long_break.iloc[0] if direction == "long" else short_break.iloc[0]
        elif not long_break.empty:
            direction = "long"; br_row = long_break.iloc[0]
        else:
            direction = "short"; br_row = short_break.iloc[0]

        btime = br_row.name
        # Look for retest touch of the OR level (no confirm-close rule)
        level = orh if direction == "long" else orl
        window = sdf[sdf.index > btime].copy()
        touch = window[(window["Low"] <= level) & (window["High"] >= level)].head(1)
        if touch.empty:
            continue

        retest_bar = touch.iloc[0]
        entry_time = retest_bar.name
        # Enter at level with slippage
        entry = apply_slippage(level, SLIPPAGE_BPS, "buy" if direction=="long" else "sell")
        stop  = orl if direction=="long" else orh
        rps   = abs(entry - stop)
        if rps <= 1e-12:
            continue

        qty = POSITION_SIZE_DOLLARS / max(entry, 1e-12)
        side_mult = 1 if direction=="long" else -1

        # Simulate forward from entry bar: stop first, else EOD
        run = sdf[sdf.index >= entry_time].copy()
        exits = []
        stopped = False
        for ts, row in run.iterrows():
            hi, lo = float(row["High"]), float(row["Low"])
            if lo <= stop <= hi:
                px = apply_slippage(stop, SLIPPAGE_BPS, "sell" if direction=="long" else "buy")
                exits.append(("stop", ts, px, qty))
                stopped = True
                break

        if not stopped:
            last = run.iloc[-1]
            px = apply_slippage(float(last["Close"]), SLIPPAGE_BPS, "sell" if direction=="long" else "buy")
            exits.append(("eod", run.index[-1], px, qty))

        cash_pnl = sum((px - entry) * side_mult * q for (_, _, px, q) in exits) - FEES_PER_TRADE

        trades.append({
            "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
            "Session": ses,
            "Direction": direction,
            "ORH": orh, "ORL": orl,
            "EntryTime": entry_time,
            "Entry": entry, "Stop": stop,
            "Exits": exits,
            "Qty": qty,
            "PnL_$": cash_pnl,
            "R_multiple": cash_pnl / (rps * max(qty, 1e-12)),
        })

        # Pure spec: only one trade per session
        # (we break after recording the first valid trade)
    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Runner
# =============================
def run_folder(folder: str):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"Found {len(files)} parquet files.")
    all_trades, all_equity = [], []

    for i, fpath in enumerate(files, 1):
        tk = os.path.splitext(os.path.basename(fpath))[0].upper().replace("_","").replace("-","")
        print(f"\n== {tk} ({i}/{len(files)}) ==\n{fpath}")
        try:
            df15 = load_parquet_15m(fpath)
            if df15.empty:
                print("Skipped: data not suitable for 15m ORB.")
                continue
            df15["Ticker"] = tk
        except Exception as e:
            print(f"Skipping {tk}: {e}")
            continue

        tlog, eq = backtest_orb_retest_pure(df15)

        if not tlog.empty:
            out = tlog.copy()
            out["Exits"] = out["Exits"].apply(lambda xs: ";".join(f"{t}|{ts}|{px:.6f}|{q:.6f}" for (t, ts, px, q) in xs))
            all_trades.append(out)
        if not eq.empty:
            eq2 = eq.copy(); eq2["Ticker"] = tk
            all_equity.append(eq2)

        if AUTOSAVE_EVERY and i % AUTOSAVE_EVERY == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

        time.sleep(0.2)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

def print_overall_totals(trades: pd.DataFrame):
    print("\n=== Overall Totals ===")
    if trades is None or trades.empty:
        print("Total trades: 0")
        print("Total PnL ($): 0.00")
        print("Win rate: 0.00%")
        print("Avg R: 0.000 | Median R: 0.000")
        print("Profit Factor: 0.000")
        print("Avg Win ($): 0.00 | Avg Loss ($): 0.00")
        return
    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0)
    wins = pnl > 0; losses = pnl < 0
    total_trades = int(len(pnl))
    total_pnl = float(pnl.sum())
    win_rate = float(wins.mean()*100.0) if total_trades else 0.0
    avg_r = float(pd.to_numeric(trades["R_multiple"], errors="coerce").mean())
    med_r = float(pd.to_numeric(trades["R_multiple"], errors="coerce").median())
    gp = float(pnl[wins].sum()); gl = float(-pnl[losses].sum())
    pf = (gp / gl) if gl > 0 else (float("inf") if gp > 0 else 0.0)
    print(f"Total trades: {total_trades}")
    print(f"Total PnL ($): {total_pnl:,.2f}")
    print(f"Win rate: {win_rate:.2f}%")
    print(f"Avg R: {avg_r:.3f} | Median R: {med_r:.3f}")
    print(f"Profit Factor: {pf:.3f}")

def main():
    trades, equity = run_folder(PARQUET_DIR)

    print_overall_totals(trades)

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

    if not trades.empty:
        summary = (
            trades.groupby("Ticker")["PnL_$"]
                .agg(['count','sum','mean','std'])
                .rename(columns={'count':'Trades','sum':'PnL_sum','mean':'AvgPnL','std':'StdPnL'})
                .reset_index()
                .sort_values("PnL_sum", ascending=False)
        )
        print("\n=== Summary by Ticker (top 50) ===")
        print(summary.head(50).to_string(index=False))
        summary.to_csv(SUMMARY_CSV, index=False)
    else:
        print("No trades generated with current settings.")

if __name__ == "__main__":
    main()


Found 20 parquet files.

== AAPL (1/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AAPL.parquet

== AMD (2/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMD.parquet

== AMZN (3/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMZN.parquet

== CAT (4/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CAT.parquet

== CHWY (5/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CHWY.parquet

== CVNA (6/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CVNA.parquet

== GLD (7/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GLD.parquet

== GOOGL (8/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GOOGL.parquet

== GS (9/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GS.parquet

== JPM (10/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\JPM.parquet

== MSFT (11/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\MSFT.pa

In [23]:
# orb_15min_retest_from_parquet.py

import warnings
warnings.filterwarnings("ignore")

import os, glob, time
from datetime import timedelta
from typing import List, Tuple
import numpy as np
import pandas as pd

# =============================
# USER PARAMETERS
# =============================

# Parquet directory (ALL files inside will be processed)
PARQUET_DIR = r"C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet"

# If timestamps in your parquet are NAIVE and represent UTC, keep True
ASSUME_NAIVE_TIMESTAMPS_ARE_UTC = True

# Session / timezone
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (IDENTICAL BEHAVIOR)
RETEST_CONFIRM_CLOSE = True          # retest bar must CLOSE back through the level
MAX_RETEST_MIN = 120                 # minutes after breakout to wait for retest

# Risk / exits (IDENTICAL)
R_MULTIPLES = [1.0, 2.0]             # 1R and 2R targets
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00

# Batch / persistence (keep the autosave cadence)
AUTOSAVE_EVERY = 25
SLEEP_BETWEEN_TICKERS = 0.2

# Output files (same filenames for continuity)
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"

# =============================
# Helpers
# =============================
STD_MAP = {"open":"Open","high":"High","low":"Low","close":"Close","volume":"Volume"}

def _standardize_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    lower = {c.lower(): c for c in df.columns}
    for k, std in STD_MAP.items():
        if k in lower:                      rename[lower[k]] = std
        elif k.capitalize() in df.columns:  rename[k.capitalize()] = std
        elif k.upper() in df.columns:       rename[k.upper()] = std
    out = df.rename(columns=rename)
    drop_cols = [c for c in out.columns if str(c).lower().startswith("adj")]
    return out.drop(columns=drop_cols, errors="ignore")

def _choose_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for cand in ["EventAt","Datetime","datetime","Timestamp","timestamp","Date","date","Time","time"]:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            df = df.set_index(cand)
            break
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No datetime index/column found.")
    return df

def _infer_median_interval_minutes(idx: pd.DatetimeIndex) -> float:
    if len(idx) < 2: return np.nan
    d = (idx[1:] - idx[:-1]).total_seconds() / 60.0
    return float(np.median(d)) if len(d) else np.nan

def _resample_to_15m_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    """
    Match Yahoo 15m behavior:
      - If bars are finer than 15m, resample to exact 15m OHLC(V)
        using right-closed, right-labeled bins so the first bar is 09:45.
      - If already ~15m, keep as-is.
      - If coarser than 15m, skip (return empty for this strategy).
    """
    m = _infer_median_interval_minutes(df.index)
    if np.isnan(m): return pd.DataFrame()
    if m < 15 - 1e-6:
        agg = {"Open":"first","High":"max","Low":"min","Close":"last"}
        if "Volume" in df.columns: agg["Volume"] = "sum"
        out = (df.sort_index()
                 .resample("15T", label="right", closed="right")
                 .agg(agg)
                 .dropna(subset=["Open","High","Low","Close"]))
        return out
    elif abs(m - 15) <= 1e-6:
        return df.sort_index()
    else:
        return pd.DataFrame()  # coarser than 15m → not usable here

def load_parquet_15m(path: str, tz: str = TZ) -> pd.DataFrame:
    df = pd.read_parquet(path, engine="pyarrow")
    df = _choose_dt_index(df).sort_index()

    # TZ handling
    if df.index.tz is None:
        if ASSUME_NAIVE_TIMESTAMPS_ARE_UTC:
            df = df.tz_localize("UTC").tz_convert(tz)
        else:
            df = df.tz_localize(tz)
    else:
        df = df.tz_convert(tz)

    df = _standardize_ohlcv_columns(df)
    for need in ["Open","High","Low","Close"]:
        if need not in df.columns:
            raise ValueError(f"{os.path.basename(path)} missing required column: {need}")

    df = _resample_to_15m_if_needed(df)
    if df.empty:
        return df

    # Regular session only (matches the original)
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    return df.sort_index()

def sessionize(df_15: pd.DataFrame) -> pd.Series:
    return pd.to_datetime(df_15.index.date)

# =============================
# ORB logic 
# =============================
def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = df_15.copy()

    # Ensure OR columns exist
    for col in ["ORH", "ORL"]:
        if col not in df.columns:
            df[col] = np.nan

    df["Session"] = sessionize(df)

    # First 15m bar per day
    first_bar_idx = df.groupby("Session").head(1).index
    df.loc[first_bar_idx, "ORH"] = df.loc[first_bar_idx, "High"].astype(float)
    df.loc[first_bar_idx, "ORL"] = df.loc[first_bar_idx, "Low"].astype(float)

    # Forward-fill within session
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return df

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = R_MULTIPLES,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Exact behavior from orb_15min_retest_sp500.py:
      - Opening range = first 15m bar high/low
      - First breakout CLOSE beyond ORH/ORL
      - Wait up to MAX_RETEST_MIN for retest
      - If RETEST_CONFIRM_CLOSE=True, retest bar must CLOSE back across the level
      - Entry at the OR level (with slippage); stop = opposite OR
      - Exits: stop FIRST, else first TP hit (1R then 2R), else EOD
      - PnL splits size equally among realized exit legs
    """
    df = df_15.copy()
    df["Session"] = sessionize(df)
    sessions = df["Session"].unique()

    trades = []
    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh = sdf["ORH"].iloc[0]
        orl = sdf["ORL"].iloc[0]
        if pd.isna(orh) or pd.isna(orl):
            continue

        # First breakout CLOSE after the opening bar
        sdf_after = sdf.iloc[1:].copy()
        long_break  = sdf_after[sdf_after["Close"] > orh].head(1)
        short_break = sdf_after[sdf_after["Close"] < orl].head(1)

        if long_break.empty and short_break.empty:
            continue

        if not long_break.empty and not short_break.empty:
            direction = "long" if long_break.index[0] < short_break.index[0] else "short"
            breakout_row = long_break.iloc[0] if direction == "long" else short_break.iloc[0]
        elif not long_break.empty:
            direction = "long"; breakout_row = long_break.iloc[0]
        else:
            direction = "short"; breakout_row = short_break.iloc[0]

        # Retest within window
        btime = breakout_row.name
        cutoff = btime + timedelta(minutes=max_retest_min)
        after_break = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
        if after_break.empty:
            continue

        level = orh if direction == "long" else orl
        touch = after_break[(after_break["Low"] <= level) & (after_break["High"] >= level)].head(1)
        if touch.empty:
            continue

        retest_bar = touch.iloc[0]
        retest_time = retest_bar.name

        if retest_confirm_close:
            ok = (retest_bar["Close"] >= level) if direction == "long" else (retest_bar["Close"] <= level)
            if not ok:
                continue

        entry = _apply_slippage(level, slippage_bps, "buy" if direction == "long" else "sell")
        stop = orl if direction == "long" else orh
        risk_per_share = abs(entry - stop)
        if risk_per_share <= 1e-12:
            continue

        qty = dollars / entry
        side_mult = 1 if direction == "long" else -1

        # Target prices (1R, 2R, ...)
        t_prices = [(entry + r*risk_per_share) if direction == "long" else (entry - r*risk_per_share) for r in r_targets]

        # Forward simulate exits
        sdf_run = sdf[sdf.index >= retest_time].copy()
        exits = []
        for ts, row in sdf_run.iterrows():
            high, low = float(row["High"]), float(row["Low"])
            # Stop first
            if low <= stop <= high:
                px = _apply_slippage(stop, slippage_bps, "sell" if direction == "long" else "buy")
                exits.append(("stop", ts, px))
                break
            # Targets (first hit wins)
            hit = False
            for i, tp in enumerate(t_prices):
                if low <= tp <= high:
                    px = _apply_slippage(tp, slippage_bps, "sell" if direction == "long" else "buy")
                    exits.append((f"tp{i+1}", ts, px))
                    hit = True
            if hit:
                break

        if not exits:
            # Flat on last bar close (EOD)
            last = sdf_run.iloc[-1]
            px = _apply_slippage(float(last["Close"]), slippage_bps, "sell" if direction == "long" else "buy")
            exits.append(("eod", sdf_run.index[-1], px))

        # Equal partials among realized exits
        n_parts = len(exits)
        qty_each = qty / n_parts
        cash_pnl = 0.0
        for tag, ts, px in exits:
            cash_pnl += (px - entry) * side_mult * qty_each
        cash_pnl -= FEES_PER_TRADE

        trades.append({
            "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
            "Session": ses,
            "ORH": float(orh), "ORL": float(orl),
            "Direction": direction,
            "EntryTime": retest_time,
            "Entry": float(entry), "Stop": float(stop),
            "Targets": [float(x) for x in t_prices],
            "Exits": [(str(t[0]), t[1], float(t[2])) for t in exits],
            "Qty": float(qty),
            "PnL_$": float(cash_pnl),
            "R_multiple": float(cash_pnl / (risk_per_share * qty))
        })

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Runner (folder of parquet files)
# =============================
def run_folder(folder: str):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"Found {len(files)} parquet files.")
    all_trades, all_equity = [], []

    for i, fpath in enumerate(files, 1):
        ticker = os.path.splitext(os.path.basename(fpath))[0].upper().replace("_","").replace("-","")
        print(f"\n== {ticker} ({i}/{len(files)}) ==\n{fpath}")
        try:
            df15 = load_parquet_15m(fpath, tz=TZ)
            if df15.empty:
                print("Skipped (no 15m data after normalization/resample/session filter).")
                continue
            df15["Ticker"] = ticker
            df15 = compute_opening_range(df15)
        except Exception as e:
            print(f"Skipping {ticker}: {e}")
            continue

        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            r_targets=R_MULTIPLES,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=SLIPPAGE_BPS,
            fees=FEES_PER_TRADE,
            retest_confirm_close=RETEST_CONFIRM_CLOSE
        )

        if not tlog.empty:
            out = tlog.copy()
            out["Targets"] = out["Targets"].apply(lambda xs: ";".join(f"{p:.6f}" for p in xs))
            out["Exits"] = out["Exits"].apply(lambda xs: ";".join(f"{t}|{ts}|{px:.6f}" for (t, ts, px) in xs))
            all_trades.append(out)
        if not eq.empty:
            eq2 = eq.copy(); eq2["Ticker"] = ticker
            all_equity.append(eq2)

        if AUTOSAVE_EVERY and i % AUTOSAVE_EVERY == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

        time.sleep(SLEEP_BETWEEN_TICKERS)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

    return trades, equity

# =============================
# Main
# =============================
def main():
    trades, equity = run_folder(PARQUET_DIR)

    # Overall totals (same style as original)
    if trades is not None and not trades.empty:
        total_trades = len(trades)
        total_pnl    = trades["PnL_$"].sum()
        win_rate     = (trades["PnL_$"] > 0).mean() * 100.0
        avg_r        = trades["R_multiple"].mean()
        median_r     = trades["R_multiple"].median()

        print("\n=== Overall Totals ===")
        print(f"Total trades: {total_trades}")
        print(f"Total PnL ($): {total_pnl:,.2f}")
        print(f"Win rate: {win_rate:.2f}%")
        print(f"Avg R: {avg_r:.3f} | Median R: {median_r:.3f}")
    else:
        print("No trades generated with current settings.")

if __name__ == "__main__":
    main()


Found 20 parquet files.

== AAPL (1/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AAPL.parquet

== AMD (2/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMD.parquet

== AMZN (3/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMZN.parquet

== CAT (4/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CAT.parquet

== CHWY (5/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CHWY.parquet

== CVNA (6/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CVNA.parquet

== GLD (7/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GLD.parquet

== GOOGL (8/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GOOGL.parquet

== GS (9/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GS.parquet

== JPM (10/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\JPM.parquet

== MSFT (11/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\MSFT.pa

In [25]:
# orb_15min_retest_from_parquet_with_plots.py
# Same ORB 15m Retest strategy as orb_15min_retest_sp500.py, but reads local parquet files
# AND produces per-day ORB charts + overall equity curve PNGs.

import warnings
warnings.filterwarnings("ignore")

import os, glob, time
from datetime import timedelta
from typing import List, Tuple
import numpy as np
import pandas as pd

# =============================
# USER PARAMETERS
# =============================

# Parquet directory (ALL files inside will be processed)
PARQUET_DIR = r"C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet"

# If timestamps in your parquet are NAIVE and represent UTC, keep True
ASSUME_NAIVE_TIMESTAMPS_ARE_UTC = True

# Session / timezone
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (matches orb_15min_retest_sp500.py)
RETEST_CONFIRM_CLOSE = True
MAX_RETEST_MIN = 120

# Risk / exits (matches orb_15min_retest_sp500.py)
R_MULTIPLES = [1.0, 2.0]
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00

# Batch / persistence
AUTOSAVE_EVERY = 25
SLEEP_BETWEEN_TICKERS = 0.2

# Output files
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"

# ===== Plotting =====
PLOT_MAX_DAYS_PER_TICKER = 3   # 0 = disable day charts
PLOT_SHOW = False              # True = plt.show(); False = save only
PLOT_SAVE_DIR = "orb_plots"
PLOT_EQUITY = True

# =============================
# Helpers
# =============================
STD_MAP = {"open":"Open","high":"High","low":"Low","close":"Close","volume":"Volume"}

def _standardize_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    lower = {c.lower(): c for c in df.columns}
    for k, std in STD_MAP.items():
        if k in lower:                      rename[lower[k]] = std
        elif k.capitalize() in df.columns:  rename[k.capitalize()] = std
        elif k.upper() in df.columns:       rename[k.upper()] = std
    out = df.rename(columns=rename)
    drop_cols = [c for c in out.columns if str(c).lower().startswith("adj")]
    return out.drop(columns=drop_cols, errors="ignore")

def _choose_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for cand in ["EventAt","Datetime","datetime","Timestamp","timestamp","Date","date","Time","time"]:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            df = df.set_index(cand)
            break
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No datetime index/column found.")
    return df

def _infer_median_interval_minutes(idx: pd.DatetimeIndex) -> float:
    if len(idx) < 2: return np.nan
    d = (idx[1:] - idx[:-1]).total_seconds() / 60.0
    return float(np.median(d)) if len(d) else np.nan

def _resample_to_15m_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    """
    If bars finer than 15m -> resample to 15m OHLC(V) using right-closed/right-labeled bins.
    If ~15m -> keep.
    If coarser -> return empty (not suitable for 15m ORB).
    """
    m = _infer_median_interval_minutes(df.index)
    if np.isnan(m): return pd.DataFrame()
    if m < 15 - 1e-6:
        agg = {"Open":"first","High":"max","Low":"min","Close":"last"}
        if "Volume" in df.columns: agg["Volume"] = "sum"
        out = (df.sort_index()
                 .resample("15T", label="right", closed="right")
                 .agg(agg)
                 .dropna(subset=["Open","High","Low","Close"]))
        return out
    elif abs(m - 15) <= 1e-6:
        return df.sort_index()
    else:
        return pd.DataFrame()

def load_parquet_15m(path: str, tz: str = TZ) -> pd.DataFrame:
    df = pd.read_parquet(path, engine="pyarrow")
    df = _choose_dt_index(df).sort_index()

    # TZ handling
    if df.index.tz is None:
        if ASSUME_NAIVE_TIMESTAMPS_ARE_UTC:
            df = df.tz_localize("UTC").tz_convert(tz)
        else:
            df = df.tz_localize(tz)
    else:
        df = df.tz_convert(tz)

    df = _standardize_ohlcv_columns(df)
    for need in ["Open","High","Low","Close"]:
        if need not in df.columns:
            raise ValueError(f"{os.path.basename(path)} missing required column: {need}")

    df = _resample_to_15m_if_needed(df)
    if df.empty:
        return df

    # Regular session only
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    return df.sort_index()

def sessionize(df_15: pd.DataFrame) -> pd.Series:
    return pd.to_datetime(df_15.index.date)

# =============================
# ORB logic (same as your Yahoo version)
# =============================
def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = df_15.copy()
    for col in ["ORH","ORL"]:
        if col not in df.columns:
            df[col] = np.nan
    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df.loc[first_bar_idx, "ORH"] = df.loc[first_bar_idx, "High"].astype(float)
    df.loc[first_bar_idx, "ORL"] = df.loc[first_bar_idx, "Low"].astype(float)
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return df

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = R_MULTIPLES,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    - OR = first 15m bar high/low
    - First breakout CLOSE beyond ORH/ORL
    - Wait up to max_retest_min for retest (touch of level)
    - If retest_confirm_close: retest bar must CLOSE back across the level
    - Entry at level (with slippage); stop = opposite OR
    - Exits: stop first, else first TP hit (1R then 2R), else EOD
    - Equal partials across realized exits
    """
    df = df_15.copy()
    df["Session"] = sessionize(df)
    sessions = df["Session"].unique()

    trades = []
    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh = sdf["ORH"].iloc[0]
        orl = sdf["ORL"].iloc[0]
        if pd.isna(orh) or pd.isna(orl):
            continue

        # First breakout CLOSE after the opening bar
        sdf_after = sdf.iloc[1:].copy()
        long_break  = sdf_after[sdf_after["Close"] > orh].head(1)
        short_break = sdf_after[sdf_after["Close"] < orl].head(1)

        if long_break.empty and short_break.empty:
            continue

        if not long_break.empty and not short_break.empty:
            direction = "long" if long_break.index[0] < short_break.index[0] else "short"
            breakout_row = long_break.iloc[0] if direction == "long" else short_break.iloc[0]
        elif not long_break.empty:
            direction = "long"; breakout_row = long_break.iloc[0]
        else:
            direction = "short"; breakout_row = short_break.iloc[0]

        # Retest within window
        btime = breakout_row.name
        cutoff = btime + timedelta(minutes=max_retest_min)
        after_break = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
        if after_break.empty:
            continue

        level = orh if direction == "long" else orl
        touch = after_break[(after_break["Low"] <= level) & (after_break["High"] >= level)].head(1)
        if touch.empty:
            continue

        retest_bar = touch.iloc[0]
        retest_time = retest_bar.name

        if retest_confirm_close:
            ok = (retest_bar["Close"] >= level) if direction == "long" else (retest_bar["Close"] <= level)
            if not ok:
                continue

        entry = _apply_slippage(level, slippage_bps, "buy" if direction == "long" else "sell")
        stop = orl if direction == "long" else orh
        risk_per_share = abs(entry - stop)
        if risk_per_share <= 1e-12:
            continue

        qty = dollars / max(entry, 1e-12)
        side_mult = 1 if direction == "long" else -1

        # Targets
        t_prices = [(entry + r*risk_per_share) if direction=="long" else (entry - r*risk_per_share) for r in r_targets]

        # Forward simulate exits
        sdf_run = sdf[sdf.index >= retest_time].copy()
        exits = []
        for ts, row in sdf_run.iterrows():
            high, low = float(row["High"]), float(row["Low"])
            # Stop first
            if low <= stop <= high:
                px = _apply_slippage(stop, slippage_bps, "sell" if direction=="long" else "buy")
                exits.append(("stop", ts, px))
                break
            # Targets (first hit wins)
            hit = False
            for i, tp in enumerate(t_prices):
                if low <= tp <= high:
                    px = _apply_slippage(tp, slippage_bps, "sell" if direction=="long" else "buy")
                    exits.append((f"tp{i+1}", ts, px))
                    hit = True
            if hit:
                break

        if not exits:
            last = sdf_run.iloc[-1]
            px = _apply_slippage(float(last["Close"]), slippage_bps, "sell" if direction=="long" else "buy")
            exits.append(("eod", sdf_run.index[-1], px))

        # Equal partials across realized exits
        n_parts = len(exits)
        qty_each = qty / n_parts
        cash_pnl = 0.0
        for tag, ts, px in exits:
            cash_pnl += (px - entry) * side_mult * qty_each
        cash_pnl -= FEES_PER_TRADE

        trades.append({
            "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
            "Session": ses,
            "Direction": direction,
            "ORH": float(orh), "ORL": float(orl),
            "EntryTime": retest_time,
            "Entry": float(entry), "Stop": float(stop),
            "Targets": [float(x) for x in t_prices],
            "Exits": [(str(t[0]), t[1], float(t[2])) for t in exits],
            "Qty": float(qty),
            "PnL_$": float(cash_pnl),
            "R_multiple": float(cash_pnl / (risk_per_share * qty))
        })

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Plotting
# =============================
def _safe_import_matplotlib():
    try:
        import matplotlib.pyplot as plt
        return plt
    except Exception as _:
        return None

def plot_example_days(df_15: pd.DataFrame, trade_log: pd.DataFrame, ticker: str,
                      max_days: int = 0, save_dir: str = "orb_plots", show: bool = False):
    if max_days <= 0 or trade_log.empty:
        return
    plt = _safe_import_matplotlib()
    if plt is None:
        print("[plot] matplotlib not available; skipping plots.")
        return

    os.makedirs(save_dir, exist_ok=True)
    sessions = trade_log["Session"].drop_duplicates().sort_values().tolist()[:max_days]

    for ses in sessions:
        sdf = df_15[df_15.index.date == pd.to_datetime(ses).date()]
        if sdf.empty:
            continue
        # pick the trade for this session
        tr = trade_log[trade_log["Session"] == ses].iloc[0]
        plt.figure(figsize=(11, 5))
        plt.plot(sdf.index, sdf["Close"], label="Close")
        # OR lines
        plt.axhline(tr["ORH"], linestyle="--", label="ORH")
        plt.axhline(tr["ORL"], linestyle="--", label="ORL")
        # Entry marker
        m = "^" if tr["Direction"] == "long" else "v"
        plt.scatter([tr["EntryTime"]], [tr["Entry"]], marker=m, s=80, label="Entry")
        # Exits
        for tag, tstamp, px in tr["Exits"]:
            plt.scatter([tstamp], [px], marker="x", s=80, label=f"Exit {tag}")
        plt.title(f"{ticker} {pd.to_datetime(ses).date()} ORB Retest (15m)")
        plt.legend()
        plt.tight_layout()
        out_path = os.path.join(save_dir, f"{ticker}_{pd.to_datetime(ses).date()}.png")
        plt.savefig(out_path, dpi=130)
        if show:
            plt.show()
        plt.close()
        print(f"[plot] saved {out_path}")

def plot_overall_equity(equity_df: pd.DataFrame, save_dir: str = "orb_plots", show: bool = False):
    if equity_df is None or equity_df.empty:
        return
    plt = _safe_import_matplotlib()
    if plt is None:
        print("[plot] matplotlib not available; skipping equity plot.")
        return
    os.makedirs(save_dir, exist_ok=True)
    plt.figure(figsize=(11, 4))
    plt.plot(pd.to_datetime(equity_df["Session"]), equity_df["CumPnL_$"])
    plt.title("Overall Equity Curve (Cum PnL $)")
    plt.xlabel("Session")
    plt.ylabel("CumPnL $")
    plt.tight_layout()
    out_path = os.path.join(save_dir, "equity_overall.png")
    plt.savefig(out_path, dpi=130)
    if show:
        plt.show()
    plt.close()
    print(f"[plot] saved {out_path}")

# =============================
# Runner (folder of parquet files)
# =============================
def run_folder(folder: str):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"Found {len(files)} parquet files.")
    all_trades, all_equity = [], []

    for i, fpath in enumerate(files, 1):
        ticker = os.path.splitext(os.path.basename(fpath))[0].upper().replace("_","").replace("-","")
        print(f"\n== {ticker} ({i}/{len(files)}) ==\n{fpath}")
        try:
            df15 = load_parquet_15m(fpath, tz=TZ)
            if df15.empty:
                print("Skipped (no 15m data after normalization/resample/session filter).")
                continue
            df15["Ticker"] = ticker
            df15 = compute_opening_range(df15)
        except Exception as e:
            print(f"Skipping {ticker}: {e}")
            continue

        # Run backtest
        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            r_targets=R_MULTIPLES,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=SLIPPAGE_BPS,
            fees=FEES_PER_TRADE,
            retest_confirm_close=RETEST_CONFIRM_CLOSE
        )

        # --- Plot per-day examples BEFORE serializing exits ---
        if PLOT_MAX_DAYS_PER_TICKER > 0 and not tlog.empty:
            plot_example_days(df15, tlog, ticker,
                              max_days=PLOT_MAX_DAYS_PER_TICKER,
                              save_dir=PLOT_SAVE_DIR,
                              show=PLOT_SHOW)

        # Save outputs
        if not tlog.empty:
            out = tlog.copy()
            out["Targets"] = out["Targets"].apply(lambda xs: ";".join(f"{p:.6f}" for p in xs))
            out["Exits"] = out["Exits"].apply(lambda xs: ";".join(f"{t}|{ts}|{px:.6f}" for (t, ts, px) in xs))
            all_trades.append(out)
        if not eq.empty:
            eq2 = eq.copy(); eq2["Ticker"] = ticker
            all_equity.append(eq2)

        if AUTOSAVE_EVERY and i % AUTOSAVE_EVERY == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

        time.sleep(SLEEP_BETWEEN_TICKERS)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

# =============================
# Main
# =============================
def main():
    trades, equity = run_folder(PARQUET_DIR)

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

    if not equity.empty:
        # overall equity is already aggregated within each ticker chunk; aggregate again overall
        overall = (trades.groupby("Session")["PnL_$"].sum()
                         .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
        if PLOT_EQUITY:
            plot_overall_equity(overall, save_dir=PLOT_SAVE_DIR, show=PLOT_SHOW)

        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")
    else:
        print("No trades generated with current settings.")

if __name__ == "__main__":
    main()


Found 20 parquet files.

== AAPL (1/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AAPL.parquet
[plot] saved orb_plots\AAPL_2016-01-04.png
[plot] saved orb_plots\AAPL_2016-01-06.png
[plot] saved orb_plots\AAPL_2016-01-11.png

== AMD (2/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMD.parquet
[plot] saved orb_plots\AMD_2016-01-05.png
[plot] saved orb_plots\AMD_2016-01-06.png
[plot] saved orb_plots\AMD_2016-01-12.png

== AMZN (3/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMZN.parquet
[plot] saved orb_plots\AMZN_2016-01-07.png
[plot] saved orb_plots\AMZN_2016-01-12.png
[plot] saved orb_plots\AMZN_2016-01-14.png

== CAT (4/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CAT.parquet
[plot] saved orb_plots\CAT_2016-01-07.png
[plot] saved orb_plots\CAT_2016-01-13.png
[plot] saved orb_plots\CAT_2016-01-14.png

== CHWY (5/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CHWY.parquet
[plot] saved orb_plo

In [33]:
# orb_15min_retest_from_parquet_with_equity_losses.py
# ORB 15m Retest strategy (same logic), plus equity curve with losses/drawdown + optional bear shading.

import warnings
warnings.filterwarnings("ignore")

import os, glob, time
from datetime import timedelta
from typing import List, Tuple
import numpy as np
import pandas as pd

# =============================
# USER PARAMETERS
# =============================
PARQUET_DIR = r"C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet"

ASSUME_NAIVE_TIMESTAMPS_ARE_UTC = True
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (kept identical to your Yahoo version)
RETEST_CONFIRM_CLOSE = True
MAX_RETEST_MIN = 120

# Risk / exits (kept identical)
R_MULTIPLES = [1.0, 2.0]
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00

# Batch / persistence
AUTOSAVE_EVERY = 25
SLEEP_BETWEEN_TICKERS = 0.2

# Output files
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"

# Plotting
PLOT_SAVE_DIR = "orb_plots"
PLOT_SHOW = False                 # set True to display interactively
PLOT_EXAMPLES_PER_TICKER = 0      # set >0 to also save day-level ORB charts
SHADE_BEAR_WITH_SPY = True        # requires SPY.parquet in the folder

# =============================
# Helpers
# =============================
STD_MAP = {"open":"Open","high":"High","low":"Low","close":"Close","volume":"Volume"}

def _standardize_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    lower = {c.lower(): c for c in df.columns}
    for k, std in STD_MAP.items():
        if k in lower:                      rename[lower[k]] = std
        elif k.capitalize() in df.columns:  rename[k.capitalize()] = std
        elif k.upper() in df.columns:       rename[k.upper()] = std
    out = df.rename(columns=rename)
    drop_cols = [c for c in out.columns if str(c).lower().startswith("adj")]
    return out.drop(columns=drop_cols, errors="ignore")

def _choose_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for cand in ["EventAt","Datetime","datetime","Timestamp","timestamp","Date","date","Time","time"]:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            df = df.set_index(cand)
            break
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No datetime index/column found.")
    return df

def _infer_median_interval_minutes(idx: pd.DatetimeIndex) -> float:
    if len(idx) < 2: return np.nan
    d = (idx[1:] - idx[:-1]).total_seconds() / 60.0
    return float(np.median(d)) if len(d) else np.nan

def _resample_to_15m_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    m = _infer_median_interval_minutes(df.index)
    if np.isnan(m): return pd.DataFrame()
    if m < 15 - 1e-6:
        agg = {"Open":"first","High":"max","Low":"min","Close":"last"}
        if "Volume" in df.columns: agg["Volume"] = "sum"
        out = (df.sort_index()
                 .resample("15T", label="right", closed="right")
                 .agg(agg)
                 .dropna(subset=["Open","High","Low","Close"]))
        return out
    elif abs(m - 15) <= 1e-6:
        return df.sort_index()
    else:
        return pd.DataFrame()

def load_parquet_15m(path: str, tz: str = TZ) -> pd.DataFrame:
    df = pd.read_parquet(path, engine="pyarrow")
    df = _choose_dt_index(df).sort_index()
    if df.index.tz is None:
        if ASSUME_NAIVE_TIMESTAMPS_ARE_UTC:
            df = df.tz_localize("UTC").tz_convert(tz)
        else:
            df = df.tz_localize(tz)
    else:
        df = df.tz_convert(tz)
    df = _standardize_ohlcv_columns(df)
    for need in ["Open","High","Low","Close"]:
        if need not in df.columns:
            raise ValueError(f"{os.path.basename(path)} missing required column: {need}")
    df = _resample_to_15m_if_needed(df)
    if df.empty: return df
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    return df.sort_index()

def sessionize(df_15: pd.DataFrame) -> pd.Series:
    return pd.to_datetime(df_15.index.date)

# =============================
# ORB logic (same as your Yahoo version)
# =============================
def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = df_15.copy()
    for col in ["ORH","ORL"]:
        if col not in df.columns:
            df[col] = np.nan
    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df.loc[first_bar_idx, "ORH"] = df.loc[first_bar_idx, "High"].astype(float)
    df.loc[first_bar_idx, "ORL"] = df.loc[first_bar_idx, "Low"].astype(float)
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return df

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = R_MULTIPLES,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = df_15.copy()
    df["Session"] = sessionize(df)
    sessions = df["Session"].unique()

    trades = []
    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh = sdf["ORH"].iloc[0]
        orl = sdf["ORL"].iloc[0]
        if pd.isna(orh) or pd.isna(orl):
            continue

        sdf_after = sdf.iloc[1:].copy()
        long_break  = sdf_after[sdf_after["Close"] > orh].head(1)
        short_break = sdf_after[sdf_after["Close"] < orl].head(1)
        if long_break.empty and short_break.empty:
            continue

        if not long_break.empty and not short_break.empty:
            direction = "long" if long_break.index[0] < short_break.index[0] else "short"
            breakout_row = long_break.iloc[0] if direction=="long" else short_break.iloc[0]
        elif not long_break.empty:
            direction = "long"; breakout_row = long_break.iloc[0]
        else:
            direction = "short"; breakout_row = short_break.iloc[0]

        btime = breakout_row.name
        cutoff = btime + timedelta(minutes=max_retest_min)
        after_break = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
        if after_break.empty:
            continue

        level = orh if direction == "long" else orl
        touch = after_break[(after_break["Low"] <= level) & (after_break["High"] >= level)].head(1)
        if touch.empty:
            continue

        retest_bar = touch.iloc[0]
        retest_time = retest_bar.name

        if retest_confirm_close:
            ok = (retest_bar["Close"] >= level) if direction=="long" else (retest_bar["Close"] <= level)
            if not ok:
                continue

        entry = _apply_slippage(level, slippage_bps, "buy" if direction=="long" else "sell")
        stop = orl if direction=="long" else orh
        rps = abs(entry - stop)
        if rps <= 1e-12:
            continue

        qty = POSITION_SIZE_DOLLARS / entry
        side_mult = 1 if direction=="long" else -1

        t_prices = [(entry + r*rps) if direction=="long" else (entry - r*rps) for r in r_targets]

        sdf_run = sdf[sdf.index >= retest_time].copy()
        exits = []
        for ts, row in sdf_run.iterrows():
            high, low = float(row["High"]), float(row["Low"])
            if low <= stop <= high:
                px = _apply_slippage(stop, slippage_bps, "sell" if direction=="long" else "buy")
                exits.append(("stop", ts, px))
                break
            hit = False
            for i, tp in enumerate(t_prices):
                if low <= tp <= high:
                    px = _apply_slippage(tp, slippage_bps, "sell" if direction=="long" else "buy")
                    exits.append((f"tp{i+1}", ts, px))
                    hit = True
            if hit:
                break

        if not exits:
            last = sdf_run.iloc[-1]
            px = _apply_slippage(float(last["Close"]), slippage_bps, "sell" if direction=="long" else "buy")
            exits.append(("eod", sdf_run.index[-1], px))

        n_parts = len(exits)
        qty_each = qty / n_parts
        cash_pnl = sum((px - entry) * side_mult * qty_each for _, _, px in exits) - FEES_PER_TRADE

        trades.append({
            "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
            "Session": ses,
            "ORH": float(orh), "ORL": float(orl),
            "Direction": direction,
            "EntryTime": retest_time,
            "Entry": float(entry), "Stop": float(stop),
            "Targets": [float(x) for x in t_prices],
            "Exits": [(str(t[0]), t[1], float(t[2])) for t in exits],
            "Qty": float(qty),
            "PnL_$": float(cash_pnl),
            "R_multiple": float(cash_pnl / (rps * qty))
        })

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Plotting (equity + losing days + drawdown + optional bear shading)
# =============================
def _safe_import_matplotlib():
    try:
        import matplotlib.pyplot as plt
        return plt
    except Exception:
        return None

def _load_spy_daily_for_shading(folder: str) -> pd.DataFrame:
    """Optional: if SPY.parquet exists (any frequency), build daily close and drawdown."""
    spy_path = os.path.join(folder, "SPY.parquet")
    if not os.path.exists(spy_path):
        return pd.DataFrame()
    try:
        spy = load_parquet_15m(spy_path)  # if 15m available; otherwise fallback below
        if spy.empty:
            spy_raw = pd.read_parquet(spy_path, engine="pyarrow")
            spy_raw = _choose_dt_index(spy_raw)
            if spy_raw.index.tz is None:
                spy_raw = spy_raw.tz_localize("UTC").tz_convert(TZ)
            else:
                spy_raw = spy_raw.tz_convert(TZ)
            spy_raw = _standardize_ohlcv_columns(spy_raw)
            spy = spy_raw
        spy = spy.sort_index()
        dclose = spy["Close"].resample("1D").last().dropna()
        if dclose.empty:
            return pd.DataFrame()
        dd = (dclose / dclose.cummax() - 1.0)
        out = pd.DataFrame({"SPY_Close": dclose, "SPY_Drawdown": dd})
        return out
    except Exception:
        return pd.DataFrame()

def _bear_spans(df: pd.DataFrame):
    """Yield (start, end) spans where SPY_Drawdown <= -0.20."""
    if df is None or df.empty or "SPY_Drawdown" not in df.columns: return []
    bear = (df["SPY_Drawdown"] <= -0.20).astype(int)
    spans = []
    in_bear, start = False, None
    for ts, val in bear.items():
        if val and not in_bear:
            in_bear, start = True, ts
        elif not val and in_bear:
            in_bear = False
            spans.append((start, ts))
    if in_bear and start is not None:
        spans.append((start, df.index[-1]))
    return spans

def plot_equity_with_losses(trades: pd.DataFrame, folder: str, save_dir: str, show: bool):
    if trades is None or trades.empty:
        print("[plot] No trades — skipping equity plot.")
        return
    plt = _safe_import_matplotlib()
    if plt is None:
        print("[plot] matplotlib not available; skipping plots.")
        return
    os.makedirs(save_dir, exist_ok=True)

    daily = (trades.groupby("Session")["PnL_$"].sum()
                    .sort_index()
                    .reset_index())
    daily["Session"] = pd.to_datetime(daily["Session"])
    daily["CumPnL_$"] = daily["PnL_$"].cumsum()
    daily["Peak"] = daily["CumPnL_$"].cummax()
    daily["Drawdown_$"] = daily["CumPnL_$"] - daily["Peak"]

    spy_daily = _load_spy_daily_for_shading(folder) if SHADE_BEAR_WITH_SPY else pd.DataFrame()
    spans = _bear_spans(spy_daily)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                                   gridspec_kw={"height_ratios":[3,1]})

    ax1.plot(daily["Session"], daily["CumPnL_$"], linewidth=1.5, label="Equity (Cum PnL $)")
    losers = daily[daily["PnL_$"] < 0]
    ax1.scatter(losers["Session"], losers["CumPnL_$"], s=15, color="red", alpha=0.8, label="Losing sessions")

    for s, e in spans:
        ax1.axvspan(s, e, color="gray", alpha=0.15, label=None)

    ax1.set_title("ORB 15m Retest — Equity Curve (cum PnL)")
    ax1.set_ylabel("CumPnL ($)")
    ax1.grid(alpha=0.25)
    ax1.legend(loc="best")

    ax2.plot(daily["Session"], daily["Drawdown_$"], linewidth=1.2)
    ax2.fill_between(daily["Session"], daily["Drawdown_$"], 0, alpha=0.2)
    ax2.set_title("Drawdown ($)")
    ax2.set_ylabel("DD ($)")
    ax2.set_xlabel("Session")
    ax2.grid(alpha=0.25)

    out_path = os.path.join(save_dir, "equity_with_losses_and_dd.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=140)
    if show:
        plt.show()
    plt.close()
    print(f"[plot] saved {out_path}")

def plot_example_days(df_15: pd.DataFrame, trade_log: pd.DataFrame, ticker: str,
                      max_days: int, save_dir: str, show: bool):
    if max_days <= 0 or trade_log.empty: return
    plt = _safe_import_matplotlib()
    if plt is None:
        print("[plot] matplotlib not available; skipping day charts.")
        return
    os.makedirs(save_dir, exist_ok=True)
    sessions = trade_log["Session"].drop_duplicates().sort_values().tolist()[:max_days]
    for ses in sessions:
        sdf = df_15[df_15.index.date == pd.to_datetime(ses).date()]
        if sdf.empty: continue
        tr = trade_log[trade_log["Session"] == ses].iloc[0]
        plt.figure(figsize=(11,5))
        plt.plot(sdf.index, sdf["Close"], label="Close")
        plt.axhline(tr["ORH"], linestyle="--", label="ORH")
        plt.axhline(tr["ORL"], linestyle="--", label="ORL")
        m = "^" if tr["Direction"]=="long" else "v"
        plt.scatter([tr["EntryTime"]], [tr["Entry"]], marker=m, s=80, label="Entry")
        for tag, tstamp, px in tr["Exits"]:
            plt.scatter([tstamp], [px], marker="x", s=80, label=f"Exit {tag}")
        plt.title(f"{ticker} {pd.to_datetime(ses).date()} ORB Retest (15m)")
        plt.legend(); plt.tight_layout()
        out_path = os.path.join(save_dir, f"{ticker}_{pd.to_datetime(ses).date()}.png")
        plt.savefig(out_path, dpi=130)
        if show: plt.show()
        plt.close()
        print(f"[plot] saved {out_path}")

# =============================
# Runner
# =============================
def run_folder(folder: str):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"Found {len(files)} parquet files.")
    all_trades, all_equity = [], []

    for i, fpath in enumerate(files, 1):
        # ---- FIXED ticker parsing (no AttributeError) ----
        base = os.path.basename(fpath)
        name, _ext = os.path.splitext(base)
        ticker = name.upper().replace("_", "").replace("-", "").replace(".", "")
        print(f"\n== {ticker} ({i}/{len(files)}) ==\n{fpath}")
        try:
            df15 = load_parquet_15m(fpath, tz=TZ)
            if df15.empty:
                print("Skipped (no 15m data after normalization/resample/session filter).")
                continue
            df15["Ticker"] = ticker
            df15 = compute_opening_range(df15)
        except Exception as e:
            print(f"Skipping {ticker}: {e}")
            continue

        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            r_targets=R_MULTIPLES,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=SLIPPAGE_BPS,
            fees=FEES_PER_TRADE,
            retest_confirm_close=RETEST_CONFIRM_CLOSE
        )

        if PLOT_EXAMPLES_PER_TICKER and not tlog.empty:
            plot_example_days(df15, tlog, ticker,
                              max_days=PLOT_EXAMPLES_PER_TICKER,
                              save_dir=PLOT_SAVE_DIR,
                              show=PLOT_SHOW)

        if not tlog.empty:
            out = tlog.copy()
            out["Targets"] = out["Targets"].apply(lambda xs: ";".join(f"{p:.6f}" for p in xs))
            out["Exits"] = out["Exits"].apply(lambda xs: ";".join(f"{t}|{ts}|{px:.6f}" for (t, ts, px) in xs))
            all_trades.append(out)
        if not eq.empty:
            eq2 = eq.copy(); eq2["Ticker"] = ticker
            all_equity.append(eq2)

        if AUTOSAVE_EVERY and i % AUTOSAVE_EVERY == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

        time.sleep(SLEEP_BETWEEN_TICKERS)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

# =============================
# Main
# =============================
def main():
    os.makedirs(PLOT_SAVE_DIR, exist_ok=True)

    trades, equity = run_folder(PARQUET_DIR)

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

        # Equity/losses/drawdown plot
        plot_equity_with_losses(trades, folder=PARQUET_DIR, save_dir=PLOT_SAVE_DIR, show=PLOT_SHOW)
    else:
        print("No trades generated with current settings.")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

if __name__ == "__main__":
    main()


Found 20 parquet files.

== AAPL (1/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AAPL.parquet

== AMD (2/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMD.parquet

== AMZN (3/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMZN.parquet

== CAT (4/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CAT.parquet

== CHWY (5/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CHWY.parquet

== CVNA (6/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CVNA.parquet

== GLD (7/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GLD.parquet

== GOOGL (8/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GOOGL.parquet

== GS (9/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GS.parquet

== JPM (10/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\JPM.parquet

== MSFT (11/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\MSFT.pa